# 🚀 OmniVoice Studio — Kaggle Batch Generator (v1 to v5)
Run all cells sequentially in Kaggle with **GPU T4** enabled.

In [ ]:
# ⚙️ Step 1: Check GPU Status
!nvidia-smi
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 📦 Step 2: Clone & Install OmniVoice Studio
!pip install -q uv
!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
# ⚡ Step 3: Launch OmniVoice Backend Server in Background
import os, time, subprocess
%cd /kaggle/working/omnivoice-studio

log_file = open("/tmp/omnivoice.log", "w")
subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log_file, stderr=log_file)

print("⏳ Waiting 15s for server startup...")
time.sleep(15)
!curl -sf http://localhost:3900/health || echo '❌ Server starting failed! Check /tmp/omnivoice.log'

In [ ]:
# 🎵 Step 4: Run Full Batch Generation (v1 to v5)
import os, time, json, urllib.request

BASE_OUTPUT_DIR = "/kaggle/working/outputs"
API_URL = "http://localhost:3900/v1/audio/speech"

def gen(text, voice="shimmer", model="tts-1", output_path="", speed=1.0, num_step=32, guidance_scale=2.0, instruct=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {"model": model, "voice": voice, "input": text, "response_format": "wav", "speed": speed, "num_step": num_step, "guidance_scale": guidance_scale}
    if instruct: payload["instruct"] = instruct
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data, headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            print(f"  ✅ {os.path.basename(output_path)} ({len(content)//1024} KB in {time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")

print("🚀 Starting Batch Generation (v1-v5)...")

# V1 Test
print("\n--- V1 Test ---")
gen("Hello! OmniVoice Studio on Kaggle GPU.", voice="alloy", output_path=f"{BASE_OUTPUT_DIR}/v1_test/v1_test_alloy.wav")
gen("Testing echo voice preset.", voice="echo", output_path=f"{BASE_OUTPUT_DIR}/v1_test/v1_test_echo.wav")

# V2 McCullen
print("\n--- V2 McCullen Nanomites ---")
mccullen = "Nanomites... programmed to devour metal, steel, flesh. But more importantly, they can be programmed to stop. The real-world applications are endless... So, you tell me... is it working?"
gen(mccullen, voice="alloy", output_path=f"{BASE_OUTPUT_DIR}/v2_mccullen/v2_mccullen_alloy.wav")
gen(mccullen, voice="onyx", output_path=f"{BASE_OUTPUT_DIR}/v2_mccullen/v2_mccullen_onyx.wav")
gen(mccullen, voice="shimmer", output_path=f"{BASE_OUTPUT_DIR}/v2_mccullen/v2_mccullen_shimmer.wav")

# V3 Movies
print("\n--- V3 Movie Dialogues ---")
gen("Kitne aadmi the? ... Do? ... Aur tum teen! ... Phir bhi wapas aa gaye... Khaali haath!", voice="alloy", output_path=f"{BASE_OUTPUT_DIR}/v3_movies/v3_sholay_gabbar_hindi.wav")
gen("আম্মাজান! আপনি শুধু একটা বার নির্দেশ দেন, আজ পুরো পৃথিবীকে আমি আপনার পায়ের নিচে এনে হাজির করব!", voice="onyx", output_path=f"{BASE_OUTPUT_DIR}/v3_movies/v3_ammajan_manna_bangla.wav")
gen("My name is Maximus Decimus Meridius, commander of the Armies of the North, General of the Felix Legions, and loyal servant to the true emperor, Marcus Aurelius. Father to a murdered son, husband to a murdered wife. And I will have my vengeance, in this life or the next.", voice="shimmer", output_path=f"{BASE_OUTPUT_DIR}/v3_movies/v3_gladiator_maximus_english.wav")

# V4 Deep Male
print("\n--- V4 Deep Male Voices ---")
deep = "The darkness does not scare me. I have walked through fire, through war, through loss. And still I stand. Because I am not built from hope alone. I am forged from pain, from rage, from the silence between heartbeats."
for v in ["shimmer", "onyx", "echo", "alloy", "demo0001"]:
    gen(deep, voice=v, model="tts-1-hd", num_step=32, speed=0.9, output_path=f"{BASE_OUTPUT_DIR}/v4_deep_male/v4_deep_male_hd_{v}.wav")

# V5 Ultra Human & Emotions
print("\n--- V5 Ultra Human & Emotions (Shimmer) ---")
gen("The darkness does not scare me anymore. I have walked through fire... through war... through loss. And still... I stand.", voice="shimmer", speed=0.85, num_step=32, guidance_scale=2.5, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_shimmer_slow_hq.wav")
gen("No! NO! I will NOT let this happen again! You hear me?! I have given EVERYTHING! Every last drop of blood, every ounce of strength!", voice="shimmer", speed=1.05, num_step=32, guidance_scale=3.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_rage.wav")
gen("She's gone... I... I can't believe she's actually gone. We had so many plans... so many dreams we never... We never got to live them out. And now... this silence... it's deafening.", voice="shimmer", speed=0.85, num_step=32, guidance_scale=2.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_sadness.wav")
gen("Ha ha ha! Oh man, you should have seen the look on his face! I swear, I haven't laughed this hard in years! Oh god, my stomach hurts from laughing!", voice="shimmer", speed=1.0, num_step=32, guidance_scale=2.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_laughter.wav")
gen("Listen carefully... I'm only going to say this once. They're watching us. Don't turn around. Just keep walking, and when I say run... you run.", voice="shimmer", speed=0.8, num_step=32, guidance_scale=1.5, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_whisper.wav")
gen("Ugh... I feel terrible. This cold is killing me. Can barely breathe through my nose... My throat is raw and scratchy. Every time I try to speak... it hurts.", voice="shimmer", speed=0.9, num_step=32, guidance_scale=2.0, output_path=f"{BASE_OUTPUT_DIR}/v5_ultra_human/v5_emotion_sick.wav")

print("\n🎉 ALL SAMPLES GENERATED SUCCESSFULLY!")

In [ ]:
# =============================================================================
# 🎭 Step 5: ULTIMATE VOICE & EMOTION SHOWCASE (Single Script)
# =============================================================================
# এই স্ক্রিপ্ট একবার রান করলেই সব ভয়েস × সব ইমোশন কম্বিনেশনে
# আলাদা আলাদা হাই-কোয়ালিটি অডিও ফাইল জেনারেট হবে।
# Kaggle নতুন সেলে পেস্ট করে রান করুন।
# =============================================================================

import os, time, json, urllib.request

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v5_ultimate_showcase"
API_URL = "http://localhost:3900/v1/audio/speech"

# ─── Generation Helper ───────────────────────────────────────────────────────
def gen(text, voice="shimmer", model="tts-1-hd", output_path="",
        speed=1.0, num_step=32, guidance_scale=2.0, instruct=None):
    """OmniVoice TTS দিয়ে একটি WAV ফাইল জেনারেট করে।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model,
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": num_step,
        "guidance_scale": guidance_scale,
    }
    if instruct:
        payload["instruct"] = instruct
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            print(f"  ✅ {os.path.basename(output_path):50s} │ {kb:5d} KB │ {dur:5.1f}s │ voice={voice}")
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")


# =============================================================================
# 📋 VOICE LIST — সব ভয়েস এখানে
# =============================================================================
ALL_VOICES = ["shimmer", "alloy", "onyx", "echo", "nova", "fable", "demo0001"]

# =============================================================================
# 🎬 EMOTION SCRIPTS — প্রতিটি ইমোশনের জন্য আলাদা ডায়ালগ ও সেটিংস
# =============================================================================
EMOTION_SCRIPTS = [
    # ── 1. RAGE / ক্রোধ ──────────────────────────────────────────────────────
    {
        "tag": "rage",
        "label": "🔥 RAGE (ক্রোধ)",
        "text": (
            "No! NO! I will NOT let this happen again! You hear me?! "
            "I have given EVERYTHING! Every last drop of blood, every ounce of strength! "
            "And you DARE stand there and tell me it wasn't enough?! "
            "I will BURN this whole world down before I let them take what's mine! "
            "You want war? FINE. You've got one. And I promise you... "
            "you will NOT survive it."
        ),
        "speed": 1.05,
        "guidance_scale": 3.0,
    },
    # ── 2. DEEP SADNESS / গভীর দুঃখ ─────────────────────────────────────────
    {
        "tag": "sadness",
        "label": "😢 SADNESS (দুঃখ)",
        "text": (
            "She's gone... I... I can't believe she's actually gone. "
            "We had so many plans... so many dreams we never... "
            "we never got to live them out. "
            "I keep reaching for my phone to call her... "
            "and then I remember. And it hits me all over again. "
            "The house is so quiet now. Her chair is still there, "
            "right where she left it. And this silence... "
            "this silence is deafening."
        ),
        "speed": 0.82,
        "guidance_scale": 2.0,
    },
    # ── 3. LAUGHTER / হাসি ────────────────────────────────────────────────────
    {
        "tag": "laughter",
        "label": "😂 LAUGHTER (হাসি)",
        "text": (
            "Ha ha ha! Oh man, you should have seen the look on his face! "
            "I swear, I haven't laughed this hard in YEARS! "
            "He just stood there, completely frozen, with spaghetti all over his shirt! "
            "And then... and then the dog jumped on him! Ha ha ha! "
            "Oh god, my stomach hurts from laughing! "
            "I can't breathe! Someone stop, please! Ha ha ha!"
        ),
        "speed": 1.05,
        "guidance_scale": 2.0,
    },
    # ── 4. WHISPER / ফিসফিস ───────────────────────────────────────────────────
    {
        "tag": "whisper",
        "label": "🤫 WHISPER (ফিসফিস)",
        "text": (
            "Listen carefully... I'm only going to say this once. "
            "They're watching us. Don't turn around. Don't look at the window. "
            "Just keep walking, nice and slow, like nothing's wrong. "
            "There's a car waiting two blocks east. Silver sedan, engine running. "
            "When I say run... you run. And don't look back. "
            "No matter what you hear... don't... look... back."
        ),
        "speed": 0.78,
        "guidance_scale": 1.5,
    },
    # ── 5. SICK / অসুস্থ ──────────────────────────────────────────────────────
    {
        "tag": "sick",
        "label": "🤒 SICK (অসুস্থ)",
        "text": (
            "Ugh... I feel terrible. This cold is killing me. "
            "Can barely breathe through my nose... My throat is raw and scratchy. "
            "Every time I try to speak... it hurts. "
            "I've gone through like two boxes of tissues already. "
            "My head is pounding. My body aches everywhere. "
            "I just want to crawl into bed and not move for a week..."
        ),
        "speed": 0.88,
        "guidance_scale": 2.0,
    },
    # ── 6. HEROIC MONOLOGUE / বীরত্বপূর্ণ ────────────────────────────────────
    {
        "tag": "heroic",
        "label": "⚔️ HEROIC (বীরত্ব)",
        "text": (
            "The darkness does not scare me anymore. "
            "I have walked through fire... through war... through loss. "
            "And still... I stand. "
            "Because I am not built from hope alone. "
            "I am forged from pain, from rage, from the silence between heartbeats. "
            "They thought they could break me. They were wrong. "
            "I am the storm they never saw coming. "
            "And when I rise... the earth will tremble."
        ),
        "speed": 0.85,
        "guidance_scale": 2.5,
    },
    # ── 7. VILLAIN / খলনায়ক ──────────────────────────────────────────────────
    {
        "tag": "villain",
        "label": "🦹 VILLAIN (খলনায়ক)",
        "text": (
            "You really thought you could stop me? How... adorable. "
            "Let me explain something to you, hero. "
            "I have been planning this for fifteen years. Every move, every breath, "
            "every little accident that brought you here... was me. "
            "You are not the hunter. You never were. "
            "You are the mouse... and this maze? I built it. "
            "Now... shall we play one last game?"
        ),
        "speed": 0.90,
        "guidance_scale": 2.5,
    },
    # ── 8. ROMANTIC / রোমান্টিক ───────────────────────────────────────────────
    {
        "tag": "romantic",
        "label": "💕 ROMANTIC (রোমান্টিক)",
        "text": (
            "I don't know when it started. Maybe it was the way you laugh, "
            "the way your eyes light up when you talk about the things you love. "
            "Maybe it was that rainy Tuesday when you held my hand "
            "and didn't let go. I just know that... somewhere between "
            "all the chaos and the noise... I fell. Completely, hopelessly, "
            "irreversibly... in love with you. And I wouldn't change a single moment."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
    },
    # ── 9. EPIC NARRATOR / মহাকাব্যিক বর্ণনা ─────────────────────────────────
    {
        "tag": "narrator",
        "label": "📖 NARRATOR (বর্ণনাকারী)",
        "text": (
            "In the year twenty-one forty-seven, humanity stood at the edge of extinction. "
            "The oceans had risen. The cities had fallen. "
            "And from the ashes of the old world, a new order emerged. "
            "They called themselves the Architects. They promised salvation. "
            "But salvation, as history has taught us, always comes with a price. "
            "This is the story of those who refused to pay it. "
            "This is the story... of the last rebellion."
        ),
        "speed": 0.88,
        "guidance_scale": 2.2,
    },
    # ── 10. MOTIVATIONAL / অনুপ্রেরণামূলক ────────────────────────────────────
    {
        "tag": "motivational",
        "label": "🚀 MOTIVATIONAL (অনুপ্রেরণা)",
        "text": (
            "Listen to me. I don't care how many times you've failed. "
            "I don't care how many people told you it's impossible. "
            "You are still here. You are still breathing. "
            "And that means you still have a chance. "
            "Every single champion, every legend, every person you admire... "
            "they all had a moment where they wanted to quit. "
            "The difference is... they didn't. So get up. Wipe the dust off. "
            "And show the world what you're made of."
        ),
        "speed": 0.92,
        "guidance_scale": 2.5,
    },
    # ── 11. BANGLA EMOTIONAL / বাংলা আবেগ ─────────────────────────────────────
    {
        "tag": "bangla_emotion",
        "label": "🇧🇩 BANGLA EMOTIONAL (বাংলা আবেগ)",
        "text": (
            "আম্মাজান! আপনি শুধু একটা বার নির্দেশ দেন, "
            "আজ পুরো পৃথিবীকে আমি আপনার পায়ের নিচে এনে হাজির করব! "
            "আমি জানি আমার জীবনে কত কষ্ট আছে, কত যন্ত্রণা আছে... "
            "কিন্তু আপনার একটা হাসি... সেটাই আমার সবচেয়ে বড় শক্তি। "
            "এই দুনিয়া আমাকে ভেঙে দিতে চেয়েছে বারবার... "
            "কিন্তু আপনার দোয়া আমাকে দাঁড় করিয়ে রেখেছে।"
        ),
        "speed": 0.88,
        "guidance_scale": 2.2,
    },
    # ── 12. HORROR / ভয়ংকর ────────────────────────────────────────────────────
    {
        "tag": "horror",
        "label": "👻 HORROR (ভয়ংকর)",
        "text": (
            "Do you hear that? That scratching sound... behind the wall. "
            "It's been going on for three nights now. Always at exactly three fifteen. "
            "I told myself it was rats. I told myself it was the pipes. "
            "But then last night... last night, it whispered my name. "
            "And when I pressed my ear against the wall... "
            "I felt something press back."
        ),
        "speed": 0.80,
        "guidance_scale": 1.8,
    },
]


# =============================================================================
# 🚀 MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    total_start = time.time()
    total_files = len(EMOTION_SCRIPTS) * len(ALL_VOICES)

    print("=" * 75)
    print("🎭  ULTIMATE VOICE & EMOTION SHOWCASE  —  V5")
    print("=" * 75)
    print(f"📊  Voices : {len(ALL_VOICES)}  →  {', '.join(ALL_VOICES)}")
    print(f"🎬  Emotions: {len(EMOTION_SCRIPTS)}")
    print(f"📁  Total Files: {total_files}")
    print(f"📂  Output Dir : {BASE_OUTPUT_DIR}")
    print("=" * 75)

    generated = 0
    failed = 0

    for emo in EMOTION_SCRIPTS:
        print(f"\n{'─' * 70}")
        print(f"  {emo['label']}")
        print(f"{'─' * 70}")
        print(f"  📝 \"{emo['text'][:80]}...\"")
        print(f"  ⚙️  speed={emo['speed']}  guidance={emo['guidance_scale']}")
        print()

        for voice in ALL_VOICES:
            filename = f"v5_{emo['tag']}_{voice}.wav"
            output_path = os.path.join(BASE_OUTPUT_DIR, emo["tag"], filename)

            gen(
                text=emo["text"],
                voice=voice,
                model="tts-1-hd",
                output_path=output_path,
                speed=emo["speed"],
                num_step=32,
                guidance_scale=emo["guidance_scale"],
            )
            generated += 1

    # ─── Summary ──────────────────────────────────────────────────────────────
    total_time = time.time() - total_start
    print(f"\n{'=' * 75}")
    print(f"🎉  GENERATION COMPLETE!")
    print(f"{'=' * 75}")
    print(f"  ✅ Generated : {generated} files")
    print(f"  ⏱️  Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"  📂 Output    : {BASE_OUTPUT_DIR}")
    print(f"{'=' * 75}")

    # ─── File listing ─────────────────────────────────────────────────────────
    print("\n📂 Generated File Tree:")
    for root, dirs, files in os.walk(BASE_OUTPUT_DIR):
        level = root.replace(BASE_OUTPUT_DIR, "").count(os.sep)
        indent = "  " * level
        folder = os.path.basename(root)
        print(f"  {indent}📁 {folder}/")
        for f in sorted(files):
            fpath = os.path.join(root, f)
            size_kb = os.path.getsize(fpath) // 1024
            print(f"  {indent}  🎵 {f} ({size_kb} KB)")


In [ ]:
# 📦 Step 5: Zip all outputs for 1-click Download from Kaggle Output tab
!apt-get install -y zip
!zip -r /kaggle/working/omnivoice_v5_ultimate_emotion_showcase.zip /kaggle/working/outputs/v5_ultimate_showcase

print("\n✅ DOWNLOAD READY!")
print("📁 Kaggle Output Tab -> Download 'omnivoice_v5_ultimate_emotion_showcase.zip'")
print(f"📊 Total Emotions: 12 | Voices: 7 | Files: 84")

In [ ]:
import os
from IPython.display import FileLink

# ১. ফাইলটিকে বর্তমান ডিরেক্টরিতে কপি করে আনা
!cp /kaggle/working/omnivoice_v5_ultimate_emotion_showcase.zip ./

# ২. সঠিক ডাউনলোড লিংক তৈরি করা
FileLink('omnivoice_v5_ultimate_emotion_showcase.zip')

In [ ]:
# 📦 Step 5: Zip all outputs for 1-click Download from Kaggle Output tab
!apt-get install -y zip
!zip -r /kaggle/working/omnivoice_outputs_v1_to_v5.zip /kaggle/working/outputs
print("\n✅ DOWNLOAD READY!")
print("📁 Kaggle Output Tab -> Download 'omnivoice_outputs_v1_to_v5.zip'")

In [ ]:
# =============================================================================
# 🎭 VERSION 6: ONE VOICE, ALL EMOTIONS — SINGLE FILE
# =============================================================================
# একটি মাত্র অডিও ফাইল জেনারেট হবে (1-3 মিনিট)
# একই ভয়েসে হাসি, কান্না, ফিসফিস, রাগ, ভয় — সব ইমোশন একসাথে!
# Kaggle নোটবুকে নতুন সেলে পেস্ট করে রান করুন।
# =============================================================================

import os, time, json, urllib.request

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v6_single_all_emotions"
API_URL = "http://localhost:3900/v1/audio/speech"

def gen(text, voice="shimmer", model="tts-1-hd", output_path="",
        speed=1.0, num_step=32, guidance_scale=2.0):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=600) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            minutes = (kb / 176)  # rough WAV estimate: ~176 KB/sec @ 16-bit mono
            print(f"  ✅ DONE!")
            print(f"     📁 {os.path.basename(output_path)}")
            print(f"     📊 {kb} KB (~{minutes:.1f} sec estimated)")
            print(f"     ⏱️  Generation time: {dur:.1f}s")
    except Exception as e:
        print(f"  ❌ FAILED: {e}")

# =============================================================================
# 🎬 THE MONOLOGUE — এক ভয়েসে সব ইমোশন
# =============================================================================
# এটা একটা ক্যারেক্টারের মনোলগ যেখানে সে স্বাভাবিকভাবে এক ইমোশন থেকে
# আরেক ইমোশনে যায় — ঠিক যেন একটা সিনেমার ক্লাইম্যাক্স সিন।
# =============================================================================

MONOLOGUE = """
You know what's funny? Ha ha ha! I actually thought we'd make it. I really did. I used to sit there, planning our future, laughing about the stupid things we'd do when we got old. Ha ha ha! Remember that time you tried to cook pasta and set the fire alarm off? Oh man, I laughed so hard I couldn't breathe! Ha ha ha!

But then... then everything changed.

She's gone. I still can't say it without my voice breaking. We had so many plans... so many dreams we never got to live out. I keep reaching for my phone to call her, and then I remember. Every single time, it hits me like the first time. The house is so quiet now. Her chair is still there, exactly where she left it. I haven't moved it. I can't. Sometimes I sit next to it and just... talk to her. Like she's still here. Like she can still hear me.

Listen... listen carefully. I'm only going to say this once. They're watching us. Don't turn around. Don't look at the window. Just keep walking, nice and slow, like nothing is wrong. There's a car waiting two blocks east. Silver sedan, engine running. When I say run... you run. And don't look back. No matter what you hear behind us... don't... look... back.

No! NO! I will NOT let this happen again! You hear me?! I have given EVERYTHING! Every last drop of blood, every ounce of strength I had left! And you DARE stand there and tell me it wasn't enough?! I will burn this whole world down before I let them take what's mine! You want war? FINE! You've got one! And I promise you... I PROMISE you... you will NOT survive it!

Do you hear that? That scratching sound... behind the wall. It's been going on for three nights now. Always at exactly three fifteen in the morning. I told myself it was rats. I told myself it was the old pipes. But then last night... last night, it whispered my name. And when I pressed my ear against the wall... I felt something... press back.

Ugh... I feel terrible. This cold is killing me. Can barely breathe through my nose. My throat is raw and scratchy. Every time I try to speak, it hurts. My head is pounding. My body aches everywhere. I just want to crawl into bed and not move for a week.

But I can't stop. Not now. Not after everything.

The darkness does not scare me anymore. I have walked through fire. Through war. Through loss. And still... I stand. Because I am not built from hope alone. I am forged from pain, from rage, from the silence between heartbeats. They thought they could break me. They were wrong. I am the storm they never saw coming. And when I rise... the earth will tremble.

And you know what keeps me going through all of this? You. I don't know when it started. Maybe it was the way you laugh, the way your eyes light up when you talk about the things you love. Maybe it was that rainy Tuesday when you held my hand and didn't let go. Somewhere between all the chaos and the noise... I fell. Completely, hopelessly, irreversibly in love with you.

So no... I'm not giving up. Not today. Not ever. Listen to me. I don't care how many times I've fallen. I don't care how many people said it's impossible. I am still here. I am still breathing. And that means I still have a chance. So I'll get up. I'll wipe the dust off. And I'll show this world... what I'm made of.
"""

# =============================================================================
# 🚀 GENERATE
# =============================================================================
print("=" * 70)
print("🎭  VERSION 6 — ONE VOICE, ALL EMOTIONS, SINGLE FILE")
print("=" * 70)
print(f"📝  Script length: {len(MONOLOGUE)} characters")
print(f"📂  Output: {BASE_OUTPUT_DIR}")
print("=" * 70)

# --- Main generation: shimmer voice ---
print("\n🎤 Generating with SHIMMER voice (best emotional range)...")
gen(
    text=MONOLOGUE.strip(),
    voice="shimmer",
    model="tts-1-hd",
    output_path=f"{BASE_OUTPUT_DIR}/v6_all_emotions_shimmer.wav",
    speed=0.88,
    num_step=32,
    guidance_scale=2.5,
)

print(f"\n{'=' * 70}")
print("🎉  VERSION 6 — SINGLE FILE GENERATED!")
print(f"{'=' * 70}")
print(f"📂 Output: {BASE_OUTPUT_DIR}/v6_all_emotions_shimmer.wav")
print()
print("🎬 এই একটি ফাইলে যা যা আছে:")
print("   😂 হাসি (Laughter)        — প্যারাগ্রাফ ১")
print("   😢 কান্না (Sadness)        — প্যারাগ্রাফ ২")
print("   🤫 ফিসফিস (Whisper)       — প্যারাগ্রাফ ৩")
print("   🔥 ক্রোধ (Rage)           — প্যারাগ্রাফ ৪")
print("   👻 ভয়ংকর (Horror)        — প্যারাগ্রাফ ৫")
print("   🤒 অসুস্থ (Sick)          — প্যারাগ্রাফ ৬")
print("   ⚔️ বীরত্ব (Heroic)        — প্যারাগ্রাফ ৭")
print("   💕 রোমান্টিক (Romantic)    — প্যারাগ্রাফ ৮")
print("   🚀 অনুপ্রেরণা (Motivational) — প্যারাগ্রাফ ৯")
print(f"{'=' * 70}")


In [ ]:
import os, shutil
from IPython.display import FileLink, display

filename = "v6_all_emotions_shimmer.wav"

# ১. ফাইলটি Kaggle এরিয়াত কোথায় আছে তা খুঁজে বের করা
found_path = None
for root, dirs, files in os.walk("/kaggle/working"):
    if filename in files:
        found_path = os.path.join(root, filename)
        break

# ২. বর্তমান ডিরেক্টরিতে (./) কপি করে সঠিক ডাউনলোড লিংক দেখানো
if found_path:
    shutil.copy(found_path, f"./{filename}")
    print("✅ ফাইল পাওয়া গেছে! নিচের লিংকে ক্লিক করে ডাউনলোড করুন:")
    display(FileLink(filename))
else:
    print("❌ ফাইলটি খুঁজে পাওয়া যায়নি।")

In [ ]:
# =============================================================================
# 🎤 VERSION 7: YOUR VOICE, YOUR SCRIPT — CUSTOM PROMPT INPUT
# =============================================================================
# আপনার নিজের লেখা স্ক্রিপ্ট ইনপুট দিন, মেল ভয়েসে জেনারেট হবে!
# Kaggle নোটবুকে নতুন সেলে পেস্ট করে রান করুন।
# =============================================================================

import os, time, json, urllib.request

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v7_custom2_prompt"
API_URL = "http://localhost:3900/v1/audio/speech"

def gen(text, voice="onyx", model="tts-1-hd", output_path="",
        speed=1.0, num_step=32, guidance_scale=2.0):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=600) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            print(f"  ✅ DONE!")
            print(f"     📁 {os.path.basename(output_path)}")
            print(f"     📊 {kb} KB")
            print(f"     ⏱️  Generation time: {dur:.1f}s")
    except Exception as e:
        print(f"  ❌ FAILED: {e}")


# =============================================================================
# ✏️ আপনার স্ক্রিপ্ট নিচে লিখুন
# =============================================================================
# 💡 প্রম্পটিং গাইড:
#
# 🔥 রাগ/ক্রোধ      → CAPS ব্যবহার করুন, ! দিন, ছোট ছোট বাক্য
#                      "NO! I will NOT let this happen! You HEAR me?!"
#
# 😢 দুঃখ/কান্না     → "..." দিয়ে থামুন, ধীর গতির বাক্য
#                      "She's gone... I... I can't believe it..."
#
# 🤫 ফিসফিস          → "..." বেশি দিন, ছোট বাক্য, সরাসরি সম্বোধন
#                      "Listen... don't move... they're right outside..."
#
# 😂 উত্তেজনা/হাসি   → "Ha ha ha!", "!", দ্রুত বাক্য, ইন্টারজেকশন
#                      "Ha ha ha! Oh man! I can't believe it!"
#
# 👻 ভয়/আতঙ্ক       → "..." দিয়ে থামুন, প্রশ্ন করুন, ধীরে বলুন
#                      "Do you hear that?... What was that sound?..."
#
# ⚔️ বীরত্ব          → গম্ভীর, দীর্ঘ বাক্য, মেটাফোর ব্যবহার করুন
#                      "I am the storm. I will not fall. I will rise."
#
# 💕 রোমান্টিক       → নরম শব্দ, "..." দিয়ে পজ, আবেগপূর্ণ
#                      "I fell... completely... in love with you."
#
# 🤒 অসুস্থ/ক্লান্ত  → "Ugh...", ছোট বাক্য, শ্বাসকষ্ট বোঝানো
#                      "Ugh... can barely breathe... everything hurts..."
# =============================================================================

MY_SCRIPT = """

What if the greatest invention in human history wasn't a machine that traveled through space... but one that crossed possibilities?

A machine that could open another universe.

They called it... The Parallel Universe Bridge.

For fifty years, people laughed. "Ha ha ha! Another universe? Seriously? Ha ha ha!" Scientists kept working. Everyone else kept doubting.

Until one day... the Bridge... opened.

And suddenly, the impossible became REAL!

We saw dinosaurs, still walking, beneath blood-red skies. We found another Earth where humanity was born on Mars. We saw floating oceans, cities above the clouds, worlds that no human mind was ever supposed to witness.

People screamed. People cried. People celebrated. "We did it! We actually DID IT!"

But me?

...

No. None of those worlds mattered. Not one.

Because I wasn't searching for dinosaurs. I wasn't searching for aliens. I was searching... for someone.

Three... years ago... cancer... stole my mother.

No warning. No mercy. No second chance.

I watched the hospital monitor... become... silent.

I held her hand... hoping... just hoping... she'd squeeze mine one... last... time.

She never did.

...

Every birthday, I still buy her favorite flowers.

I still dial her phone number.

Just... to hear her voice... on voicemail.

I know... she'll never answer. I KNOW THAT!

But I still call. Because sometimes hope hurts more than reality.

Then the Bridge became operational. Every scientist asked, "What universe should we explore?"

I only typed one sentence.

Listen... take me to the universe... where my mother... never died.

...

The machine accepted. The countdown began.

Three... Two... One...

I opened my eyes.

...

There she was. Alive. Smiling. Making breakfast. Humming the exact same song she used to sing every Sunday morning.

I froze. My legs wouldn't move. My heart forgot how to beat.

She looked at me and smiled. "Oh... you're home early."

...

She didn't know I wasn't her son. I was a broken man... borrowing someone else's miracle.
"""

# =============================================================================
# ⚙️ SETTINGS — প্রয়োজনে পরিবর্তন করুন
# =============================================================================
VOICE = "onyx"          # মেল ভয়েস: "onyx" (গভীর), "echo" (মধ্যম), "alloy" (হালকা)
SPEED = 0.88            # 0.7 = খুব ধীর, 0.88 = সিনেমাটিক, 1.0 = স্বাভাবিক, 1.1 = দ্রুত
GUIDANCE = 2.5          # 1.5 = শান্ত, 2.0 = স্বাভাবিক, 2.5 = আবেগপূর্ণ, 3.0 = চরম আবেগ
FILENAME = "v7_custom2"  # আউটপুট ফাইলের নাম (এক্সটেনশন ছাড়া)

# =============================================================================
# 🚀 GENERATE
# =============================================================================
text = MY_SCRIPT.strip()

print("=" * 70)
print("🎤  VERSION 7 — YOUR VOICE, YOUR SCRIPT")
print("=" * 70)
print(f"📝  Script: {len(text)} chars")
print(f"🎙️  Voice: {VOICE}")
print(f"⚡  Speed: {SPEED} | Guidance: {GUIDANCE}")
print(f"📂  Output: {BASE_OUTPUT_DIR}")
print("=" * 70)

if text == "PUT YOUR SCRIPT HERE. REPLACE THIS ENTIRE TEXT WITH YOUR OWN.":
    print("\n⚠️  আপনি এখনো নিজের স্ক্রিপ্ট লেখেননি!")
    print("    MY_SCRIPT ভ্যারিয়েবলে আপনার টেক্সট পেস্ট করুন।")
else:
    print(f"\n🎤 Generating with {VOICE} voice...")
    gen(
        text=text,
        voice=VOICE,
        model="tts-1-hd",
        output_path=f"{BASE_OUTPUT_DIR}/{FILENAME}_{VOICE}.wav",
        speed=SPEED,
        num_step=32,
        guidance_scale=GUIDANCE,
    )
    print(f"\n{'=' * 70}")
    print(f"🎉  GENERATED: {FILENAME}_{VOICE}.wav")
    print(f"{'=' * 70}")


In [ ]:
import os, base64
from IPython.display import HTML, display

filename = "v7_custom2_onyx.wav"

# ১. ফাইলটি খুঁজে বের করা
found_path = None
for root, dirs, files in os.walk("/kaggle/working"):
    if filename in files:
        found_path = os.path.join(root, filename)
        break

# ২. Base64 দিয়ে সরাসরি ব্রাউজার ডাউনলোড বাটন তৈরি (৪০৪ এরর মুক্ত)
if found_path:
    with open(found_path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    
    html_code = f'''
    <a download="{filename}" href="data:audio/wav;base64,{b64}">
        <button style="padding: 12px 24px; background-color: #20beff; color: white; border: none; border-radius: 6px; font-weight: bold; font-size: 16px; cursor: pointer;">
            ⬇️ Download {filename}
        </button>
    </a>
    '''
    print("✅ ফাইল প্রস্তুত! নিচের নীল বাটনে ক্লিক করে নামিয়ে নিন:")
    display(HTML(html_code))
else:
    print("❌ ফাইলটি খুঁজে পাওয়া যায়নি।")

In [ ]:
# =============================================================================
# 🎭 VERSION 8: DOCUMENTATION-CORRECT EMOTIONAL TTS
# =============================================================================
# OmniVoice docs/expressive-speech.md (Jul 20, 2026) অনুযায়ী সঠিক approach:
#
# ✅ প্রতিটা emotional beat = আলাদা API call (carryover বন্ধ)
# ✅ শুধু documented tags: [laughter], [sigh], [pause], [breath]
# ✅ Punctuation = prosody control (documented)
# ✅ class_temperature দিয়ে expressive variation
# ✅ postprocess_output OFF = silence/breath preserved
# ✅ আলাদা segments → WAV concatenate → একটি ফাইনাল ফাইল
#
# ❌ CAPS = চেঁচানো (undocumented, কাজ করে না)
# ❌ [cry], [whisper], [trembling] (unsupported, noise আসে)
# ❌ পুরো স্ক্রিপ্ট এক call-এ (emotion carryover হয়)
#
# Kaggle নোটবুকে নতুন সেলে পেস্ট করে রান করুন।
# =============================================================================

import os, time, json, urllib.request, struct, wave, io

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_documented_emotions"
API_URL = "http://localhost:3900/v1/audio/speech"


# ─── WAV Concatenation Helper ────────────────────────────────────────────────
def concat_wavs(wav_paths, output_path):
    """একাধিক WAV ফাইল জোড়া লাগিয়ে একটি ফাইনাল WAV তৈরি করে।"""
    params_set = False
    all_frames = b""

    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())

    if not params_set:
        print("  ❌ কোনো WAV ফাইল পাওয়া যায়নি!")
        return

    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)

    kb = os.path.getsize(output_path) // 1024
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB)")


# ─── Silence Generator ───────────────────────────────────────────────────────
def make_silence_wav(duration_ms, output_path, sample_rate=22050, channels=1, sampwidth=2):
    """নির্দিষ্ট সময়ের নীরবতা WAV ফাইল হিসেবে তৈরি করে।"""
    num_samples = int(sample_rate * duration_ms / 1000)
    silence = b"\x00\x00" * num_samples * channels
    with wave.open(output_path, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sampwidth)
        wf.setframerate(sample_rate)
        wf.writeframes(silence)


# ─── Generation Helper ───────────────────────────────────────────────────────
def gen_segment(text, voice="onyx", model="tts-1-hd", output_path="",
                speed=1.0, num_step=32, guidance_scale=2.0,
                class_temperature=0.0, postprocess_output=True,
                instruct=None, seed=None):
    """একটি সেগমেন্ট জেনারেট করে। প্রতিটা beat আলাদা call।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model,
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": num_step,
        "guidance_scale": guidance_scale,
    }

    # Documented Production Overrides
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    if instruct:
        payload["instruct"] = instruct
    if seed is not None:
        payload["seed"] = seed

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {kb:5d} KB │ {dur:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False


# =============================================================================
# ✏️ আপনার স্ক্রিপ্ট এখানে — সেগমেন্ট আকারে
# =============================================================================
# 📖 প্রতিটা emotional beat = একটা আলাদা সেগমেন্ট
#
# 📋 DOCUMENTED TAGS (শুধু এগুলো কাজ করে):
#    [laughter]     → হাসি
#    [sigh]         → দীর্ঘশ্বাস
#    [breath]       → শ্বাস নেওয়ার শব্দ
#    [pause]        → 350ms default silence
#    [pause 500ms]  → নির্দিষ্ট সময়ের silence
#    [pause 1.5s]   → ১০ সেকেন্ড পর্যন্ত
#
# ⚙️ SETTINGS:
#    speed          → 0.78-1.05 (ধীর-দ্রুত)
#    guidance_scale → 1.5-3.0 (শান্ত-চরম আবেগ)
#    class_temperature → 0.0-0.7 (greedy → more "human" edges)
#    postprocess_output → False = silence/breath preserved (sad segments-এ)
#
# ❌ যা ব্যবহার করবেন না:
#    [cry], [whisper], [trembling], [excited] → unsupported, noise আসবে
#    CAPS দিয়ে চেঁচানো → documented নয়
# =============================================================================

VOICE = "onyx"          # মেল ভয়েস: "onyx" (গভীর), "echo" (মধ্যম), "alloy" (হালকা)
FILENAME = "v8_final"   # আউটপুট ফাইলের নাম

SEGMENTS = [
    # ─── EXAMPLE SCRIPT: "The Parallel Universe Bridge" ───────────────────
    # এটা একটা উদাহরণ। আপনার নিজের স্ক্রিপ্ট দিয়ে রিপ্লেস করুন।
    # প্রতিটা সেগমেন্ট আলাদা API call-এ যাবে → emotion carryover হবে না।
    # ──────────────────────────────────────────────────────────────────────

    # 1. NARRATOR — উত্তেজনা, anticipation
    {
        "tag": "01_narrator_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.95,
        "guidance_scale": 2.2,
        "class_temperature": 0.3,
    },

    # --- 500ms pause ---
    {"tag": "pause_1", "silence_ms": 500},

    # 2. LAUGHTER + DOUBT — হাসি, সন্দেহ
    {
        "tag": "02_doubt_laughter",
        "text": (
            "[laughter] Another universe? Seriously? [laughter] "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 1.05,
        "guidance_scale": 2.0,
        "class_temperature": 0.2,
    },

    # --- 400ms pause ---
    {"tag": "pause_2", "silence_ms": 400},

    # 3. EXCITEMENT — বিস্ময়, উত্তেজনা
    {
        "tag": "03_excitement_discovery",
        "text": (
            "And suddenly — the impossible became real! "
            "We saw dinosaurs, still walking, beneath blood-red skies! "
            "We found another Earth where humanity was born on Mars!"
        ),
        "speed": 1.05,
        "guidance_scale": 2.5,
        "class_temperature": 0.4,
    },

    # --- 800ms pause (mood shift) ---
    {"tag": "pause_3", "silence_ms": 800},

    # 4. PERSONAL SHIFT — ধীর, ব্যক্তিগত
    {
        "tag": "04_personal_shift",
        "text": (
            "But me? [pause 800ms] No. None of those worlds mattered. "
            "Not one. [pause 500ms] I was searching... for someone."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "class_temperature": 0.2,
        "postprocess_output": False,
    },

    # --- 600ms pause ---
    {"tag": "pause_4", "silence_ms": 600},

    # 5. GRIEF — কষ্ট, ক্যান্সারের কথা
    {
        "tag": "05_grief_cancer",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "[pause 1s] No warning. No mercy. No second chance."
        ),
        "speed": 0.80,
        "guidance_scale": 2.0,
        "class_temperature": 0.3,
        "postprocess_output": False,
    },

    # --- 800ms pause ---
    {"tag": "pause_5", "silence_ms": 800},

    # 6. DEEP SADNESS — গভীর কান্না, হাসপাতালের দৃশ্য
    {
        "tag": "06_hospital_memory",
        "text": (
            "[sigh] I watched the hospital monitor... become... silent. "
            "[pause 1.5s] I held her hand — hoping — just hoping — "
            "she'd squeeze mine one... last... time. "
            "[pause 2s] She never did."
        ),
        "speed": 0.78,
        "guidance_scale": 2.0,
        "class_temperature": 0.4,
        "postprocess_output": False,
    },

    # --- 1000ms pause (heavy moment) ---
    {"tag": "pause_6", "silence_ms": 1000},

    # 7. PAIN + FRUSTRATION — কষ্ট মেশানো রাগ
    {
        "tag": "07_pain_frustration",
        "text": (
            "I know she'll never answer. I know that! "
            "[pause 500ms] But I still call. [pause 800ms] "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.82,
        "guidance_scale": 2.5,
        "class_temperature": 0.5,
    },

    # --- 800ms pause ---
    {"tag": "pause_7", "silence_ms": 800},

    # 8. QUIET PLEA — ফিসফিস-মতো অনুরোধ
    {
        "tag": "08_quiet_plea",
        "text": (
            "[breath] Listen. [pause 500ms] "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.80,
        "guidance_scale": 1.5,
        "class_temperature": 0.2,
        "postprocess_output": False,
    },

    # --- 1200ms pause (universe jump) ---
    {"tag": "pause_8", "silence_ms": 1200},

    # 9. WARMTH + RELIEF — উষ্ণতা, মাকে দেখে স্বস্তি
    {
        "tag": "09_warmth_relief",
        "text": (
            "There she was. [pause 500ms] Alive. Smiling. "
            "Making breakfast. Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88,
        "guidance_scale": 2.2,
        "class_temperature": 0.3,
    },

    # --- 800ms pause ---
    {"tag": "pause_9", "silence_ms": 800},

    # 10. BITTERSWEET ENDING — তিক্ত-মধুর শেষ
    {
        "tag": "10_bittersweet_ending",
        "text": (
            "[pause 1s] She didn't know I wasn't her son. "
            "[pause 800ms] I was a broken man... "
            "borrowing someone else's miracle."
        ),
        "speed": 0.82,
        "guidance_scale": 2.0,
        "class_temperature": 0.3,
        "postprocess_output": False,
    },
]


# =============================================================================
# 🚀 GENERATE ALL SEGMENTS → CONCATENATE → ONE FINAL FILE
# =============================================================================
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "_segments")
os.makedirs(seg_dir, exist_ok=True)

segment_count = len([s for s in SEGMENTS if "text" in s])
pause_count = len([s for s in SEGMENTS if "silence_ms" in s])

print("=" * 70)
print("🎭  VERSION 8 — DOCUMENTATION-CORRECT EMOTIONAL TTS")
print("=" * 70)
print(f"🎙️  Voice     : {VOICE}")
print(f"🎬  Segments  : {segment_count} speech + {pause_count} pauses")
print(f"📂  Output    : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()
print("📖 Approach: প্রতিটা emotional beat = আলাদা API call")
print("   → emotion carryover হবে না")
print("   → সবশেষে WAV concatenate হয়ে একটি ফাইনাল ফাইল হবে")
print()

wav_order = []
generated = 0
failed = 0

for i, seg in enumerate(SEGMENTS):
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    # Silence segment
    if "silence_ms" in seg:
        make_silence_wav(seg["silence_ms"], wav_path)
        wav_order.append(wav_path)
        print(f"  ⏸️  {tag:40s} │ {seg['silence_ms']}ms silence")
        continue

    # Speech segment
    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        model="tts-1-hd",
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        num_step=32,
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=seg.get("class_temperature", 0.0),
        postprocess_output=seg.get("postprocess_output", True),
        instruct=seg.get("instruct"),
        seed=seg.get("seed"),
    )

    if success:
        wav_order.append(wav_path)
        generated += 1
    else:
        failed += 1

# ─── Concatenate all segments into final file ─────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating all segments...")

final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
concat_wavs(wav_order, final_path)

# ─── Summary ──────────────────────────────────────────────────────────────
total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  VERSION 8 — GENERATION COMPLETE!")
print(f"{'=' * 70}")
print(f"  ✅ Generated : {generated} segments")
if failed:
    print(f"  ❌ Failed    : {failed} segments")
print(f"  ⏱️  Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 Final File: {final_path}")
print(f"{'=' * 70}")
print()
print("📋 Segments breakdown:")
for seg in SEGMENTS:
    if "silence_ms" in seg:
        print(f"   ⏸️  {seg['tag']:30s} — {seg['silence_ms']}ms silence")
    else:
        emo = seg['tag'].split('_', 1)[1] if '_' in seg['tag'] else seg['tag']
        print(f"   🎤 {seg['tag']:30s} — speed={seg.get('speed',0.88)} "
              f"guide={seg.get('guidance_scale',2.0)} "
              f"temp={seg.get('class_temperature',0.0)}")
print(f"{'=' * 70}")


In [ ]:
import os, base64
from IPython.display import HTML, display

# ভার্সন ৮ আউটপুট পাথ
target_file = "/kaggle/working/outputs/v8_documented_emotions/v8_final_onyx.wav"

# যদি ফাইলটি নির্দিষ্ট পাথে না পেয়ে অন্য পাথে থাকে, তা অটোমেটিক সার্চ করবে
if not os.path.exists(target_file):
    for root, _, files in os.walk("/kaggle/working/outputs"):
        for f in files:
            if "v8_final" in f and f.endswith(".wav"):
                target_file = os.path.join(root, f)
                break

if os.path.exists(target_file):
    filename = os.path.basename(target_file)
    with open(target_file, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    html = f'''
    <a download="{filename}" href="data:audio/wav;base64,{b64}">
        <button style="padding: 12px 24px; background-color: #20beff; color: white; border: none; border-radius: 6px; font-weight: bold; font-size: 16px; cursor: pointer;">
            ⬇️ Download {filename}
        </button>
    </a>
    '''
    print(f"✅ {filename} নামানোর জন্য প্রস্তুত!")
    display(HTML(html))
else:
    print("❌ ফাইলটি খুঁজে পাওয়া যায়নি। সেলের জেনারেশন সম্পন্ন হয়েছে কিনা তা চেক করুন।")

In [ ]:
# =============================================================================
# 🎭 Step 4: V8.1 — DOCUMENTED EMOTIONAL TTS (IMPROVED)
# =============================================================================
# পরিবর্তন:
#   1. Hospital scene → instruct + higher class_temperature + [breath]
#   2. আলাদা silence segment বাদ → text-এর শেষে [pause Xms] keyword
# =============================================================================

import os, time, json, urllib.request, wave, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_1_improved"
API_URL = "http://localhost:3900/v1/audio/speech"

# ─── WAV Concatenation ─────────────────────────────────────────────────────
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files found!")
        return
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB)")

# ─── Segment Generator ─────────────────────────────────────────────────────
def gen_segment(text, voice="onyx", model="tts-1-hd", output_path="",
                speed=1.0, num_step=32, guidance_scale=2.0,
                class_temperature=0.0, postprocess_output=True,
                instruct=None, seed=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    if instruct:
        payload["instruct"] = instruct
    if seed is not None:
        payload["seed"] = seed

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {kb:5d} KB │ {dur:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 📋 DOCUMENTED TAGS ONLY:
#    [laughter] [sigh] [breath] [pause] [pause 500ms] [pause 1.5s]
# ❌ DO NOT USE: [cry] [whisper] [trembling] [excited]
#
# 📌 [pause Xms] = KEYWORD — text-এর শেষে বসাও,
#    engine নিজে "real stitched silence" render করবে।
#    আলাদা silence WAV বানানোর দরকার নেই।
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"
FILENAME = "v8_1_final"

SEGMENTS = [
    # ── 01: Narrator opening — anticipation ──
    {
        "tag": "01_narrator_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities? "
            "[pause 600ms]"
        ),
        "speed": 0.95,
        "guidance_scale": 2.2,
        "class_temperature": 0.3,
    },

    # ── 02: Doubt + laughter ──
    {
        "tag": "02_doubt_laughter",
        "text": (
            "[laughter] Another universe? Seriously? [laughter] "
            "Scientists kept working. Everyone else kept doubting. "
            "[pause 400ms]"
        ),
        "speed": 1.05,
        "guidance_scale": 2.0,
        "class_temperature": 0.2,
    },

    # ── 03: Excitement / wonder ──
    {
        "tag": "03_excitement",
        "text": (
            "And suddenly — the impossible became real! "
            "We saw dinosaurs, still walking, beneath blood-red skies! "
            "We found another Earth where humanity was born on Mars! "
            "[pause 900ms]"
        ),
        "speed": 1.05,
        "guidance_scale": 2.5,
        "class_temperature": 0.4,
    },

    # ── 04: Personal shift — tone drops ──
    {
        "tag": "04_personal_shift",
        "text": (
            "But me? [pause 800ms] "
            "No. None of those worlds mattered. Not one. "
            "[pause 500ms] "
            "I was searching... for someone. "
            "[pause 700ms]"
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "class_temperature": 0.2,
        "postprocess_output": False,
    },

    # ── 05: Grief — mother's death ──
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "[pause 1s] "
            "No warning. No mercy. No second chance. "
            "[pause 900ms]"
        ),
        "speed": 0.80,
        "guidance_scale": 2.0,
        "class_temperature": 0.3,
        "postprocess_output": False,
    },

    # ── 06: HOSPITAL — কান্না-জড়িত কণ্ঠ ──────────────────────────
    #    কৌশল:
    #      • instruct → CosyVoice3 হলে natural-language emotion follow করবে
    #                   default engine হলে safely ignore হবে
    #      • class_temperature 0.6 → বেশি expressive variation
    #      • guidance_scale 1.5 → কম rigid, বেশি natural
    #      • [breath] + [sigh] → documented non-verbal tags
    #      • postprocess_output False → breath/silence artifacts preserve
    #      • speed 0.75 → সবচেয়ে ধীর, ভাঙা গলা
    # ──────────────────────────────────────────────────────────────
    {
        "tag": "06_hospital_crying",
        "text": (
            "[breath] "
            "I watched the hospital monitor... become... silent. "
            "[pause 1.5s] "
            "[sigh] "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time. "
            "[pause 2s] "
            "[breath] "
            "She never did. "
            "[pause 1.5s]"
        ),
        "speed": 0.75,
        "guidance_scale": 1.5,
        "class_temperature": 0.6,
        "postprocess_output": False,
        "instruct": "sound heartbroken, voice breaking, barely holding back tears",
    },

    # ── 07: Pain / frustration ──
    {
        "tag": "07_pain",
        "text": (
            "I know she'll never answer. I know that! "
            "[pause 500ms] "
            "But I still call. "
            "[pause 800ms] "
            "Because sometimes... hope hurts more than reality. "
            "[pause 900ms]"
        ),
        "speed": 0.82,
        "guidance_scale": 2.5,
        "class_temperature": 0.5,
    },

    # ── 08: Plea — quiet, desperate ──
    {
        "tag": "08_plea",
        "text": (
            "[breath] Listen. "
            "[pause 500ms] "
            "Take me to the universe... where my mother... never died. "
            "[pause 1.2s]"
        ),
        "speed": 0.80,
        "guidance_scale": 1.5,
        "class_temperature": 0.2,
        "postprocess_output": False,
    },

    # ── 09: Warmth — seeing her alive ──
    {
        "tag": "09_warmth",
        "text": (
            "There she was. "
            "[pause 500ms] "
            "Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning. "
            "[pause 900ms]"
        ),
        "speed": 0.88,
        "guidance_scale": 2.2,
        "class_temperature": 0.3,
    },

    # ── 10: Ending — reflective, soft ──
    {
        "tag": "10_ending",
        "text": (
            "[pause 1s] "
            "She didn't know I wasn't her son. "
            "[pause 800ms] "
            "I was a broken man... "
            "borrowing someone else's miracle. "
            "[pause 2s]"
        ),
        "speed": 0.82,
        "guidance_scale": 2.0,
        "class_temperature": 0.3,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

print("=" * 70)
print("🎭  VERSION 8.1 — IMPROVED EMOTIONAL TTS")
print("=" * 70)
print(f"🎙️  Voice     : {VOICE}")
print(f"🎬  Segments  : {len(SEGMENTS)} (no separate silence files)")
print(f"📂  Output    : {BASE_OUTPUT_DIR}")
print(f"📌  [pause]   : keyword — engine renders stitched silence")
print(f"🏥  Hospital  : instruct + temp 0.6 + [breath][sigh]")
print("=" * 70)
print()

wav_order = []
generated = 0

for seg in SEGMENTS:
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        model="tts-1-hd",
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        num_step=32,
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=seg.get("class_temperature", 0.0),
        postprocess_output=seg.get("postprocess_output", True),
        instruct=seg.get("instruct"),
        seed=seg.get("seed"),
    )
    if success:
        wav_order.append(wav_path)
        generated += 1

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating all segments...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated} segments → 1 file")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD BUTTON
# ═════════════════════════════════════════════════════════════════════════════
if os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl_name = os.path.basename(final_path)
    html = f'''<a download="{dl_name}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl_name}
    </button></a>'''
    print("\n⬇️  নিচের বাটনে ক্লিক করে সরাসরি ডাউনলোড করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি — উপরের error চেক করুন।")

In [ ]:
# =============================================================================
# 🎭 Step 4: V8.2 — FIXED (Hospital 500 error + Download button)
# =============================================================================
# Fix 1: hospital segment → instruct বাদ, temp 0.6→0.45, [breath]→বাদ
# Fix 2: download → FileLink (base64 বাদ, large file safe)
# Fix 3: 40s generate = normal (final audio ~97s)
# =============================================================================

import os, time, json, urllib.request, wave
from IPython.display import display, FileLink

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_2_fixed"
API_URL = "http://localhost:3900/v1/audio/speech"

# ─── WAV Concatenation ─────────────────────────────────────────────────────
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files found!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    # Audio duration calculate
    duration_s = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {duration_s:.1f}s audio)")
    return output_path

# ─── Segment Generator ─────────────────────────────────────────────────────
def gen_segment(text, voice="onyx", model="tts-1-hd", output_path="",
                speed=1.0, num_step=32, guidance_scale=2.0,
                class_temperature=0.0, postprocess_output=True,
                seed=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    if seed is not None:
        payload["seed"] = seed

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {kb:5d} KB │ {dur:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 📋 DOCUMENTED TAGS ONLY:
#    [laughter] [sigh] [pause] [pause 500ms] [pause 1.5s]
# ❌ DO NOT USE: [cry] [whisper] [trembling] [excited] [breath]
# ⚠️ instruct parameter: DEFAULT ENGINE SUPPORT করে না → 500 error
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"
FILENAME = "v8_2_final"

SEGMENTS = [
    # ── 01: Narrator opening — anticipation ──
    {
        "tag": "01_narrator_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities? "
            "[pause 600ms]"
        ),
        "speed": 0.95,
        "guidance_scale": 2.2,
        "class_temperature": 0.3,
    },

    # ── 02: Doubt + laughter ──
    {
        "tag": "02_doubt_laughter",
        "text": (
            "[laughter] Another universe? Seriously? [laughter] "
            "Scientists kept working. Everyone else kept doubting. "
            "[pause 400ms]"
        ),
        "speed": 1.05,
        "guidance_scale": 2.0,
        "class_temperature": 0.2,
    },

    # ── 03: Excitement / wonder ──
    {
        "tag": "03_excitement",
        "text": (
            "And suddenly — the impossible became real! "
            "We saw dinosaurs, still walking, beneath blood-red skies! "
            "We found another Earth where humanity was born on Mars! "
            "[pause 900ms]"
        ),
        "speed": 1.05,
        "guidance_scale": 2.5,
        "class_temperature": 0.4,
    },

    # ── 04: Personal shift — tone drops ──
    {
        "tag": "04_personal_shift",
        "text": (
            "But me? [pause 800ms] "
            "No. None of those worlds mattered. Not one. "
            "[pause 500ms] "
            "I was searching... for someone. "
            "[pause 700ms]"
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "class_temperature": 0.2,
        "postprocess_output": False,
    },

    # ── 05: Grief — mother's death ──
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "[pause 1s] "
            "No warning. No mercy. No second chance. "
            "[pause 900ms]"
        ),
        "speed": 0.80,
        "guidance_scale": 2.0,
        "class_temperature": 0.3,
        "postprocess_output": False,
    },

    # ── 06: HOSPITAL — কান্না-জড়িত কণ্ঠ (FIXED) ─────────────────
    #    আগের error কারণ:
    #      ❌ instruct="sound heartbroken..." → default engine 500 দেয়
    #      ❌ class_temperature 0.6 → অনেক বেশি, unstable
    #      ❌ [breath] → default engine-এ supported না
    #    এখন:
    #      ✅ instruct বাদ
    #      ✅ class_temperature 0.45 (safe range)
    #      ✅ [sigh] রাখা (documented)
    #      ✅ guidance_scale 1.8 (1.5 অনেক কম ছিল)
    #      ✅ speed 0.75 (সবচেয়ে ধীর = ভাঙা গলা)
    #      ✅ postprocess_output False (breath artifacts preserve)
    # ──────────────────────────────────────────────────────────────
    {
        "tag": "06_hospital_crying",
        "text": (
            "[sigh] "
            "I watched the hospital monitor... become... silent. "
            "[pause 1.5s] "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time. "
            "[pause 2s] "
            "She never did. "
            "[pause 1.5s]"
        ),
        "speed": 0.75,
        "guidance_scale": 1.8,
        "class_temperature": 0.45,
        "postprocess_output": False,
    },

    # ── 07: Pain / frustration ──
    {
        "tag": "07_pain",
        "text": (
            "I know she'll never answer. I know that! "
            "[pause 500ms] "
            "But I still call. "
            "[pause 800ms] "
            "Because sometimes... hope hurts more than reality. "
            "[pause 900ms]"
        ),
        "speed": 0.82,
        "guidance_scale": 2.5,
        "class_temperature": 0.5,
    },

    # ── 08: Plea — quiet, desperate ──
    {
        "tag": "08_plea",
        "text": (
            "Listen. "
            "[pause 500ms] "
            "Take me to the universe... where my mother... never died. "
            "[pause 1.2s]"
        ),
        "speed": 0.80,
        "guidance_scale": 1.8,
        "class_temperature": 0.2,
        "postprocess_output": False,
    },

    # ── 09: Warmth — seeing her alive ──
    {
        "tag": "09_warmth",
        "text": (
            "There she was. "
            "[pause 500ms] "
            "Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning. "
            "[pause 900ms]"
        ),
        "speed": 0.88,
        "guidance_scale": 2.2,
        "class_temperature": 0.3,
    },

    # ── 10: Ending — reflective, soft ──
    {
        "tag": "10_ending",
        "text": (
            "[pause 1s] "
            "She didn't know I wasn't her son. "
            "[pause 800ms] "
            "I was a broken man... "
            "borrowing someone else's miracle. "
            "[pause 2s]"
        ),
        "speed": 0.82,
        "guidance_scale": 2.0,
        "class_temperature": 0.3,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

print("=" * 70)
print("🎭  VERSION 8.2 — FIXED EMOTIONAL TTS")
print("=" * 70)
print(f"🎙️  Voice     : {VOICE}")
print(f"🎬  Segments  : {len(SEGMENTS)}")
print(f"📂  Output    : {BASE_OUTPUT_DIR}")
print(f"📌  [pause]   : keyword — engine renders stitched silence")
print(f"🏥  Hospital  : sigh + temp 0.45 + speed 0.75 (NO instruct)")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []

for seg in SEGMENTS:
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        model="tts-1-hd",
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        num_step=32,
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=seg.get("class_temperature", 0.0),
        postprocess_output=seg.get("postprocess_output", True),
        seed=seg.get("seed"),
    )
    if success:
        wav_order.append(wav_path)
        generated += 1
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating all segments...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(SEGMENTS)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  Generate time: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DOWNLOAD — FileLink (base64 বাদ, large file safe)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    size_mb = os.path.getsize(final_path) / (1024 * 1024)
    print(f"\n📥 Download ({size_mb:.1f} MB):")
    print(f"   নিচের লিংকে ক্লিক করুন, অথবা")
    print(f"   Kaggle ডান পাশে 'Output' panel → outputs → v8_2_fixed → ফাইল ডাউনলোড")
    print()
    display(FileLink(final_path))
else:
    print("\n❌ ফাইল তৈরি হয়নি — উপরের error চেক করুন।")

In [ ]:
import os, base64
from IPython.display import HTML, display

# v8.2 জেনারেট হওয়া অডিও ফাইলের পাথ
target_file = "/kaggle/working/outputs/v8_2_fixed/v8_2_final_onyx.wav"

# যদি ফাইল অন্য কোনো পাথে থাকে তবে তা খুঁজে বের করা
if not os.path.exists(target_file):
    for root, _, files in os.walk("/kaggle/working"):
        if "v8_2_final_onyx.wav" in files:
            target_file = os.path.join(root, "v8_2_final_onyx.wav")
            break

if os.path.exists(target_file):
    filename = os.path.basename(target_file)
    with open(target_file, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    html = f'''
    <a download="{filename}" href="data:audio/wav;base64,{b64}">
        <button style="padding: 12px 24px; background-color: #20beff; color: white; border: none; border-radius: 6px; font-weight: bold; font-size: 16px; cursor: pointer;">
            ⬇️ Download {filename} (4.8 MB)
        </button>
    </a>
    '''
    print(f"✅ {filename} ডাউনলোডের জন্য প্রস্তুত!")
    display(HTML(html))
else:
    print("❌ ফাইলটি খুঁজে পাওয়া যায়নি।")

In [ ]:
 # ⚡ Step 3: Launch Server
import os, time, subprocess
%cd /kaggle/working/omnivoice-studio

log_file = open("/tmp/omnivoice.log", "w")
subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log_file, stderr=log_file)

print("⏳ Waiting 15s for server startup...")
time.sleep(15)
!curl -sf http://localhost:3900/health || echo '❌ Server failed! Check /tmp/omnivoice.log'


In [ ]:
# =============================================================================
# 🎭 Step 4: V8.3 — ERROR-CORRECTED VERSION
# =============================================================================
# Fix 1: FileNotFoundError → cwd fallback + path auto-detect
# Fix 2: FileLink 404 → base64 download button
# Fix 3: Server restart → graceful fallback if already running
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_3_corrected"
API_URL = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"

# ═════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER AUTO-RESTART (ERROR-CORRECTED)
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        req = urllib.request.Request(HEALTH_URL)
        with urllib.request.urlopen(req, timeout=5) as resp:
            return resp.status == 200
    except:
        return False

def find_studio_dir():
    """OmniVoice Studio ডিরেক্টরি অটো-ডিটেক্ট করে।"""
    candidates = [
        "/kaggle/working/omnivoice-studio",
        "/kaggle/working/OmniVoice-Studio",
        os.getcwd(),  # current dir (if %cd already ran)
    ]
    # os.walk দিয়ে খোঁজা (যদি অন্য নামে থাকে)
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            candidates.insert(0, root)
            break
    for path in candidates:
        if os.path.exists(os.path.join(path, "backend", "main.py")):
            return path
    return None

def start_server():
    studio_dir = find_studio_dir()
    if not studio_dir:
        print("  ❌ OmniVoice Studio ফোল্ডার পাওয়া যায়নি!")
        print("     Step 2 (Clone & Install) সেলটি আগে রান করুন।")
        return False

    print(f"  📂 Studio Dir: {studio_dir}")
    print("  🔄 Starting server...")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(
        ["uv", "run", "python", "backend/main.py"],
        stdout=log_file, stderr=log_file,
        cwd=studio_dir
    )
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Still waiting... ({(i+1)*2}s)")
    print("  ❌ Server failed! Check: !cat /tmp/omnivoice.log")
    return False

# ─── Check & Restart ──────────────────────────────────────────────────────
print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)

if server_alive():
    print("  ✅ Server already running on :3900")
else:
    print("  ⚠️  Server not responding — restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 cells first.")

print()

# ═════════════════════════════════════════════════════════════════════════════
# 🎵 TTS FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files found!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    duration_s = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {duration_s:.1f}s audio)")
    return output_path

def gen_segment(text, voice="onyx", model="tts-1-hd", output_path="",
                speed=1.0, num_step=32, guidance_scale=2.0,
                class_temperature=0.0, postprocess_output=True, seed=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    if seed is not None:
        payload["seed"] = seed

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {kb:5d} KB │ {dur:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 📋 DOCUMENTED TAGS ONLY:
#    ✅ [laughter] [sigh] [breath] [pause] [pause 500ms] [pause 2s]
#    ✅ ... (ellipses) — (em dash) ! (exclamation) = prosody control
#    ❌ [cry] [whisper] [trembling] [excited] = unsupported
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"
FILENAME = "v8_3_final"

SEGMENTS = [
    {
        "tag": "01_narrator_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.95, "guidance_scale": 2.2, "class_temperature": 0.3,
    },
    {
        "tag": "02_doubt_laughter",
        "text": (
            "[laughter] Another universe? Seriously? [laughter] "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 1.05, "guidance_scale": 2.0, "class_temperature": 0.2,
    },
    {
        "tag": "03_excitement",
        "text": (
            "And suddenly — the impossible became real! "
            "We saw dinosaurs, still walking, beneath blood-red skies! "
            "We found another Earth where humanity was born on Mars!"
        ),
        "speed": 1.05, "guidance_scale": 2.5, "class_temperature": 0.4,
    },
    {
        "tag": "04_personal_shift",
        "text": (
            "But me? "
            "No. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85, "guidance_scale": 2.0, "class_temperature": 0.2,
        "postprocess_output": False,
    },
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.80, "guidance_scale": 2.0, "class_temperature": 0.3,
        "postprocess_output": False,
    },
    {
        "tag": "06_hospital_crying",
        "text": (
            "[sigh] "
            "I watched the hospital monitor... become... silent. "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time. "
            "[pause 2s] "
            "She never did."
        ),
        "speed": 0.75, "guidance_scale": 1.8, "class_temperature": 0.45,
        "postprocess_output": False,
    },
    {
        "tag": "07_pain",
        "text": (
            "I know she'll never answer. I know that! "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.82, "guidance_scale": 2.5, "class_temperature": 0.5,
    },
    {
        "tag": "08_plea",
        "text": (
            "[breath] Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.80, "guidance_scale": 1.8, "class_temperature": 0.2,
        "postprocess_output": False,
    },
    {
        "tag": "09_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88, "guidance_scale": 2.2, "class_temperature": 0.3,
    },
    {
        "tag": "10_ending",
        "text": (
            "[pause 1.5s] "
            "She didn't know I wasn't her son. "
            "I was a broken man... borrowing someone else's miracle."
        ),
        "speed": 0.82, "guidance_scale": 2.0, "class_temperature": 0.3,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

print("=" * 70)
print("🎭  VERSION 8.3 — ERROR-CORRECTED")
print("=" * 70)
print(f"🎙️  Voice     : {VOICE}")
print(f"🎬  Segments  : {len(SEGMENTS)}")
print(f"📂  Output    : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []

for seg in SEGMENTS:
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        model="tts-1-hd",
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        num_step=32,
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=seg.get("class_temperature", 0.0),
        postprocess_output=seg.get("postprocess_output", True),
        seed=seg.get("seed"),
    )
    if success:
        wav_order.append(wav_path)
        generated += 1
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating all segments...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(SEGMENTS)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD BUTTON (base64 — 404-proof)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl_name = os.path.basename(final_path)
    size_mb = os.path.getsize(final_path) / (1024 * 1024)
    html = f'''<a download="{dl_name}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl_name} ({size_mb:.1f} MB)
    </button></a>'''
    print(f"\n⬇️  নিচের বাটনে ক্লিক করে সরাসরি ডাউনলোড করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি — উপরের error চেক করুন।")


In [ ]:
import os, base64
from IPython.display import HTML, display

# v8.3 জেনারেট হওয়া ফোল্ডারের পাথ
target_dir = "/kaggle/working/outputs/v8_3_corrected"
target_file = None

# ফোল্ডারের ভেতরে জেনারেট হওয়া ফাইনাল ফাইলটি খুঁজে বের করা
if os.path.exists(target_dir):
    for f in os.listdir(target_dir):
        if f.endswith(".wav") and not f.startswith("_"):
            target_file = os.path.join(target_dir, f)
            break

# ব্যাকআপ অটো-সার্চ (যদি ফাইলটি অন্য পাথে তৈরি হয়ে থাকে)
if not target_file:
    for root, _, files in os.walk("/kaggle/working/outputs"):
        for f in files:
            if "v8_3" in f and f.endswith(".wav"):
                target_file = os.path.join(root, f)
                break

if target_file and os.path.exists(target_file):
    filename = os.path.basename(target_file)
    with open(target_file, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    html = f'''
    <a download="{filename}" href="data:audio/wav;base64,{b64}">
        <button style="padding: 12px 24px; background-color: #20beff; color: white; border: none; border-radius: 6px; font-weight: bold; font-size: 16px; cursor: pointer;">
            ⬇️ Download {filename}
        </button>
    </a>
    '''
    print(f"✅ {filename} নামানোর জন্য প্রস্তুত!")
    display(HTML(html))
else:
    print("❌ ফাইলটি খুঁজে পাওয়া যায়নি। সেলের জেনারেশন সম্পন্ন হওয়া পর্যন্ত অপেক্ষা করুন।")

In [ ]:
# 📦 Step 2: Clone & Install OmniVoice Studio
!pip install -q uv
!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
!pip install -q uv
import os, shutil

# পুরোনো ফোল্ডার মুছে ফ্রেশ ক্লোন করা
if os.path.exists("/kaggle/working/omnivoice-studio"):
    shutil.rmtree("/kaggle/working/omnivoice-studio")

!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
import os, shutil

# ১. মূল /kaggle/working ডিরেক্টরিতে ফিরে যাওয়া (এরর সমাধানের মূল ধাপ)
%cd /kaggle/working

# ২. পুরোনো ভাঙা ফোল্ডার থাকলে মুছে ফেলা
if os.path.exists("/kaggle/working/omnivoice-studio"):
    shutil.rmtree("/kaggle/working/omnivoice-studio")

# ৩. ফ্রেশ ক্লোন ও ডিপেন্ডেন্সি ইনস্টল
!git clone https://github.com/debpalash/OmniVoice-Studio.git /kaggle/working/omnivoice-studio
%cd /kaggle/working/omnivoice-studio
!uv sync

In [ ]:
!curl -sf http://localhost:3900/health && echo "✅ ALIVE" || echo "❌ DEAD"

In [ ]:
import subprocess, time, os

%cd /kaggle/working/omnivoice-studio

log_file = open("/tmp/omnivoice.log", "w")
subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log_file, stderr=log_file)

print("⏳ Waiting 15s for server startup...")
time.sleep(15)

!curl -sf http://localhost:3900/health && echo "✅ READY" || echo "❌ FAILED"

In [ ]:
# =============================================================================
# 🎭 Step 4: V8.3 — ERROR-CORRECTED VERSION
# =============================================================================
# Fix 1: FileNotFoundError → cwd fallback + path auto-detect
# Fix 2: FileLink 404 → base64 download button
# Fix 3: Server restart → graceful fallback if already running
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_3_corrected"
API_URL = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"

# ═════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER AUTO-RESTART (ERROR-CORRECTED)
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        req = urllib.request.Request(HEALTH_URL)
        with urllib.request.urlopen(req, timeout=5) as resp:
            return resp.status == 200
    except:
        return False

def find_studio_dir():
    """OmniVoice Studio ডিরেক্টরি অটো-ডিটেক্ট করে।"""
    candidates = [
        "/kaggle/working/omnivoice-studio",
        "/kaggle/working/OmniVoice-Studio",
        os.getcwd(),  # current dir (if %cd already ran)
    ]
    # os.walk দিয়ে খোঁজা (যদি অন্য নামে থাকে)
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            candidates.insert(0, root)
            break
    for path in candidates:
        if os.path.exists(os.path.join(path, "backend", "main.py")):
            return path
    return None

def start_server():
    studio_dir = find_studio_dir()
    if not studio_dir:
        print("  ❌ OmniVoice Studio ফোল্ডার পাওয়া যায়নি!")
        print("     Step 2 (Clone & Install) সেলটি আগে রান করুন।")
        return False

    print(f"  📂 Studio Dir: {studio_dir}")
    print("  🔄 Starting server...")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(
        ["uv", "run", "python", "backend/main.py"],
        stdout=log_file, stderr=log_file,
        cwd=studio_dir
    )
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Still waiting... ({(i+1)*2}s)")
    print("  ❌ Server failed! Check: !cat /tmp/omnivoice.log")
    return False

# ─── Check & Restart ──────────────────────────────────────────────────────
print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)

if server_alive():
    print("  ✅ Server already running on :3900")
else:
    print("  ⚠️  Server not responding — restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 cells first.")

print()

# ═════════════════════════════════════════════════════════════════════════════
# 🎵 TTS FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files found!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    duration_s = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {duration_s:.1f}s audio)")
    return output_path

def gen_segment(text, voice="onyx", model="tts-1-hd", output_path="",
                speed=1.0, num_step=32, guidance_scale=2.0,
                class_temperature=0.0, postprocess_output=True, seed=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": model, "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": num_step, "guidance_scale": guidance_scale,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    if seed is not None:
        payload["seed"] = seed

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            dur = time.time() - t0
            kb = len(content) // 1024
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {kb:5d} KB │ {dur:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 📋 DOCUMENTED TAGS ONLY:
#    ✅ [laughter] [sigh] [breath] [pause] [pause 500ms] [pause 2s]
#    ✅ ... (ellipses) — (em dash) ! (exclamation) = prosody control
#    ❌ [cry] [whisper] [trembling] [excited] = unsupported
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"
FILENAME = "v8_3_final"

SEGMENTS = [
    {
        "tag": "01_narrator_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.95, "guidance_scale": 2.2, "class_temperature": 0.3,
    },
    {
        "tag": "02_doubt_laughter",
        "text": (
            "[laughter] Another universe? Seriously? [laughter] "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 1.05, "guidance_scale": 2.0, "class_temperature": 0.2,
    },
    {
        "tag": "03_excitement",
        "text": (
            "And suddenly — the impossible became real! "
            "We saw dinosaurs, still walking, beneath blood-red skies! "
            "We found another Earth where humanity was born on Mars!"
        ),
        "speed": 1.05, "guidance_scale": 2.5, "class_temperature": 0.4,
    },
    {
        "tag": "04_personal_shift",
        "text": (
            "But me? "
            "No. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85, "guidance_scale": 2.0, "class_temperature": 0.2,
        "postprocess_output": False,
    },
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.80, "guidance_scale": 2.0, "class_temperature": 0.3,
        "postprocess_output": False,
    },
    {
        "tag": "06_hospital_crying",
        "text": (
            "[sigh] "
            "I watched the hospital monitor... become... silent. "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time. "
            "[pause 2s] "
            "She never did."
        ),
        "speed": 0.75, "guidance_scale": 1.8, "class_temperature": 0.45,
        "postprocess_output": False,
    },
    {
        "tag": "07_pain",
        "text": (
            "I know she'll never answer. I know that! "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.82, "guidance_scale": 2.5, "class_temperature": 0.5,
    },
    {
        "tag": "08_plea",
        "text": (
            "[breath] Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.80, "guidance_scale": 1.8, "class_temperature": 0.2,
        "postprocess_output": False,
    },
    {
        "tag": "09_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88, "guidance_scale": 2.2, "class_temperature": 0.3,
    },
    {
        "tag": "10_ending",
        "text": (
            "[pause 1.5s] "
            "She didn't know I wasn't her son. "
            "I was a broken man... borrowing someone else's miracle."
        ),
        "speed": 0.82, "guidance_scale": 2.0, "class_temperature": 0.3,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

print("=" * 70)
print("🎭  VERSION 8.3 — ERROR-CORRECTED")
print("=" * 70)
print(f"🎙️  Voice     : {VOICE}")
print(f"🎬  Segments  : {len(SEGMENTS)}")
print(f"📂  Output    : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []

for seg in SEGMENTS:
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        model="tts-1-hd",
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        num_step=32,
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=seg.get("class_temperature", 0.0),
        postprocess_output=seg.get("postprocess_output", True),
        seed=seg.get("seed"),
    )
    if success:
        wav_order.append(wav_path)
        generated += 1
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating all segments...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(SEGMENTS)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD BUTTON (base64 — 404-proof)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl_name = os.path.basename(final_path)
    size_mb = os.path.getsize(final_path) / (1024 * 1024)
    html = f'''<a download="{dl_name}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl_name} ({size_mb:.1f} MB)
    </button></a>'''
    print(f"\n⬇️  নিচের বাটনে ক্লিক করে সরাসরি ডাউনলোড করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি — উপরের error চেক করুন।")


In [ ]:
# =============================================================================
# 🎭 V8.4 — SINGLE CHARACTER, MALE ONLY, NO TAGS SPOKEN
# =============================================================================
# Fix 1: Multiple characters → একই voice + consistent speed/temp
# Fix 2: Cancer part excited → grief-tuned settings
# Fix 3: [pause] spoken aloud → সব tag বাদ, শুধু punctuation
# Fix 4: Studio dir auto-detect + base64 download
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_4_single_char"
API_URL = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"

# ═════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! Step 2 আগে রান করুন।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()

# ═════════════════════════════════════════════════════════════════════════════
# 🎵 FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp): continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def gen_segment(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                class_temperature=0.3, postprocess_output=True):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd", "voice": voice, "input": text,
        "response_format": "wav", "speed": speed,
        "num_step": 32, "guidance_scale": guidance_scale,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — SINGLE CHARACTER, NO TAGS, PUNCTUATION ONLY
# ═════════════════════════════════════════════════════════════════════════════
# ❌ কোনো [pause], [sigh], [breath], [laughter] নেই — engine বলে ফেলে
# ✅ শুধু ... — ! ? এবং ছোট fragments দিয়ে pacing
# ✅ একটাই voice, একটাই character, narrow speed range (0.82-0.92)
# ✅ একটাই class_temperature (0.3) — voice consistency
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"
FILENAME = "v8_4_single_character"

SEGMENTS = [
    # ── 01: Opening — calm, intrigued narrator ──
    {
        "tag": "01_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 400},

    # ── 02: Doubt — light, slightly amused ──
    {
        "tag": "02_doubt",
        "text": (
            "Another universe? Seriously? "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 0.92,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 300},

    # ── 03: Wonder — building energy, but same voice ──
    {
        "tag": "03_wonder",
        "text": (
            "And suddenly — the impossible became real. "
            "We saw dinosaurs, still walking, beneath blood-red skies. "
            "We found another Earth where humanity was born on Mars."
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 600},

    # ── 04: Shift — tone drops, personal ──
    {
        "tag": "04_shift",
        "text": (
            "But me... "
            "no. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 800},

    # ── 05: Grief — slow, heavy, weight in every word ──
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 06: Hospital — deepest sadness, fragments ──
    {
        "tag": "06_hospital",
        "text": (
            "I watched the hospital monitor... become... silent. "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1500},

    # ── 07: She never did — separate for weight ──
    {
        "tag": "07_she_never_did",
        "text": "She never did.",
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 08: Pain — frustration leaking through grief ──
    {
        "tag": "08_pain",
        "text": (
            "I know she'll never answer. I know that. "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.85,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 800},

    # ── 09: Plea — quiet, desperate request ──
    {
        "tag": "09_plea",
        "text": (
            "Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1200},

    # ── 10: Warmth — relief, but restrained ──
    {
        "tag": "10_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 1000},

    # ── 11: Ending — bittersweet, soft ──
    {
        "tag": "11_ending",
        "text": (
            "She didn't know I wasn't her son. "
            "I was a broken man... "
            "borrowing someone else's miracle."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

speech_segs = [s for s in SEGMENTS if "text" in s]

print("=" * 70)
print("🎭  V8.4 — SINGLE CHARACTER, MALE ONLY")
print("=" * 70)
print(f"🎙️  Voice      : {VOICE} (onyx — deep male)")
print(f"🎬  Segments   : {len(speech_segs)} speech")
print(f"🚫  Tags       : NONE (no [pause], [sigh], [breath])")
print(f"✅  Pacing     : ... — ! fragments only")
print(f"📂  Output     : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []
sil_idx = 0

for seg in SEGMENTS:
    # Silence
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    # Speech
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")
    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=0.3,  # FIXED: একই temp সব segments-এ
        postprocess_output=seg.get("postprocess_output", True),
    )
    if success:
        wav_order.append(wav_path)
        generated += 1
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD (base64 — no 404)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024*1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print(f"\n⬇️  নিচের বাটনে ক্লিক করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V8.5 — SINGLE CHARACTER FIX (DOCUMENTATION-BASED)
# =============================================================================
#
# 🔍 ROOT CAUSE (docs/expressive-speech.md থেকে):
# ─────────────────────────────────────────────────
# "Seed — unpinned by default, so every render differs."
#
# যখন seed unpinned থাকে, প্রতিটা API call ভিন্ন random state ব্যবহার করে।
# তাই প্রতিটা segment ভিন্ন speaker identity পায় — কোনোটা পুরুষ, কোনোটা
# মহিলা। এটাই "multiple characters" এবং "female voice" সমস্যার কারণ।
#
# ✅ FIX: সব segments-এ একই seed ব্যবহার → একই speaker identity
#
# 📋 ALSO FIXED:
# - [pause] ও tags: docs বলে [pause] সব engine-এ কাজ করে (stitched silence)
#   কিন্তু real test-এ কথা বলে ফেলে → তাই শুধু manual silence WAV ব্যবহার
# - [laughter], [sigh]: docs বলে default engine-এ কাজ করে, কিন্তু
#   voice consistency-র জন্য এখানে বাদ রেখেছি
# - download: base64 button (404-proof)
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_5_fixed"
API_URL = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"

# ═════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! Step 2 আগে রান করুন।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()

# ═════════════════════════════════════════════════════════════════════════════
# 🎵 FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp): continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_segment(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                class_temperature=0.3, postprocess_output=True, seed=42):
    """
    seed প্যারামিটার দিয়ে voice identity lock করা হয়।
    docs: "Keep this seed... pins reference + seed, making the voice bit-reproducible"
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,  # ← KEY FIX: same seed = same voice identity
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — একটাই CHARACTER, একটাই SEED
# ═════════════════════════════════════════════════════════════════════════════
#
# 📖 docs/expressive-speech.md:
#    - [pause] = stitched silence (every engine) — কিন্তু কথা বলে ফেলছে
#      → তাই manual silence WAV ব্যবহার করছি
#    - Punctuation ... — ! = prosody control (every engine) ✅
#    - seed pin = same speaker identity ✅
#    - class_temperature 0.3 fixed = consistent variation ✅
#
# ❌ কোনো bracket tag নেই
# ❌ কোনো female voice নেই
# ✅ একই seed (42) সব segments-এ
# ✅ একই voice ("onyx") সব segments-এ
# ✅ একই class_temperature (0.3) সব segments-এ
# ═════════════════════════════════════════════════════════════════════════════

VOICE = "onyx"            # গভীর মেল ভয়েস
SEED = 42                 # ← একই seed = একই character সবসময়
FILENAME = "v8_5_single_character"

SEGMENTS = [
    # ── 01: Opening — calm anticipation ──
    {
        "tag": "01_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 400},

    # ── 02: Doubt — slight amusement ──
    {
        "tag": "02_doubt",
        "text": (
            "Another universe? Seriously? "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 0.92,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 300},

    # ── 03: Wonder — building but controlled ──
    {
        "tag": "03_wonder",
        "text": (
            "And suddenly — the impossible became real. "
            "We saw dinosaurs, still walking, beneath blood-red skies. "
            "We found another Earth where humanity was born on Mars."
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 700},

    # ── 04: Shift — personal, tone drops ──
    {
        "tag": "04_shift",
        "text": (
            "But me... "
            "no. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
    {"silence_ms": 800},

    # ── 05: Grief — slow, heavy ──
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 06: Hospital — deepest sadness ──
    {
        "tag": "06_hospital",
        "text": (
            "I watched the hospital monitor... become... silent. "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1500},

    # ── 07: Weight — one line, devastating ──
    {
        "tag": "07_she_never_did",
        "text": "She never did.",
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 08: Pain — grief meets frustration ──
    {
        "tag": "08_pain",
        "text": (
            "I know she'll never answer. I know that. "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.85,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 800},

    # ── 09: Plea — quiet, desperate ──
    {
        "tag": "09_plea",
        "text": (
            "Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1200},

    # ── 10: Warmth — relief, restrained ──
    {
        "tag": "10_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 1000},

    # ── 11: Ending — bittersweet, soft ──
    {
        "tag": "11_ending",
        "text": (
            "She didn't know I wasn't her son. "
            "I was a broken man... "
            "borrowing someone else's miracle."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

speech_segs = [s for s in SEGMENTS if "text" in s]

print("=" * 70)
print("🎭  V8.5 — SINGLE CHARACTER (SEED-LOCKED)")
print("=" * 70)
print(f"🎙️  Voice      : {VOICE} (deep male)")
print(f"🔒  Seed       : {SEED} (same voice every segment)")
print(f"🎬  Segments   : {len(speech_segs)} speech")
print(f"🚫  Tags       : NONE")
print(f"✅  Pacing     : ... — ! fragments + manual silence")
print(f"📂  Output     : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []
sil_idx = 0

for seg in SEGMENTS:
    # Silence
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    # Speech — same seed every time
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")
    success = gen_segment(
        text=seg["text"],
        voice=VOICE,
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=0.3,  # fixed for consistency
        postprocess_output=seg.get("postprocess_output", True),
        seed=SEED,  # ← KEY: same seed = same character
    )
    if success:
        wav_order.append(wav_path)
        generated += 1
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{VOICE}.wav")
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD (base64 — no 404)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024*1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print(f"\n⬇️  নিচের বাটনে ক্লিক করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:

    import urllib.request, json
    req = urllib.request.Request("http://localhost:3900/v1/audio/voices")
    with urllib.request.urlopen(req, timeout=10) as resp:
        voices = json.loads(resp.read())
    print(json.dumps(voices, indent=2))


In [ ]:
 import urllib.request, json
    payload = json.dumps({
        "model": "tts-1-hd", "voice": "demo0001",
        "input": "Three years ago, cancer stole my mother. No warning. No mercy.",
        "response_format": "wav", "speed": 0.85
    }).encode("utf-8")
    req = urllib.request.Request("http://localhost:3900/v1/audio/speech",
        data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        with open("test_demo0001.wav", "wb") as f:
            f.write(resp.read())
    print("✅ Done! Play it below:")

    import IPython.display as ipd
    ipd.display(ipd.Audio("test_demo0001.wav"))

In [ ]:


    import urllib.request, json
    payload = json.dumps({
        "model": "tts-1-hd", "voice": "demo0001",
        "input": "Three years ago, cancer stole my mother. No warning. No mercy.",
        "response_format": "wav", "speed": 0.85
    }).encode("utf-8")
    req = urllib.request.Request("http://localhost:3900/v1/audio/speech",
        data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        with open("test_demo0001.wav", "wb") as f:
            f.write(resp.read())
    print("✅ Done! Play it below:")

    import IPython.display as ipd
    ipd.display(ipd.Audio("test_demo0001.wav"))



In [ ]:
    # 🔍 Voice Design API endpoint খুঁজে বের করা
    impimport urllib.request, json
    reqreq = urllib.request.Request("http://localhost:3900/openapi.json")
    witwith urllib.request.urlopen(req, timeout=10) as resp:
           spec = json.loads(resp.read())

    # শু# শুধু voice-related endpoints দেখাও
    for pfor path, methods in spec.get("paths", {}).items():
        i    if "voice" in path.lower() or "design" in path.lower():
            for method in methods:
                print(f"  {method.upper():6s} {path}")



In [ ]:
# =============================================================================
# 🎭 V8.6 — VOICE-LOCKED: নিজের ভয়েস + প্রথম জেনারেশন = রেফারেন্স
# =============================================================================
#
# 🆕 V8.5 থেকে কি বদলেছে:
# ─────────────────────────
# 1. CUSTOM VOICE SUPPORT:
#    → নিজের ভয়েস WAV আপলোড করুন → voice profile তৈরি হবে
#    → সেই profile দিয়ে সব segment generate হবে
#    → character change সমস্যা SOLVE — কারণ আপনার নিজের ভয়েস
#      থেকে timbre + delivery clone হবে
#
# 2. FIRST-GEN-AS-REFERENCE:
#    → প্রথম segment generate হওয়ার পর সেটাই reference WAV
#    → বাকি সব segment-এ সেই reference ব্যবহার হয়
#    → ফলে voice identity 100% consistent থাকে
#
# 📖 docs/expressive-speech.md:
#    "The reference clip is a performance direction. Zero-shot
#     cloning mirrors the delivery of the reference, not just the timbre"
#
# ✅ seed lock (V8.5 থেকে) + voice profile lock (নতুন)
# ✅ first-gen reference lock (নতুন)
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64, io
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_6_voice_locked"
API_URL = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"
VOICES_URL = "http://localhost:3900/v1/audio/voices"

# ═════════════════════════════════════════════════════════════════════════════
# ⚙️ CONFIGURATION — এখানে আপনার সেটিংস দিন
# ═════════════════════════════════════════════════════════════════════════════

# ─── OPTION 1: নিজের ভয়েস দিতে চাইলে ────────────────────────────────────
# আপনার WAV/MP3 ফাইলের পাথ দিন (৩-১০ সেকেন্ড, পরিষ্কার, একজনের গলা)
# Kaggle-এ ফাইল আপলোড করুন → /kaggle/input/ ফোল্ডারে থাকবে
#
# উদাহরণ:
#   CUSTOM_VOICE_PATH = "/kaggle/input/my-voice/my_narration.wav"
#
# দিতে না চাইলে None রাখুন — তখন preset voice (onyx) ব্যবহার হবে
CUSTOM_VOICE_PATH = None  # ← আপনার WAV ফাইলের পাথ দিন

# আপনার ভয়েস প্রোফাইলের নাম (যেকোনো ইউনিক নাম)
CUSTOM_VOICE_NAME = "my_narrator"

# reference clip-এ কি বলা হয়েছে তার transcript (ভালো cloning-এর জন্য)
# যদি জানা না থাকে তাহলে "" রাখুন
CUSTOM_VOICE_TRANSCRIPT = ""

# ─── OPTION 2: Preset voice ব্যবহার করতে চাইলে ───────────────────────────
PRESET_VOICE = "onyx"  # onyx=গভীর মেল, echo=মধ্যম, alloy=হালকা

# ─── OPTION 3: প্রথম জেনারেশনকে reference হিসেবে ব্যবহার ─────────────────
# True = প্রথম segment generate হলে, সেটা voice profile হিসেবে upload হবে
#        এবং বাকি সব segment সেই voice দিয়ে generate হবে
# False = শুধু seed lock (V8.5 এর মতো)
USE_FIRST_GEN_AS_REFERENCE = True

# ─── Common settings ──────────────────────────────────────────────────────
SEED = 42
FILENAME = "v8_6_voice_locked"

# ═════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! Step 2 আগে রান করুন।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()

# ═════════════════════════════════════════════════════════════════════════════
# 🎙️ VOICE PROFILE MANAGEMENT
# ═════════════════════════════════════════════════════════════════════════════

def list_voice_profiles():
    """সার্ভারে কোন কোন voice profile আছে সেটা দেখায়"""
    try:
        req = urllib.request.Request(VOICES_URL)
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode())
            voices = data.get("voices", [])
            return {v.get("voice_id", v.get("name", "")): v for v in voices}
    except Exception as e:
        print(f"  ⚠️ Voice list পাওয়া যায়নি: {e}")
        return {}

def upload_voice_profile(audio_path, profile_name, ref_text=""):
    """
    নিজের WAV ফাইল থেকে voice profile তৈরি করে।

    POST /v1/audio/voices
    - multipart/form-data
    - fields: name, consent, ref_text (optional)
    - file: audio_sample
    """
    if not os.path.exists(audio_path):
        print(f"  ❌ ফাইল পাওয়া যায়নি: {audio_path}")
        return None

    file_size = os.path.getsize(audio_path)
    print(f"  📎 আপলোড করা হচ্ছে: {os.path.basename(audio_path)} ({file_size//1024} KB)")

    # multipart/form-data তৈরি করি (urllib দিয়ে)
    boundary = "----OmniVoiceBoundary" + str(int(time.time()))

    body = b""

    # name field
    body += f"--{boundary}\r\n".encode()
    body += b"Content-Disposition: form-data; name=\"name\"\r\n\r\n"
    body += f"{profile_name}\r\n".encode()

    # consent field
    body += f"--{boundary}\r\n".encode()
    body += b"Content-Disposition: form-data; name=\"consent\"\r\n\r\n"
    body += b"yes\r\n"

    # ref_text field (যদি দেওয়া থাকে)
    if ref_text:
        body += f"--{boundary}\r\n".encode()
        body += b"Content-Disposition: form-data; name=\"ref_text\"\r\n\r\n"
        body += f"{ref_text}\r\n".encode()

    # audio file
    filename = os.path.basename(audio_path)
    ext = os.path.splitext(filename)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")

    body += f"--{boundary}\r\n".encode()
    body += f'Content-Disposition: form-data; name="audio_sample"; filename="{filename}"\r\n'.encode()
    body += f"Content-Type: {mime}\r\n\r\n".encode()

    with open(audio_path, "rb") as f:
        body += f.read()
    body += b"\r\n"

    # closing boundary
    body += f"--{boundary}--\r\n".encode()

    req = urllib.request.Request(
        VOICES_URL,
        data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            result = json.loads(resp.read().decode())
            voice_id = result.get("voice_id", result.get("id", profile_name))
            print(f"  ✅ Voice profile তৈরি হয়েছে: {voice_id}")
            return voice_id
    except urllib.error.HTTPError as e:
        error_body = e.read().decode() if hasattr(e, 'read') else str(e)
        print(f"  ❌ Upload failed (HTTP {e.code}): {error_body}")

        # যদি profile already exist করে, সেই নামেই ব্যবহার করি
        if e.code in (409, 422):
            print(f"  ℹ️  Profile '{profile_name}' হয়তো আগে থেকেই আছে। সেটাই ব্যবহার হবে।")
            return profile_name
        return None
    except Exception as e:
        print(f"  ❌ Upload error: {e}")
        return None

def upload_generated_as_reference(wav_path, profile_name):
    """
    প্রথম generated WAV কে voice profile হিসেবে আপলোড করে।
    এতে বাকি সব segment একই voice identity পাবে।
    """
    print(f"\n  🔄 প্রথম জেনারেশন → voice reference হিসেবে আপলোড হচ্ছে...")

    # generated WAV-এর info দেখি
    try:
        with wave.open(wav_path, "rb") as wf:
            dur = wf.getnframes() / wf.getframerate()
            print(f"  📊 Reference clip: {dur:.1f}s")
    except:
        pass

    return upload_voice_profile(wav_path, profile_name)

# ═════════════════════════════════════════════════════════════════════════════
# 🎵 CORE FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp): continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_segment(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                class_temperature=0.3, postprocess_output=True, seed=42):
    """
    seed প্যারামিটার দিয়ে voice identity lock করা হয়।
    voice = preset name (onyx/echo) অথবা custom profile ID
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — একটাই CHARACTER, VOICE-LOCKED
# ═════════════════════════════════════════════════════════════════════════════

SEGMENTS = [
    # ── 01: Opening — calm anticipation ──
    {
        "tag": "01_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 400},

    # ── 02: Doubt — slight amusement ──
    {
        "tag": "02_doubt",
        "text": (
            "Another universe? Seriously? "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 0.92,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 300},

    # ── 03: Wonder — building but controlled ──
    {
        "tag": "03_wonder",
        "text": (
            "And suddenly — the impossible became real. "
            "We saw dinosaurs, still walking, beneath blood-red skies. "
            "We found another Earth where humanity was born on Mars."
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 700},

    # ── 04: Shift — personal, tone drops ──
    {
        "tag": "04_shift",
        "text": (
            "But me... "
            "no. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
    {"silence_ms": 800},

    # ── 05: Grief — slow, heavy ──
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 06: Hospital — deepest sadness ──
    {
        "tag": "06_hospital",
        "text": (
            "I watched the hospital monitor... become... silent. "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1500},

    # ── 07: Weight — one line, devastating ──
    {
        "tag": "07_she_never_did",
        "text": "She never did.",
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 08: Pain — grief meets frustration ──
    {
        "tag": "08_pain",
        "text": (
            "I know she'll never answer. I know that. "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.85,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 800},

    # ── 09: Plea — quiet, desperate ──
    {
        "tag": "09_plea",
        "text": (
            "Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1200},

    # ── 10: Warmth — relief, restrained ──
    {
        "tag": "10_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 1000},

    # ── 11: Ending — bittersweet, soft ──
    {
        "tag": "11_ending",
        "text": (
            "She didn't know I wasn't her son. "
            "I was a broken man... "
            "borrowing someone else's miracle."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 VOICE SETUP → GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

speech_segs = [s for s in SEGMENTS if "text" in s]

# ─── Step 1: Voice নির্ধারণ ──────────────────────────────────────────────
print("=" * 70)
print("🎙️  VOICE SETUP")
print("=" * 70)

# বর্তমান voice profiles দেখি
existing_profiles = list_voice_profiles()
if existing_profiles:
    print(f"  📋 সার্ভারে {len(existing_profiles)} টি voice profile আছে:")
    for vid, vinfo in existing_profiles.items():
        print(f"     • {vid} ({vinfo.get('name', 'N/A')})")

active_voice = PRESET_VOICE  # default

# OPTION 1: নিজের ভয়েস আপলোড
if CUSTOM_VOICE_PATH:
    print(f"\n  🎤 আপনার ভয়েস আপলোড হচ্ছে: {CUSTOM_VOICE_PATH}")
    uploaded_id = upload_voice_profile(
        CUSTOM_VOICE_PATH,
        CUSTOM_VOICE_NAME,
        CUSTOM_VOICE_TRANSCRIPT
    )
    if uploaded_id:
        active_voice = uploaded_id
        print(f"  ✅ Custom voice সেট হয়েছে: {active_voice}")
    else:
        print(f"  ⚠️ Upload ব্যর্থ — preset voice '{PRESET_VOICE}' ব্যবহার হবে")
else:
    print(f"\n  ℹ️  Custom voice দেওয়া হয়নি — preset '{PRESET_VOICE}' ব্যবহার হবে")
    if USE_FIRST_GEN_AS_REFERENCE:
        print(f"  🔄 প্রথম segment generate হলে সেটা reference হিসেবে ব্যবহার হবে")

print()

# ─── Step 2: Generate ─────────────────────────────────────────────────────
print("=" * 70)
print("🎭  V8.6 — VOICE-LOCKED GENERATION")
print("=" * 70)
print(f"🎙️  Voice      : {active_voice}")
print(f"🔒  Seed       : {SEED}")
print(f"🔄  First→Ref  : {'ON' if USE_FIRST_GEN_AS_REFERENCE and not CUSTOM_VOICE_PATH else 'OFF'}")
print(f"🎬  Segments   : {len(speech_segs)} speech")
print(f"📂  Output     : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []
sil_idx = 0
first_speech_generated = False
reference_voice_name = f"v86_ref_{SEED}"

for seg in SEGMENTS:
    # Silence
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    # Speech
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    success = gen_segment(
        text=seg["text"],
        voice=active_voice,
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=0.3,
        postprocess_output=seg.get("postprocess_output", True),
        seed=SEED,
    )

    if success:
        wav_order.append(wav_path)
        generated += 1

        # ─── FIRST-GEN-AS-REFERENCE LOGIC ────────────────────────────
        # প্রথম সফল generation-এর পর সেটাকে voice profile হিসেবে upload করি
        # বাকি সব segment এই profile দিয়ে generate হবে
        if (USE_FIRST_GEN_AS_REFERENCE
                and not CUSTOM_VOICE_PATH
                and not first_speech_generated):
            first_speech_generated = True
            ref_id = upload_generated_as_reference(wav_path, reference_voice_name)
            if ref_id:
                old_voice = active_voice
                active_voice = ref_id
                print(f"  🔁 Voice switched: {old_voice} → {active_voice}")
                print(f"     বাকি সব segment এই reference দিয়ে generate হবে\n")
            else:
                print(f"  ⚠️ Reference upload ব্যর্থ — seed lock দিয়ে চলবে\n")
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = os.path.join(BASE_OUTPUT_DIR, f"{FILENAME}_{SEED}.wav")
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")

# ─── Summary ──────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("📋 VOICE CONSISTENCY REPORT:")
if CUSTOM_VOICE_PATH:
    print(f"  🎤 Custom voice ব্যবহৃত: {CUSTOM_VOICE_PATH}")
    print(f"  📛 Profile name: {CUSTOM_VOICE_NAME}")
elif USE_FIRST_GEN_AS_REFERENCE and first_speech_generated:
    print(f"  🔄 First-gen reference ব্যবহৃত")
    print(f"  📛 Reference profile: {reference_voice_name}")
else:
    print(f"  🔒 Seed-lock only (V8.5 mode)")
print(f"  🎙️  Final voice: {active_voice}")
print(f"  🌱 Seed: {SEED}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD (base64 — no 404)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024*1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print(f"\n⬇️  নিচের বাটনে ক্লিক করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V8.6 — VOICE-LOCKED: নিজের ভয়েস + প্রথম জেনারেশন = রেফারেন্স
# =============================================================================
#
# 🆕 V8.5 থেকে কি বদলেছে:
# ─────────────────────────
# 1. CUSTOM VOICE SUPPORT:
#    → নিজের ভয়েস WAV আপলোড করুন → voice profile তৈরি হবে
#    → সেই profile দিয়ে সব segment generate হবে
#    → character change সমস্যা SOLVE — কারণ আপনার নিজের ভয়েস
#      থেকে timbre + delivery clone হবে
#
# 2. FIRST-GEN-AS-REFERENCE:
#    → প্রথম segment generate হওয়ার পর সেটাই reference WAV
#    → বাকি সব segment-এ সেই reference ব্যবহার হয়
#    → ফলে voice identity 100% consistent থাকে
#
# 📖 docs/expressive-speech.md:
#    "The reference clip is a performance direction. Zero-shot
#     cloning mirrors the delivery of the reference, not just the timbre"
#
# ✅ seed lock (V8.5 থেকে) + voice profile lock (নতুন)
# ✅ first-gen reference lock (নতুন)
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64, io, glob
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_6_voice_locked"
API_URL = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"
VOICES_URL = "http://localhost:3900/v1/audio/voices"

# ═════════════════════════════════════════════════════════════════════════════
# ⚙️ CONFIGURATION — এখানে আপনার সেটিংস দিন
# ═════════════════════════════════════════════════════════════════════════════

# ─── OPTION 1: নিজের ভয়েস দিতে চাইলে ────────────────────────────────────
# আপনার WAV/MP3 ফাইলের পাথ দিন (৩-১০ সেকেন্ড, পরিষ্কার, একজনের গলা)
# Kaggle-এ ফাইল আপলোড করুন → /kaggle/input/ ফোল্ডারে থাকবে
#
# উদাহরণ:
#   CUSTOM_VOICE_PATH = "/kaggle/input/my-voice/my_narration.wav"
#
# দিতে না চাইলে None রাখুন — তখন preset voice (onyx) ব্যবহার হবে
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"  # ← আপনার ভয়েস

# আপনার ভয়েস প্রোফাইলের নাম (যেকোনো ইউনিক নাম)
CUSTOM_VOICE_NAME = "my_narrator"

# reference clip-এ কি বলা হয়েছে তার transcript (ভালো cloning-এর জন্য)
# যদি জানা না থাকে তাহলে "" রাখুন
CUSTOM_VOICE_TRANSCRIPT = ""

# ─── OPTION 2: Preset voice ব্যবহার করতে চাইলে ───────────────────────────
PRESET_VOICE = "onyx"  # onyx=গভীর মেল, echo=মধ্যম, alloy=হালকা

# ─── OPTION 3: প্রথম জেনারেশনকে reference হিসেবে ব্যবহার ─────────────────
# True = প্রথম segment generate হলে, সেটা voice profile হিসেবে upload হবে
#        এবং বাকি সব segment সেই voice দিয়ে generate হবে
# False = শুধু seed lock (V8.5 এর মতো)
USE_FIRST_GEN_AS_REFERENCE = True

# ─── Common settings ──────────────────────────────────────────────────────
SEED = 42
FILENAME = "v8_6_voice_locked"

# ═════════════════════════════════════════════════════════════════════════════
# 🔍 AUTO-DISCOVER: ভয়েস ফাইল খুঁজে বের করা
# ═════════════════════════════════════════════════════════════════════════════
def auto_discover_voice(given_path):
    """
    যদি দেওয়া পাথে ফাইল না পাওয়া যায়, তাহলে /kaggle/input/ এ
    সব MP3/WAV ফাইল খুঁজে দেখায় এবং প্রথমটা ব্যবহার করে।
    """
    if given_path and os.path.exists(given_path):
        return given_path

    if not given_path:
        return None

    print(f"  ⚠️ দেওয়া পাথে ফাইল নেই: {given_path}")
    print(f"  🔍 /kaggle/input/ এ অডিও ফাইল খুঁজছি...")

    # /kaggle/input/ এ সব audio file খুঁজি
    audio_files = []
    search_root = "/kaggle/input"
    if os.path.exists(search_root):
        for root, dirs, files in os.walk(search_root):
            for f in files:
                if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))

    if not audio_files:
        print(f"  ❌ কোনো অডিও ফাইল পাওয়া যায়নি /kaggle/input/ এ")
        print(f"  💡 Kaggle notebook-এ: Add Data → Upload → আপনার MP3/WAV দিন")
        return None

    print(f"  📋 পাওয়া গেছে {len(audio_files)} টি অডিও ফাইল:")
    for i, af in enumerate(audio_files):
        size_kb = os.path.getsize(af) // 1024
        print(f"     {i+1}. {af} ({size_kb} KB)")

    # প্রথম audio file ব্যবহার করি
    chosen = audio_files[0]
    print(f"  ✅ ব্যবহার করা হচ্ছে: {chosen}")
    return chosen

# Auto-discover চালাই
CUSTOM_VOICE_PATH = auto_discover_voice(CUSTOM_VOICE_PATH)

# ═════════════════════════════════════════════════════════════════════════════
# 📁 AUTO-VERSIONING: প্রতিবার নতুন নামে save
# ═════════════════════════════════════════════════════════════════════════════
def get_next_version(base_dir, base_name):
    """
    existing ফাইল চেক করে পরবর্তী version number বের করে।
    v8_6_voice_locked_v1.wav → v8_6_voice_locked_v2.wav → ...
    """
    os.makedirs(base_dir, exist_ok=True)
    version = 1
    while True:
        candidate = os.path.join(base_dir, f"{base_name}_v{version}.wav")
        if not os.path.exists(candidate):
            return version, candidate
        version += 1

RUN_VERSION, FINAL_OUTPUT_PATH = get_next_version(BASE_OUTPUT_DIR, FILENAME)
print(f"📁 Auto-version: v{RUN_VERSION} (নতুন ফাইল: {os.path.basename(FINAL_OUTPUT_PATH)})")
print()

# ═════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! Step 2 আগে রান করুন।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()

# ═════════════════════════════════════════════════════════════════════════════
# 🎙️ VOICE PROFILE MANAGEMENT
# ═════════════════════════════════════════════════════════════════════════════

def list_voice_profiles():
    """সার্ভারে কোন কোন voice profile আছে সেটা দেখায়"""
    try:
        req = urllib.request.Request(VOICES_URL)
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode())
            voices = data.get("voices", [])
            return {v.get("voice_id", v.get("name", "")): v for v in voices}
    except Exception as e:
        print(f"  ⚠️ Voice list পাওয়া যায়নি: {e}")
        return {}

def upload_voice_profile(audio_path, profile_name, ref_text=""):
    """
    নিজের WAV ফাইল থেকে voice profile তৈরি করে।

    POST /v1/audio/voices
    - multipart/form-data
    - fields: name, consent, ref_text (optional)
    - file: audio_sample
    """
    if not os.path.exists(audio_path):
        print(f"  ❌ ফাইল পাওয়া যায়নি: {audio_path}")
        return None

    file_size = os.path.getsize(audio_path)
    print(f"  📎 আপলোড করা হচ্ছে: {os.path.basename(audio_path)} ({file_size//1024} KB)")

    # multipart/form-data তৈরি করি (urllib দিয়ে)
    boundary = "----OmniVoiceBoundary" + str(int(time.time()))

    body = b""

    # name field
    body += f"--{boundary}\r\n".encode()
    body += b"Content-Disposition: form-data; name=\"name\"\r\n\r\n"
    body += f"{profile_name}\r\n".encode()

    # consent field
    body += f"--{boundary}\r\n".encode()
    body += b"Content-Disposition: form-data; name=\"consent\"\r\n\r\n"
    body += b"yes\r\n"

    # ref_text field (যদি দেওয়া থাকে)
    if ref_text:
        body += f"--{boundary}\r\n".encode()
        body += b"Content-Disposition: form-data; name=\"ref_text\"\r\n\r\n"
        body += f"{ref_text}\r\n".encode()

    # audio file
    filename = os.path.basename(audio_path)
    ext = os.path.splitext(filename)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")

    body += f"--{boundary}\r\n".encode()
    body += f'Content-Disposition: form-data; name="audio_sample"; filename="{filename}"\r\n'.encode()
    body += f"Content-Type: {mime}\r\n\r\n".encode()

    with open(audio_path, "rb") as f:
        body += f.read()
    body += b"\r\n"

    # closing boundary
    body += f"--{boundary}--\r\n".encode()

    req = urllib.request.Request(
        VOICES_URL,
        data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
        method="POST"
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            result = json.loads(resp.read().decode())
            voice_id = result.get("voice_id", result.get("id", profile_name))
            print(f"  ✅ Voice profile তৈরি হয়েছে: {voice_id}")
            return voice_id
    except urllib.error.HTTPError as e:
        error_body = e.read().decode() if hasattr(e, 'read') else str(e)
        print(f"  ❌ Upload failed (HTTP {e.code}): {error_body}")

        # যদি profile already exist করে, সেই নামেই ব্যবহার করি
        if e.code in (409, 422):
            print(f"  ℹ️  Profile '{profile_name}' হয়তো আগে থেকেই আছে। সেটাই ব্যবহার হবে।")
            return profile_name
        return None
    except Exception as e:
        print(f"  ❌ Upload error: {e}")
        return None

def upload_generated_as_reference(wav_path, profile_name):
    """
    প্রথম generated WAV কে voice profile হিসেবে আপলোড করে।
    এতে বাকি সব segment একই voice identity পাবে।
    """
    print(f"\n  🔄 প্রথম জেনারেশন → voice reference হিসেবে আপলোড হচ্ছে...")

    # generated WAV-এর info দেখি
    try:
        with wave.open(wav_path, "rb") as wf:
            dur = wf.getnframes() / wf.getframerate()
            print(f"  📊 Reference clip: {dur:.1f}s")
    except:
        pass

    return upload_voice_profile(wav_path, profile_name)

# ═════════════════════════════════════════════════════════════════════════════
# 🎵 CORE FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp): continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ No WAV files!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_segment(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                class_temperature=0.3, postprocess_output=True, seed=42):
    """
    seed প্যারামিটার দিয়ে voice identity lock করা হয়।
    voice = preset name (onyx/echo) অথবা custom profile ID
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:40s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False

# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — একটাই CHARACTER, VOICE-LOCKED
# ═════════════════════════════════════════════════════════════════════════════

SEGMENTS = [
    # ── 01: Opening — calm anticipation ──
    {
        "tag": "01_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space — "
            "but one that crossed possibilities?"
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 400},

    # ── 02: Doubt — slight amusement ──
    {
        "tag": "02_doubt",
        "text": (
            "Another universe? Seriously? "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 0.92,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 300},

    # ── 03: Wonder — building but controlled ──
    {
        "tag": "03_wonder",
        "text": (
            "And suddenly — the impossible became real. "
            "We saw dinosaurs, still walking, beneath blood-red skies. "
            "We found another Earth where humanity was born on Mars."
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 700},

    # ── 04: Shift — personal, tone drops ──
    {
        "tag": "04_shift",
        "text": (
            "But me... "
            "no. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
    {"silence_ms": 800},

    # ── 05: Grief — slow, heavy ──
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 06: Hospital — deepest sadness ──
    {
        "tag": "06_hospital",
        "text": (
            "I watched the hospital monitor... become... silent. "
            "I held her hand — hoping — just hoping — "
            "she'd squeeze mine... one... last... time."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1500},

    # ── 07: Weight — one line, devastating ──
    {
        "tag": "07_she_never_did",
        "text": "She never did.",
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # ── 08: Pain — grief meets frustration ──
    {
        "tag": "08_pain",
        "text": (
            "I know she'll never answer. I know that. "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.85,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 800},

    # ── 09: Plea — quiet, desperate ──
    {
        "tag": "09_plea",
        "text": (
            "Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1200},

    # ── 10: Warmth — relief, restrained ──
    {
        "tag": "10_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 1000},

    # ── 11: Ending — bittersweet, soft ──
    {
        "tag": "11_ending",
        "text": (
            "She didn't know I wasn't her son. "
            "I was a broken man... "
            "borrowing someone else's miracle."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
]

# ═════════════════════════════════════════════════════════════════════════════
# 🚀 VOICE SETUP → GENERATE → CONCATENATE → DOWNLOAD
# ═════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

speech_segs = [s for s in SEGMENTS if "text" in s]

# ─── Step 1: Voice নির্ধারণ ──────────────────────────────────────────────
print("=" * 70)
print("🎙️  VOICE SETUP")
print("=" * 70)

# বর্তমান voice profiles দেখি
existing_profiles = list_voice_profiles()
if existing_profiles:
    print(f"  📋 সার্ভারে {len(existing_profiles)} টি voice profile আছে:")
    for vid, vinfo in existing_profiles.items():
        print(f"     • {vid} ({vinfo.get('name', 'N/A')})")

active_voice = PRESET_VOICE  # default

# OPTION 1: নিজের ভয়েস আপলোড
if CUSTOM_VOICE_PATH:
    print(f"\n  🎤 আপনার ভয়েস আপলোড হচ্ছে: {CUSTOM_VOICE_PATH}")
    uploaded_id = upload_voice_profile(
        CUSTOM_VOICE_PATH,
        CUSTOM_VOICE_NAME,
        CUSTOM_VOICE_TRANSCRIPT
    )
    if uploaded_id:
        active_voice = uploaded_id
        print(f"  ✅ Custom voice সেট হয়েছে: {active_voice}")
    else:
        print(f"  ⚠️ Upload ব্যর্থ — preset voice '{PRESET_VOICE}' ব্যবহার হবে")
else:
    print(f"\n  ℹ️  Custom voice দেওয়া হয়নি — preset '{PRESET_VOICE}' ব্যবহার হবে")
    if USE_FIRST_GEN_AS_REFERENCE:
        print(f"  🔄 প্রথম segment generate হলে সেটা reference হিসেবে ব্যবহার হবে")

print()

# ─── Step 2: Generate ─────────────────────────────────────────────────────
print("=" * 70)
print("🎭  V8.6 — VOICE-LOCKED GENERATION")
print("=" * 70)
print(f"🎙️  Voice      : {active_voice}")
print(f"🔒  Seed       : {SEED}")
print(f"🔄  First→Ref  : {'ON' if USE_FIRST_GEN_AS_REFERENCE and not CUSTOM_VOICE_PATH else 'OFF'}")
print(f"🎬  Segments   : {len(speech_segs)} speech")
print(f"📂  Output     : {BASE_OUTPUT_DIR}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []
sil_idx = 0
first_speech_generated = False
reference_voice_name = f"v86_ref_{SEED}"

for seg in SEGMENTS:
    # Silence
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    # Speech
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")

    success = gen_segment(
        text=seg["text"],
        voice=active_voice,
        output_path=wav_path,
        speed=seg.get("speed", 0.88),
        guidance_scale=seg.get("guidance_scale", 2.0),
        class_temperature=0.3,
        postprocess_output=seg.get("postprocess_output", True),
        seed=SEED,
    )

    if success:
        wav_order.append(wav_path)
        generated += 1

        # ─── FIRST-GEN-AS-REFERENCE LOGIC ────────────────────────────
        # প্রথম সফল generation-এর পর সেটাকে voice profile হিসেবে upload করি
        # বাকি সব segment এই profile দিয়ে generate হবে
        if (USE_FIRST_GEN_AS_REFERENCE
                and not CUSTOM_VOICE_PATH
                and not first_speech_generated):
            first_speech_generated = True
            ref_id = upload_generated_as_reference(wav_path, reference_voice_name)
            if ref_id:
                old_voice = active_voice
                active_voice = ref_id
                print(f"  🔁 Voice switched: {old_voice} → {active_voice}")
                print(f"     বাকি সব segment এই reference দিয়ে generate হবে\n")
            else:
                print(f"  ⚠️ Reference upload ব্যর্থ — seed lock দিয়ে চলবে\n")
    else:
        failed.append(tag)

# ─── Concatenate ──────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = FINAL_OUTPUT_PATH  # auto-versioned path
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Failed: {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")

# ─── Summary ──────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("📋 VOICE CONSISTENCY REPORT:")
if CUSTOM_VOICE_PATH:
    print(f"  🎤 Custom voice ব্যবহৃত: {CUSTOM_VOICE_PATH}")
    print(f"  📛 Profile name: {CUSTOM_VOICE_NAME}")
elif USE_FIRST_GEN_AS_REFERENCE and first_speech_generated:
    print(f"  🔄 First-gen reference ব্যবহৃত")
    print(f"  📛 Reference profile: {reference_voice_name}")
else:
    print(f"  🔒 Seed-lock only (V8.5 mode)")
print(f"  🎙️  Final voice: {active_voice}")
print(f"  🌱 Seed: {SEED}")
print(f"{'=' * 70}")

# ═════════════════════════════════════════════════════════════════════════════
# 📥 DIRECT DOWNLOAD (base64 — no 404)
# ═════════════════════════════════════════════════════════════════════════════
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024*1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print(f"\n⬇️  নিচের বাটনে ক্লিক করুন:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# ðŸŽ­ V8.6 â€” VOICE-LOCKED: Reference Audio Chaining
# =============================================================================
#
# ðŸ” ROOT CAUSE (à¦°à¦¿à¦¸à¦¾à¦°à§à¦š à¦¥à§‡à¦•à§‡):
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# seed à¦¶à§à¦§à§ *à¦à¦•à¦‡ text*-à¦à¦° à¦œà¦¨à§à¦¯ reproducibility à¦¦à§‡à¦¯à¦¼à¥¤
# à¦­à¦¿à¦¨à§à¦¨ text input â†’ à¦­à¦¿à¦¨à§à¦¨ voice identity, seed à¦¯à¦¤à¦‡ fix à¦¥à¦¾à¦•à§à¦•à¥¤
# à¦à¦Ÿà¦¾à¦‡ "multiple characters" à¦¸à¦®à¦¸à§à¦¯à¦¾à¦° à¦†à¦¸à¦² à¦•à¦¾à¦°à¦£à¥¤
#
# âœ… SOLUTION: Reference Audio Chaining ("Anchor" Method)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 1. /v1/audio/speech/clone endpoint à¦¬à§à¦¯à¦¬à¦¹à¦¾à¦° à¦•à¦°à¦¿ (multipart/form-data)
# 2. à¦ªà§à¦°à¦¤à¦¿à¦Ÿà¦¾ segment-à¦ à¦à¦•à¦‡ reference audio à¦ªà¦¾à¦ à¦¾à¦‡
# 3. Reference audio = à¦†à¦ªà¦¨à¦¾à¦° à¦¦à§‡à¦“à¦¯à¦¼à¦¾ voice sample
#    à¦…à¦¥à¦¬à¦¾ à¦ªà§à¦°à¦¥à¦® generated segment
# 4. OmniVoice à¦¸à§‡à¦‡ reference à¦¥à§‡à¦•à§‡ timbre+delivery clone à¦•à¦°à§‡
#    â†’ à¦¸à¦¬ segment à¦à¦•à¦‡ à¦—à¦²à¦¾!
#
# ðŸ“– Developer forums:
#    "Use your first generated audio as the reference_audio for the
#     next segment. Seeds only ensure reproducibility for identical
#     inputs; they do NOT lock a voice persona when text changes."
#
# ðŸ“– OmniVoice docs:
#    "The reference clip is a performance direction. Zero-shot
#     cloning mirrors the delivery of the reference, not just timbre."
# =============================================================================

import os, time, json, urllib.request, wave, subprocess, base64, glob
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_6_voice_locked"
API_URL = "http://localhost:3900/v1/audio/speech"
CLONE_URL = "http://localhost:3900/v1/audio/speech/clone"
HEALTH_URL = "http://localhost:3900/health"
VOICES_URL = "http://localhost:3900/v1/audio/voices"

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# âš™ï¸ CONFIGURATION
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•

# â”€â”€â”€ à¦†à¦ªà¦¨à¦¾à¦° à¦­à¦¯à¦¼à§‡à¦¸ sample (à§©-à§§à§¦ à¦¸à§‡à¦•à§‡à¦¨à§à¦¡, à¦ªà¦°à¦¿à¦·à§à¦•à¦¾à¦°, à¦à¦•à¦œà¦¨à§‡à¦° à¦—à¦²à¦¾) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Kaggle-à¦ dataset à¦¹à¦¿à¦¸à§‡à¦¬à§‡ à¦†à¦ªà¦²à§‹à¦¡ à¦•à¦°à§à¦¨
# à¦¦à¦¿à¦¤à§‡ à¦¨à¦¾ à¦šà¦¾à¦‡à¦²à§‡ None à¦°à¦¾à¦–à§à¦¨ â€” à¦¤à¦–à¦¨ preset voice à¦¦à¦¿à¦¯à¦¼à§‡ à¦ªà§à¦°à¦¥à¦® segment generate
# à¦¹à¦¬à§‡, à¦¤à¦¾à¦°à¦ªà¦° à¦¸à§‡à¦Ÿà¦¾à¦‡ reference à¦¹à¦¿à¦¸à§‡à¦¬à§‡ à¦¬à§à¦¯à¦¬à¦¹à¦¾à¦° à¦¹à¦¬à§‡
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"

# â”€â”€â”€ Fallback preset voice (à¦¶à§à¦§à§ à¦ªà§à¦°à¦¥à¦® segment-à¦à¦° à¦œà¦¨à§à¦¯, à¦¯à¦¦à¦¿ custom à¦¨à¦¾ à¦¦à§‡à¦¨) â”€
PRESET_VOICE = "onyx"

# â”€â”€â”€ Common settings â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SEED = 42
FILENAME = "v8_6_voice_locked"

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸ” AUTO-DISCOVER: à¦­à¦¯à¦¼à§‡à¦¸ à¦«à¦¾à¦‡à¦² à¦–à§à¦à¦œà§‡ à¦¬à§‡à¦° à¦•à¦°à¦¾
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
def auto_discover_voice(given_path):
    """à¦¯à¦¦à¦¿ à¦¦à§‡à¦“à¦¯à¦¼à¦¾ à¦ªà¦¾à¦¥à§‡ à¦«à¦¾à¦‡à¦² à¦¨à¦¾ à¦¥à¦¾à¦•à§‡, /kaggle/input/ à¦ à¦¸à¦¬ audio file à¦–à§à¦à¦œà§‡ à¦¦à§‡à¦–à¦¾à¦¯à¦¼"""
    if given_path and os.path.exists(given_path):
        return given_path
    if not given_path:
        return None

    print(f"  âš ï¸ à¦¦à§‡à¦“à¦¯à¦¼à¦¾ à¦ªà¦¾à¦¥à§‡ à¦«à¦¾à¦‡à¦² à¦¨à§‡à¦‡: {given_path}")
    print(f"  ðŸ” /kaggle/input/ à¦ à¦…à¦¡à¦¿à¦“ à¦«à¦¾à¦‡à¦² à¦–à§à¦à¦œà¦›à¦¿...")

    audio_files = []
    search_root = "/kaggle/input"
    if os.path.exists(search_root):
        for root, dirs, files in os.walk(search_root):
            for f in files:
                if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))

    if not audio_files:
        print(f"  âŒ à¦•à§‹à¦¨à§‹ à¦…à¦¡à¦¿à¦“ à¦«à¦¾à¦‡à¦² à¦ªà¦¾à¦“à¦¯à¦¼à¦¾ à¦¯à¦¾à¦¯à¦¼à¦¨à¦¿ /kaggle/input/ à¦")
        return None

    print(f"  ðŸ“‹ à¦ªà¦¾à¦“à¦¯à¦¼à¦¾ à¦—à§‡à¦›à§‡ {len(audio_files)} à¦Ÿà¦¿ à¦…à¦¡à¦¿à¦“ à¦«à¦¾à¦‡à¦²:")
    for i, af in enumerate(audio_files):
        size_kb = os.path.getsize(af) // 1024
        print(f"     {i+1}. {af} ({size_kb} KB)")

    chosen = audio_files[0]
    print(f"  âœ… à¦¬à§à¦¯à¦¬à¦¹à¦¾à¦° à¦•à¦°à¦¾ à¦¹à¦šà§à¦›à§‡: {chosen}")
    return chosen

CUSTOM_VOICE_PATH = auto_discover_voice(CUSTOM_VOICE_PATH)

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸ“ AUTO-VERSIONING
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
def get_next_version(base_dir, base_name):
    """existing à¦«à¦¾à¦‡à¦² à¦šà§‡à¦• à¦•à¦°à§‡ à¦ªà¦°à¦¬à¦°à§à¦¤à§€ version number à¦¬à§‡à¦° à¦•à¦°à§‡"""
    os.makedirs(base_dir, exist_ok=True)
    version = 1
    while True:
        candidate = os.path.join(base_dir, f"{base_name}_v{version}.wav")
        if not os.path.exists(candidate):
            return version, candidate
        version += 1

RUN_VERSION, FINAL_OUTPUT_PATH = get_next_version(BASE_OUTPUT_DIR, FILENAME)
print(f"ðŸ“ Auto-version: v{RUN_VERSION} (à¦¨à¦¤à§à¦¨ à¦«à¦¾à¦‡à¦²: {os.path.basename(FINAL_OUTPUT_PATH)})")
print()

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸ”§ SERVER CHECK
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  âŒ OmniVoice Studio à¦ªà¦¾à¦“à¦¯à¦¼à¦¾ à¦¯à¦¾à¦¯à¦¼à¦¨à¦¿! Step 2 à¦†à¦—à§‡ à¦°à¦¾à¦¨ à¦•à¦°à§à¦¨à¥¤")
        return False
    print(f"  ðŸ“‚ {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  âœ… Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  â³ Waiting... ({(i+1)*2}s)")
    print("  âŒ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("ðŸ”§ SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  âœ… Server running")
else:
    print("  âš ï¸  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸ” ENDPOINT DISCOVERY: à¦¸à¦¾à¦°à§à¦­à¦¾à¦°à§‡ à¦•à§‹à¦¨ endpoints à¦†à¦›à§‡ à¦–à§à¦à¦œà§‡ à¦¬à§‡à¦° à¦•à¦°à¦¿
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
def discover_clone_endpoint():
    """
    à¦¸à¦¾à¦°à§à¦­à¦¾à¦°à§‡ /v1/audio/speech/clone endpoint à¦†à¦›à§‡ à¦•à¦¿à¦¨à¦¾ à¦šà§‡à¦• à¦•à¦°à¦¿à¥¤
    à¦¨à¦¾ à¦¥à¦¾à¦•à¦²à§‡ à¦¬à¦¿à¦•à¦²à§à¦ª endpoint à¦–à§à¦à¦œà¦¿à¥¤
    """
    # à¦ªà§à¦°à¦¥à¦®à§‡ OpenAPI spec à¦¥à§‡à¦•à§‡ à¦¸à¦¬ routes à¦¬à§‡à¦° à¦•à¦°à¦¿
    try:
        req = urllib.request.Request("http://localhost:3900/openapi.json")
        with urllib.request.urlopen(req, timeout=10) as resp:
            spec = json.loads(resp.read().decode())
            paths = spec.get("paths", {})

            print("  ðŸ“‹ à¦¸à¦¾à¦°à§à¦­à¦¾à¦°à§‡à¦° Audio API Routes:")
            clone_endpoint = None
            speech_schema = None

            for path, methods in sorted(paths.items()):
                if "audio" in path or "speech" in path or "voice" in path:
                    for method in methods:
                        if method in ("get", "post", "put", "delete"):
                            print(f"     {method.upper():6s} {path}")
                            if "clone" in path and method == "post":
                                clone_endpoint = path

            # Speech endpoint-à¦à¦° schema à¦¦à§‡à¦–à¦¿ â€” ref_audio à¦†à¦›à§‡ à¦•à¦¿à¦¨à¦¾
            speech_path = paths.get("/v1/audio/speech", {})
            if "post" in speech_path:
                post_spec = speech_path["post"]
                body = post_spec.get("requestBody", {}).get("content", {})
                for content_type, schema_info in body.items():
                    print(f"\n  ðŸ“– /v1/audio/speech accepts: {content_type}")
                    # multipart/form-data accept à¦•à¦°à¦²à§‡ ref_audio à¦ªà¦¾à¦ à¦¾à¦¨à§‹ à¦¯à¦¾à¦¬à§‡
                    if "multipart" in content_type:
                        print("     âœ… multipart/form-data à¦¸à¦¾à¦ªà§‹à¦°à§à¦Ÿ à¦•à¦°à§‡!")
                        return "/v1/audio/speech", "multipart"

            # schema components à¦¥à§‡à¦•à§‡ speech params à¦–à§à¦à¦œà¦¿
            schemas = spec.get("components", {}).get("schemas", {})
            for name, schema in schemas.items():
                if "speech" in name.lower() or "tts" in name.lower() or "clone" in name.lower():
                    props = schema.get("properties", {})
                    print(f"\n  ðŸ“– Schema '{name}' properties:")
                    for prop_name, prop_val in props.items():
                        print(f"     â€¢ {prop_name}: {prop_val.get('type', 'N/A')}")

            if clone_endpoint:
                return clone_endpoint, "clone"
            return None, None

    except Exception as e:
        print(f"  âš ï¸ OpenAPI spec à¦ªà¦¡à¦¼à¦¾ à¦¯à¦¾à¦¯à¦¼à¦¨à¦¿: {e}")
        return None, None

print("=" * 70)
print("ðŸ” ENDPOINT DISCOVERY")
print("=" * 70)
clone_ep, clone_mode = discover_clone_endpoint()
print()

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸŽµ CORE FUNCTIONS
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp): continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  âŒ No WAV files!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  ðŸŽ¬ FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_with_clone(text, ref_audio_path, output_path, speed=0.88,
                   guidance_scale=2.0, seed=42, ref_text=""):
    """
    ðŸŽ¯ /v1/audio/speech/clone endpoint â€” multipart/form-data
    à¦ªà§à¦°à¦¤à¦¿à¦Ÿà¦¾ API call-à¦ reference audio à¦ªà¦¾à¦ à¦¾à¦¯à¦¼à¥¤
    à¦à¦¤à§‡ voice identity locked à¦¥à¦¾à¦•à§‡à¥¤
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    boundary = "----OmniVoiceClone" + str(int(time.time() * 1000))
    body = b""

    # text field
    body += f"--{boundary}\r\n".encode()
    body += b"Content-Disposition: form-data; name=\"text\"\r\n\r\n"
    body += f"{text}\r\n".encode()

    # speed field
    body += f"--{boundary}\r\n".encode()
    body += b"Content-Disposition: form-data; name=\"speed\"\r\n\r\n"
    body += f"{speed}\r\n".encode()

    # ref_text field (optional â€” à¦¨à¦¾ à¦¦à¦¿à¦²à§‡ Whisper auto-transcribe à¦•à¦°à¦¬à§‡)
    if ref_text:
        body += f"--{boundary}\r\n".encode()
        body += b"Content-Disposition: form-data; name=\"ref_text\"\r\n\r\n"
        body += f"{ref_text}\r\n".encode()

    # ref_audio file
    filename = os.path.basename(ref_audio_path)
    ext = os.path.splitext(filename)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")

    body += f"--{boundary}\r\n".encode()
    body += f'Content-Disposition: form-data; name="ref_audio"; filename="{filename}"\r\n'.encode()
    body += f"Content-Type: {mime}\r\n\r\n".encode()

    with open(ref_audio_path, "rb") as f:
        body += f.read()
    body += b"\r\n"

    # closing boundary
    body += f"--{boundary}--\r\n".encode()

    url = f"http://localhost:3900{clone_ep}" if clone_ep else CLONE_URL
    req = urllib.request.Request(
        url,
        data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
        method="POST"
    )

    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  âœ… {tag:40s} â”‚ {len(content)//1024:5d} KB â”‚ {time.time()-t0:4.1f}s â”‚ ðŸ”—clone")
            return True
    except urllib.error.HTTPError as e:
        error_body = e.read().decode() if hasattr(e, 'read') else str(e)
        print(f"  âŒ Clone failed (HTTP {e.code}): {error_body[:200]}")
        return False
    except Exception as e:
        print(f"  âŒ Clone error: {e}")
        return False

def gen_with_json(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                  class_temperature=0.3, postprocess_output=True, seed=42):
    """
    Fallback: /v1/audio/speech â€” JSON endpoint (seed lock only)
    Clone endpoint à¦•à¦¾à¦œ à¦¨à¦¾ à¦•à¦°à¦²à§‡ à¦à¦Ÿà¦¾ à¦¬à§à¦¯à¦¬à¦¹à¦¾à¦° à¦¹à¦¯à¦¼à¥¤
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  âœ… {tag:40s} â”‚ {len(content)//1024:5d} KB â”‚ {time.time()-t0:4.1f}s â”‚ ðŸ”’seed")
            return True
    except Exception as e:
        print(f"  âŒ {os.path.basename(output_path)}: {e}")
        return False

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸŽ¬ SCRIPT â€” SEGMENTS
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•

SEGMENTS = [
    # â”€â”€ 01: Opening â€” calm anticipation â”€â”€
    {
        "tag": "01_opening",
        "text": (
            "What if the greatest invention in human history... "
            "wasn't a machine that traveled through space â€” "
            "but one that crossed possibilities?"
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 400},

    # â”€â”€ 02: Doubt â€” slight amusement â”€â”€
    {
        "tag": "02_doubt",
        "text": (
            "Another universe? Seriously? "
            "Scientists kept working. Everyone else kept doubting."
        ),
        "speed": 0.92,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 300},

    # â”€â”€ 03: Wonder â€” building but controlled â”€â”€
    {
        "tag": "03_wonder",
        "text": (
            "And suddenly â€” the impossible became real. "
            "We saw dinosaurs, still walking, beneath blood-red skies. "
            "We found another Earth where humanity was born on Mars."
        ),
        "speed": 0.90,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 700},

    # â”€â”€ 04: Shift â€” personal, tone drops â”€â”€
    {
        "tag": "04_shift",
        "text": (
            "But me... "
            "no. None of those worlds mattered. Not one. "
            "I was searching... for someone."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
    {"silence_ms": 800},

    # â”€â”€ 05: Grief â€” slow, heavy â”€â”€
    {
        "tag": "05_grief",
        "text": (
            "Three years ago... cancer... stole my mother. "
            "No warning. No mercy. No second chance."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # â”€â”€ 06: Hospital â€” deepest sadness â”€â”€
    {
        "tag": "06_hospital",
        "text": (
            "I watched the hospital monitor... become... silent. "
            "I held her hand â€” hoping â€” just hoping â€” "
            "she'd squeeze mine... one... last... time."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1500},

    # â”€â”€ 07: Weight â€” one line, devastating â”€â”€
    {
        "tag": "07_she_never_did",
        "text": "She never did.",
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1000},

    # â”€â”€ 08: Pain â€” grief meets frustration â”€â”€
    {
        "tag": "08_pain",
        "text": (
            "I know she'll never answer. I know that. "
            "But I still call. "
            "Because sometimes... hope hurts more than reality."
        ),
        "speed": 0.85,
        "guidance_scale": 2.2,
    },
    {"silence_ms": 800},

    # â”€â”€ 09: Plea â€” quiet, desperate â”€â”€
    {
        "tag": "09_plea",
        "text": (
            "Listen. "
            "Take me to the universe... where my mother... never died."
        ),
        "speed": 0.82,
        "guidance_scale": 1.8,
        "postprocess_output": False,
    },
    {"silence_ms": 1200},

    # â”€â”€ 10: Warmth â€” relief, restrained â”€â”€
    {
        "tag": "10_warmth",
        "text": (
            "There she was. Alive. Smiling. Making breakfast. "
            "Humming the exact same song she used to sing... "
            "every Sunday morning."
        ),
        "speed": 0.88,
        "guidance_scale": 2.0,
    },
    {"silence_ms": 1000},

    # â”€â”€ 11: Ending â€” bittersweet, soft â”€â”€
    {
        "tag": "11_ending",
        "text": (
            "She didn't know I wasn't her son. "
            "I was a broken man... "
            "borrowing someone else's miracle."
        ),
        "speed": 0.85,
        "guidance_scale": 2.0,
        "postprocess_output": False,
    },
]

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸš€ GENERATE â†’ CONCATENATE â†’ DOWNLOAD
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)

speech_segs = [s for s in SEGMENTS if "text" in s]

# â”€â”€â”€ Reference Audio à¦¨à¦¿à¦°à§à¦§à¦¾à¦°à¦£ â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Strategy:
#   1. Custom voice à¦¦à§‡à¦“à¦¯à¦¼à¦¾ à¦¥à¦¾à¦•à¦²à§‡ â†’ à¦¸à§‡à¦Ÿà¦¾ reference
#   2. à¦¨à¦¾ à¦¥à¦¾à¦•à¦²à§‡ â†’ à¦ªà§à¦°à¦¥à¦® segment preset voice à¦¦à¦¿à¦¯à¦¼à§‡ generate â†’ à¦¸à§‡à¦Ÿà¦¾ reference
#   3. Clone endpoint à¦¨à¦¾ à¦¥à¦¾à¦•à¦²à§‡ â†’ fallback to seed-lock (V8.5 mode)

ref_audio_path = CUSTOM_VOICE_PATH  # None à¦¹à¦¤à§‡ à¦ªà¦¾à¦°à§‡
use_clone = clone_ep is not None or True  # always try clone first
clone_works = None  # None=untested, True/False=tested

print("=" * 70)
print("ðŸŽ­  V8.6 â€” REFERENCE AUDIO CHAINING")
print("=" * 70)
print(f"ðŸŽ™ï¸  Reference  : {ref_audio_path or 'à¦ªà§à¦°à¦¥à¦® segment à¦¥à§‡à¦•à§‡ à¦¤à§ˆà¦°à¦¿ à¦¹à¦¬à§‡'}")
print(f"ðŸ”—  Clone EP   : {clone_ep or CLONE_URL}")
print(f"ðŸ”’  Seed       : {SEED} (fallback)")
print(f"ðŸŽ¬  Segments   : {len(speech_segs)} speech")
print(f"ðŸ“  Output     : {os.path.basename(FINAL_OUTPUT_PATH)}")
print("=" * 70)
print()

wav_order = []
generated = 0
failed = []
sil_idx = 0

for seg in SEGMENTS:
    # Silence
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    # Speech
    tag = seg["tag"]
    wav_path = os.path.join(seg_dir, f"{tag}.wav")
    success = False

    # â”€â”€â”€ METHOD 1: Clone with reference audio â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if ref_audio_path and clone_works is not False:
        success = gen_with_clone(
            text=seg["text"],
            ref_audio_path=ref_audio_path,
            output_path=wav_path,
            speed=seg.get("speed", 0.88),
            guidance_scale=seg.get("guidance_scale", 2.0),
            seed=SEED,
        )
        if success and clone_works is None:
            clone_works = True
            print(f"  ðŸŽ‰ Clone endpoint à¦•à¦¾à¦œ à¦•à¦°à¦›à§‡! à¦¬à¦¾à¦•à¦¿ à¦¸à¦¬ segment clone à¦¦à¦¿à¦¯à¦¼à§‡ à¦¹à¦¬à§‡\n")
        elif not success and clone_works is None:
            clone_works = False
            print(f"  âš ï¸ Clone endpoint à¦•à¦¾à¦œ à¦•à¦°à§‡à¦¨à¦¿ â€” JSON fallback à¦¬à§à¦¯à¦¬à¦¹à¦¾à¦° à¦¹à¦¬à§‡\n")

    # â”€â”€â”€ METHOD 2: Fallback â€” JSON with seed lock â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if not success:
        success = gen_with_json(
            text=seg["text"],
            voice=PRESET_VOICE,
            output_path=wav_path,
            speed=seg.get("speed", 0.88),
            guidance_scale=seg.get("guidance_scale", 2.0),
            class_temperature=0.3,
            postprocess_output=seg.get("postprocess_output", True),
            seed=SEED,
        )

    if success:
        wav_order.append(wav_path)
        generated += 1

        # à¦ªà§à¦°à¦¥à¦® à¦¸à¦«à¦² generation â†’ reference audio à¦¹à¦¿à¦¸à§‡à¦¬à§‡ à¦¬à§à¦¯à¦¬à¦¹à¦¾à¦° (à¦¯à¦¦à¦¿ custom à¦¨à¦¾ à¦¦à§‡à¦“à¦¯à¦¼à¦¾ à¦¥à¦¾à¦•à§‡)
        if not ref_audio_path:
            ref_audio_path = wav_path
            print(f"  ðŸ”„ à¦ªà§à¦°à¦¥à¦® segment â†’ reference audio à¦¸à§‡à¦Ÿ à¦¹à¦¯à¦¼à§‡à¦›à§‡: {tag}\n")
    else:
        failed.append(tag)

# â”€â”€â”€ Concatenate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"\n{'â”€' * 70}")
print("ðŸ”— Concatenating...")
final_path = FINAL_OUTPUT_PATH
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"ðŸŽ‰  DONE! {generated}/{len(speech_segs)} segments â†’ 1 file")
if failed:
    print(f"  âš ï¸  Failed: {', '.join(failed)}")
print(f"  â±ï¸  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  ðŸ“ {final_path}")

# â”€â”€â”€ Summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"\n{'â”€' * 70}")
print("ðŸ“‹ VOICE CONSISTENCY REPORT:")
if clone_works:
    print(f"  ðŸ”— Method: Reference Audio Cloning (BEST)")
    print(f"  ðŸŽ™ï¸ Reference: {ref_audio_path}")
elif clone_works is False:
    print(f"  ðŸ”’ Method: Seed-lock fallback (clone endpoint unavailable)")
    print(f"  ðŸŽ™ï¸ Voice: {PRESET_VOICE}, Seed: {SEED}")
else:
    print(f"  ðŸ”’ Method: Seed-lock (no reference audio)")
print(f"{'=' * 70}")

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# ðŸ“¥ DIRECT DOWNLOAD (base64 â€” no 404)
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024*1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    â¬‡ï¸ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print(f"\nâ¬‡ï¸  à¦¨à¦¿à¦šà§‡à¦° à¦¬à¦¾à¦Ÿà¦¨à§‡ à¦•à§à¦²à¦¿à¦• à¦•à¦°à§à¦¨:")
    display(HTML(html))
else:
    print("\nâŒ à¦«à¦¾à¦‡à¦² à¦¤à§ˆà¦°à¦¿ à¦¹à¦¯à¦¼à¦¨à¦¿à¥¤")


In [ ]:

Created src/data/script.ts
export const PY_SCRIPT = String.raw`# =============================================================================
# 🎭 V9.0 — ONE VOICE, START TO FINISH  (Absolute Voice Lock)
# =============================================================================
# সমস্যা: "একটু পর পর ভিন্ন মানুষ কথা বলছে"
# লক্ষ্য : শুরুতে যিনি বলছেন, শেষ পর্যন্ত তিনিই বলবেন।
#
# ─────────────────────────────────────────────────────────────────────────────
# V8.6 তে গলা বদলে যাওয়ার ৫টা আসল কারণ (সবগুলো এখানে ফিক্স করা হয়েছে)
# ─────────────────────────────────────────────────────────────────────────────
# 1) MIXED METHODS  : কিছু segment clone endpoint দিয়ে, কিছু JSON preset দিয়ে
#                     তৈরি হচ্ছিল। দুই ইঞ্জিন = দুই মানুষ। এখন একটাই mode,
#                     পুরো রান জুড়ে — preflight probe দিয়ে আগেই ঠিক করা হয়।
# 2) PER-SEGMENT PARAMS : speed 0.82→0.92, guidance_scale 1.8→2.2,
#                     postprocess_output on/off — এগুলো timbre/formant বদলে
#                     দেয়, তাই প্রতিটা segment আলাদা লোক শোনায়।
#                     এখন সব segment-এ হুবহু একই VOICE dict।
# 3) SAMPLE-RATE BUG : make_silence() 22050 Hz লিখত, কিন্তু OmniVoice সাধারণত
#                     24000 Hz দেয়। concat_wavs() প্রথম ফাইলের params নিত →
#                     বাকি সব ক্লিপ ভুল রেটে বাজত = pitch shift = অন্য মানুষ।
#                     এখন সব কিছু TARGET_SR এ resample করে জোড়া হয়।
# 4) LOUDNESS JUMP  : segment-ভেদে RMS আলাদা হলে কান "নতুন বক্তা" ধরে নেয়।
#                     এখন প্রতিটা segment একই RMS (-20 dBFS) এ normalize।
# 5) NO VERIFICATION : খারাপ generation ধরা পড়ত না। এখন প্রতিটা segment-এর
#                     spectral+pitch fingerprint anchor-এর সাথে মিলিয়ে দেখা হয়,
#                     না মিললে অটো re-generate (MAX_RETRIES বার)।
#
# 📖 নীতি: "Reference audio = voice identity. Parameters = must stay frozen."
# =============================================================================
import os, sys, time, json, math, wave, glob, base64, shutil, subprocess
import urllib.request, urllib.error
import numpy as np
from IPython.display import HTML, display
# ═════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION  — এই ব্লকটাই শুধু ছোঁবেন
# ═════════════════════════════════════════════════════════════════════════════
HOST            = "http://localhost:3900"
API_URL         = HOST + "/v1/audio/speech"
CLONE_URL       = HOST + "/v1/audio/speech/clone"
HEALTH_URL      = HOST + "/health"
OPENAPI_URL     = HOST + "/openapi.json"
BASE_OUTPUT_DIR = "/kaggle/working/outputs/v9_one_voice"
FILENAME        = "v9_one_voice"
# আপনার ভয়েস স্যাম্পল (৫–১৫ সেকেন্ড, একজনের গলা, নয়েজ-মুক্ত, স্পষ্ট)
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
# reference clip-এর হুবহু transcript জানা থাকলে লিখুন (Whisper-এর ভুল এড়ায়,
# ফলে প্রতিবার একই conditioning → গলা আরও স্থির)। না জানলে "" রাখুন।
CUSTOM_VOICE_TEXT = ""
PRESET_VOICE    = "onyx"   # শুধু fallback / anchor তৈরির জন্য
SEED            = 1234
# ── 🔒 FROZEN VOICE PARAMETERS ───────────────────────────────────────────────
# এই ডিক্ট প্রতিটা segment-এ হুবহু একইভাবে যাবে। কখনো per-segment override
# করবেন না — এটাই V8.6-এর সবচেয়ে বড় ভুল ছিল।
# আবেগ (emotion) নিয়ন্ত্রণ করুন শুধু: শব্দচয়ন, কমা/ড্যাশ/ellipsis, আর pause
# length দিয়ে — parameter দিয়ে নয়।
VOICE = {
    "speed":              0.88,
    "guidance_scale":     2.0,
    "num_step":           32,
    "class_temperature":  0.3,
    "postprocess_output": True,
}
# ── Audio pipeline ───────────────────────────────────────────────────────────
TARGET_SR            = 24000    # সব কিছু এখানে resample হবে
TARGET_RMS_DBFS      = -20.0    # সব segment একই লাউডনেসে
PEAK_CEILING         = 0.97
FADE_MS              = 12       # click/pop ঠেকাতে
# ── Voice-drift guard ────────────────────────────────────────────────────────
VERIFY_VOICE         = True
SIMILARITY_THRESHOLD = 0.86     # 0.80 = ঢিলা, 0.90 = কড়া
MAX_RETRIES          = 3
MAX_CHARS_PER_CALL   = 300      # লম্বা টেক্সট ভাঙা হয়, প্যারামিটার একই থাকে
# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — শুধু text + pause. কোনো per-segment voice parameter নেই।
# ═════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text":
        "What if the greatest invention in human history... "
        "wasn't a machine that traveled through space, "
        "but one that crossed possibilities?"},
    {"pause_ms": 450},
    {"tag": "02_doubt", "text":
        "Another universe? Seriously? "
        "Scientists kept working. Everyone else kept doubting."},
    {"pause_ms": 350},
    {"tag": "03_wonder", "text":
        "And suddenly, the impossible became real. "
        "We saw dinosaurs, still walking, beneath blood-red skies. "
        "We found another Earth, where humanity was born on Mars."},
    {"pause_ms": 700},
    {"tag": "04_shift", "text":
        "But me... no. None of those worlds mattered. Not one. "
        "I was searching... for someone."},
    {"pause_ms": 800},
    {"tag": "05_grief", "text":
        "Three years ago... cancer... stole my mother. "
        "No warning. No mercy. No second chance."},
    {"pause_ms": 1000},
    {"tag": "06_hospital", "text":
        "I watched the hospital monitor... become... silent. "
        "I held her hand, hoping, just hoping, "
        "she would squeeze mine... one... last... time."},
    {"pause_ms": 1400},
    {"tag": "07_she_never_did", "text":
        "She never did."},
    {"pause_ms": 1000},
    {"tag": "08_pain", "text":
        "I know she will never answer. I know that. "
        "But I still call. "
        "Because sometimes... hope hurts more than reality."},
    {"pause_ms": 800},
    {"tag": "09_plea", "text":
        "Listen. Take me to the universe... where my mother... never died."},
    {"pause_ms": 1200},
    {"tag": "10_warmth", "text":
        "There she was. Alive. Smiling. Making breakfast. "
        "Humming the exact same song she used to sing... "
        "every Sunday morning."},
    {"pause_ms": 1000},
    {"tag": "11_ending", "text":
        "She did not know I was not her son. "
        "I was a broken man... borrowing someone else's miracle."},
]
# calibration line — custom voice না থাকলে এটা দিয়ে anchor তৈরি হয়
ANCHOR_TEXT = ("This is my natural speaking voice, calm and steady, "
               "and I will keep this exact same voice from the first word "
               "to the very last one.")
# ═════════════════════════════════════════════════════════════════════════════
# 🧰 LOW-LEVEL AUDIO UTILITIES  (numpy only — কোনো heavy dependency নেই)
# ═════════════════════════════════════════════════════════════════════════════
def have_ffmpeg():
    return shutil.which("ffmpeg") is not None
def wav_read(path):
    """WAV → (float32 mono [-1,1], sample_rate)"""
    with wave.open(path, "rb") as wf:
        nch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
        raw = wf.readframes(n)
    if sw == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sw == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError("Unsupported sample width: " + str(sw))
    if nch > 1:
        x = x.reshape(-1, nch).mean(axis=1)
    return x.astype(np.float32), sr
def wav_write(path, x, sr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    x = np.clip(x, -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm)
def resample(x, sr_in, sr_out):
    """High-quality-enough linear resampler (mono)."""
    if sr_in == sr_out or len(x) == 0:
        return x
    n_out = int(round(len(x) * float(sr_out) / float(sr_in)))
    t_in  = np.arange(len(x), dtype=np.float64)
    t_out = np.linspace(0.0, len(x) - 1.0, n_out)
    return np.interp(t_out, t_in, x).astype(np.float32)
def decode_to_wav(src, dst, sr=TARGET_SR):
    """যেকোনো audio (mp3/m4a/flac/ogg/wav) → mono WAV @ sr"""
    if have_ffmpeg():
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
               "-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", dst]
        if subprocess.run(cmd).returncode == 0 and os.path.exists(dst):
            return dst
    if src.lower().endswith(".wav"):
        x, s = wav_read(src)
        wav_write(dst, resample(x, s, sr), sr)
        return dst
    raise RuntimeError("ffmpeg নেই এবং reference WAV নয়: " + src)
def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))
def normalize_rms(x, target_dbfs=TARGET_RMS_DBFS, ceiling=PEAK_CEILING):
    cur = rms(x)
    if cur < 1e-6:
        return x
    gain = (10.0 ** (target_dbfs / 20.0)) / cur
    y = x * gain
    peak = float(np.max(np.abs(y)) + 1e-12)
    if peak > ceiling:
        y = y * (ceiling / peak)
    return y.astype(np.float32)
def apply_fades(x, sr, ms=FADE_MS):
    n = min(int(sr * ms / 1000.0), len(x) // 2)
    if n <= 0:
        return x
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x.copy()
    x[:n]  *= ramp
    x[-n:] *= ramp[::-1]
    return x
def trim_silence(x, sr, thresh_db=-42.0, pad_ms=60):
    """শুরু/শেষের নীরবতা কাটে — reference clip পরিষ্কার করার জন্য।"""
    win = max(1, int(sr * 0.02))
    if len(x) < win * 3:
        return x
    frames = x[:len(x) - len(x) % win].reshape(-1, win)
    energy = 20.0 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)
    loud = np.where(energy > thresh_db)[0]
    if len(loud) == 0:
        return x
    pad = int(sr * pad_ms / 1000.0)
    a = max(0, loud[0] * win - pad)
    b = min(len(x), (loud[-1] + 1) * win + pad)
    return x[a:b]
def best_window(x, sr, seconds=9.0):
    """সবচেয়ে energetic টানা অংশ বেছে নেয় (reference-এর জন্য আদর্শ)।"""
    want = int(sr * seconds)
    if len(x) <= want:
        return x
    hop = int(sr * 0.25)
    best_i, best_e = 0, -1.0
    for i in range(0, len(x) - want, hop):
        e = rms(x[i:i + want])
        if e > best_e:
            best_e, best_i = e, i
    return x[best_i:best_i + want]
# ═════════════════════════════════════════════════════════════════════════════
# 🧬 SPEAKER FINGERPRINT  (spectral envelope + pitch) — drift ধরার জন্য
# ═════════════════════════════════════════════════════════════════════════════
def spectral_fingerprint(x, sr, n_bands=40):
    n, hop = 1024, 512
    if len(x) < n * 2:
        return None
    w = np.hanning(n).astype(np.float32)
    acc, cnt = None, 0
    for i in range(0, len(x) - n, hop):
        seg = x[i:i + n]
        if rms(seg) < 0.005:            # নীরব ফ্রেম বাদ
            continue
        spec = np.abs(np.fft.rfft(seg * w))
        acc = spec if acc is None else acc + spec
        cnt += 1
    if cnt < 4:
        return None
    S = acc / cnt
    edges = np.linspace(0, len(S), n_bands + 1).astype(int)
    bands = np.array([S[edges[i]:edges[i + 1]].mean() + 1e-9 for i in range(n_bands)])
    v = np.log(bands)
    v = v - v.mean()
    nrm = np.linalg.norm(v) + 1e-9
    return (v / nrm).astype(np.float32)
def median_f0(x, sr, fmin=60.0, fmax=320.0):
    n, hop = 2048, 1024
    vals = []
    for i in range(0, max(1, len(x) - n), hop):
        seg = x[i:i + n]
        if len(seg) < n or rms(seg) < 0.01:
            continue
        seg = seg - seg.mean()
        ac = np.correlate(seg, seg, mode="full")[n - 1:]
        lo, hi = int(sr / fmax), int(sr / fmin)
        if hi >= len(ac):
            continue
        k = int(np.argmax(ac[lo:hi])) + lo
        if ac[k] > 0.3 * (ac[0] + 1e-9):
            vals.append(sr / float(k))
    return float(np.median(vals)) if len(vals) >= 3 else 0.0
def voice_profile(x, sr):
    return {"spec": spectral_fingerprint(x, sr), "f0": median_f0(x, sr)}
def voice_similarity(a, b):
    """0..1 — 1 মানে হুবহু একই বক্তা।"""
    if a is None or b is None or a["spec"] is None or b["spec"] is None:
        return 1.0
    cos = float(np.dot(a["spec"], b["spec"]))
    cos = max(0.0, min(1.0, (cos + 1.0) / 2.0 * 1.0 if cos < 0 else cos))
    if a["f0"] > 0 and b["f0"] > 0:
        ratio = min(a["f0"], b["f0"]) / max(a["f0"], b["f0"])
        pitch = max(0.0, 1.0 - (1.0 - ratio) * 2.5)     # 20% pitch drift = 0.5
    else:
        pitch = cos
    return 0.72 * cos + 0.28 * pitch
# ═════════════════════════════════════════════════════════════════════════════
# 🌐 SERVER + ENDPOINT
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False
def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None
def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! আগে Step 2 রান করুন।")
        return False
    print("  📂 " + studio)
    log = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log, stderr=log, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print("  ✅ Server ready! (" + str((i + 1) * 2) + "s)")
            return True
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False
def discover_clone_endpoint():
    """clone route আছে কিনা openapi.json থেকে দেখি।"""
    try:
        with urllib.request.urlopen(urllib.request.Request(OPENAPI_URL), timeout=10) as r:
            spec = json.loads(r.read().decode())
        for path, methods in sorted(spec.get("paths", {}).items()):
            if "clone" in path and "post" in methods:
                return path
    except Exception as e:
        print("  ⚠️ openapi.json পড়া যায়নি: " + str(e))
    return None
# ═════════════════════════════════════════════════════════════════════════════
# 📡 REQUESTS
# ═════════════════════════════════════════════════════════════════════════════
def post_multipart(url, fields, file_field, file_path, timeout=600):
    boundary = "----OmniVoiceLock" + str(int(time.time() * 1000))
    body = b""
    for k, v in fields.items():
        body += ("--" + boundary + "\r\n").encode()
        body += ('Content-Disposition: form-data; name="' + k + '"\r\n\r\n').encode()
        body += (str(v) + "\r\n").encode()
    fname = os.path.basename(file_path)
    ext = os.path.splitext(fname)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")
    body += ("--" + boundary + "\r\n").encode()
    body += ('Content-Disposition: form-data; name="' + file_field +
             '"; filename="' + fname + '"\r\n').encode()
    body += ("Content-Type: " + mime + "\r\n\r\n").encode()
    with open(file_path, "rb") as f:
        body += f.read()
    body += b"\r\n"
    body += ("--" + boundary + "--\r\n").encode()
    req = urllib.request.Request(
        url, data=body, method="POST",
        headers={"Content-Type": "multipart/form-data; boundary=" + boundary})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()
def clone_fields(text, minimal=False):
    """প্রতিটা call-এ হুবহু একই parameter set — এটাই voice lock-এর মূল।"""
    f = {"text": text, "input": text, "seed": SEED, "speed": VOICE["speed"]}
    if CUSTOM_VOICE_TEXT:
        f["ref_text"] = CUSTOM_VOICE_TEXT
    if minimal:
        return {k: v for k, v in f.items() if k in ("text", "input", "ref_text")}
    f["guidance_scale"] = VOICE["guidance_scale"]
    f["num_step"] = VOICE["num_step"]
    f["response_format"] = "wav"
    return f
def gen_clone(text, ref_wav, out_path, clone_ep):
    url = HOST + clone_ep if clone_ep else CLONE_URL
    for minimal in (False, True):     # অতিরিক্ত field-এ 422 হলে minimal-এ retry
        try:
            data = post_multipart(url, clone_fields(text, minimal), "ref_audio", ref_wav)
            with open(out_path, "wb") as f:
                f.write(data)
            return True
        except urllib.error.HTTPError as e:
            msg = ""
            try:
                msg = e.read().decode()[:180]
            except Exception:
                pass
            if e.code in (400, 422) and not minimal:
                continue
            print("    ❌ clone HTTP " + str(e.code) + ": " + msg)
            return False
        except Exception as e:
            print("    ❌ clone error: " + str(e))
            return False
    return False
def gen_json(text, out_path, voice=None):
    """Fallback — clone route না থাকলে। এখানেও প্যারামিটার হুবহু frozen।"""
    payload = {
        "model": "tts-1-hd",
        "voice": voice or PRESET_VOICE,
        "input": text,
        "response_format": "wav",
        "speed": VOICE["speed"],
        "num_step": VOICE["num_step"],
        "guidance_scale": VOICE["guidance_scale"],
        "class_temperature": VOICE["class_temperature"],
        "postprocess_output": VOICE["postprocess_output"],
        "seed": SEED,
    }
    req = urllib.request.Request(API_URL, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            data = r.read()
        with open(out_path, "wb") as f:
            f.write(data)
        return True
    except Exception as e:
        print("    ❌ json error: " + str(e))
        return False
# ═════════════════════════════════════════════════════════════════════════════
# ✂️  TEXT CHUNKING — লম্বা লাইন ভাঙে, কিন্তু voice setting একই থাকে
# ═════════════════════════════════════════════════════════════════════════════
def split_text(text, limit=MAX_CHARS_PER_CALL):
    text = " ".join(text.split())
    if len(text) <= limit:
        return [text]
    parts, buf = [], ""
    for piece in text.replace("? ", "?|").replace("! ", "!|").replace(". ", ".|").split("|"):
        if len(buf) + len(piece) + 1 <= limit:
            buf = (buf + " " + piece).strip()
        else:
            if buf:
                parts.append(buf)
            buf = piece.strip()
    if buf:
        parts.append(buf)
    return parts
# ═════════════════════════════════════════════════════════════════════════════
# 🚀 PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    v = 1
    while os.path.exists(os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")):
        v += 1
    return v, os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")
def auto_discover_voice(given):
    if given and os.path.exists(given):
        return given
    print("  ⚠️ দেওয়া পাথে ফাইল নেই: " + str(given))
    found = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                found.append(os.path.join(root, f))
    if not found:
        print("  ❌ /kaggle/input/ এ কোনো অডিও নেই → anchor TTS দিয়ে বানানো হবে")
        return None
    found.sort()
    print("  📋 পাওয়া গেছে " + str(len(found)) + " টি অডিও, ব্যবহার: " + found[0])
    return found[0]
print("=" * 74)
print("🔧 SERVER CHECK")
print("=" * 74)
if server_alive():
    print("  ✅ Server running")
elif not start_server():
    raise RuntimeError("Server failed. Step 2 & 3 আগে চালান।")
clone_ep = discover_clone_endpoint()
print("  🔗 Clone endpoint: " + str(clone_ep or "(default " + CLONE_URL + ")"))
RUN_VERSION, FINAL_OUTPUT_PATH = next_version(BASE_OUTPUT_DIR, FILENAME)
seg_dir  = os.path.join(BASE_OUTPUT_DIR, "segments_v" + str(RUN_VERSION))
work_dir = os.path.join(BASE_OUTPUT_DIR, "work")
os.makedirs(seg_dir, exist_ok=True)
os.makedirs(work_dir, exist_ok=True)
# ── STEP 1: ANCHOR (একটাই গলা, পুরো ট্র্যাকের ভিত্তি) ────────────────────────
print()
print("=" * 74)
print("🎙️  STEP 1 — ANCHOR VOICE তৈরি")
print("=" * 74)
anchor_wav = os.path.join(work_dir, "anchor.wav")
src_voice  = auto_discover_voice(CUSTOM_VOICE_PATH)
MODE = None      # "clone" বা "preset" — একবার ঠিক হলে সারা রান একই থাকবে
if src_voice:
    tmp = os.path.join(work_dir, "_ref_raw.wav")
    decode_to_wav(src_voice, tmp, TARGET_SR)
    x, sr = wav_read(tmp)
    x = best_window(trim_silence(x, sr), sr, 9.0)
    x = apply_fades(normalize_rms(x), sr)
    wav_write(anchor_wav, x, sr)
    print("  ✅ Custom reference: " + src_voice)
    print("     → anchor.wav  " + str(round(len(x) / sr, 1)) + "s @ " + str(sr) + " Hz")
    MODE = "clone"
else:
    print("  🔁 custom voice নেই → preset '" + PRESET_VOICE + "' দিয়ে anchor বানাচ্ছি...")
    raw = os.path.join(work_dir, "_anchor_raw.wav")
    if gen_json(ANCHOR_TEXT, raw):
        x, sr = wav_read(raw)
        x = resample(x, sr, TARGET_SR)
        x = apply_fades(normalize_rms(best_window(trim_silence(x, TARGET_SR), TARGET_SR, 9.0)), TARGET_SR)
        wav_write(anchor_wav, x, TARGET_SR)
        print("  ✅ anchor.wav তৈরি হয়েছে preset voice থেকে")
        MODE = "clone"
    else:
        print("  ⚠️ anchor বানানো যায়নি → পুরো রান preset+seed mode এ চলবে")
        MODE = "preset"
# ── STEP 2: PREFLIGHT PROBE — mode একবারেই লক ─────────────────────────────────
print()
print("=" * 74)
print("🧪 STEP 2 — PREFLIGHT (mode লক করা হচ্ছে, মাঝপথে আর বদলাবে না)")
print("=" * 74)
if MODE == "clone":
    probe = os.path.join(work_dir, "_probe.wav")
    ok = gen_clone("Testing one single voice.", anchor_wav, probe, clone_ep)
    if ok and os.path.getsize(probe) > 2000:
        print("  ✅ CLONE mode কাজ করছে → সব segment reference-locked হবে")
    else:
        MODE = "preset"
        print("  ⚠️ clone endpoint অচল → পুরো রান preset+seed mode এ (তবুও একই গলা,")
        print("     কারণ সব segment একই voice + একই frozen parameters ব্যবহার করবে)")
else:
    print("  ▶ preset+seed mode")
# anchor profile — drift মাপার রেফারেন্স
anchor_profile = None
if VERIFY_VOICE and os.path.exists(anchor_wav):
    ax, asr = wav_read(anchor_wav)
    anchor_profile = voice_profile(ax, asr)
print()
print("=" * 74)
print("🎭  V9.0 — ONE VOICE, START TO FINISH")
print("=" * 74)
print("  🔊 Mode        : " + MODE.upper())
print("  🎙️ Reference   : " + (anchor_wav if MODE == "clone" else PRESET_VOICE))
print("  🔒 Frozen      : speed=" + str(VOICE["speed"]) +
      "  gs=" + str(VOICE["guidance_scale"]) +
      "  steps=" + str(VOICE["num_step"]) + "  seed=" + str(SEED))
print("  📈 Verify      : " + ("on (threshold " + str(SIMILARITY_THRESHOLD) + ")" if VERIFY_VOICE else "off"))
print("  🎬 Segments    : " + str(len([s for s in SEGMENTS if "text" in s])))
print("  📁 Output      : " + os.path.basename(FINAL_OUTPUT_PATH))
print("=" * 74)
print()
def synth_once(text, out_path):
    ok = (gen_clone(text, anchor_wav, out_path, clone_ep) if MODE == "clone"
          else gen_json(text, out_path))
    if not ok or not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
        return None
    x, sr = wav_read(out_path)
    x = resample(x, sr, TARGET_SR)
    x = trim_silence(x, TARGET_SR, thresh_db=-45.0, pad_ms=40)
    x = normalize_rms(x)                # ← loudness jump বন্ধ
    x = apply_fades(x, TARGET_SR)       # ← click/pop বন্ধ
    return x
def synth_verified(text, tag, idx):
    """generate → fingerprint মিলিয়ে দেখা → না মিললে retry। সেরা attempt রাখে।"""
    best_x, best_score = None, -1.0
    for attempt in range(1, MAX_RETRIES + 1):
        raw = os.path.join(work_dir, "_tmp_" + tag + "_" + str(idx) + "_" + str(attempt) + ".wav")
        x = synth_once(text, raw)
        if x is None:
            print("    ↻ attempt " + str(attempt) + " ব্যর্থ, আবার চেষ্টা...")
            time.sleep(1.5)
            continue
        if not VERIFY_VOICE or anchor_profile is None:
            return x, 1.0
        score = voice_similarity(anchor_profile, voice_profile(x, TARGET_SR))
        if score > best_score:
            best_x, best_score = x, score
        if score >= SIMILARITY_THRESHOLD:
            return x, score
        print("    ⚠️ voice drift ধরা পড়েছে (score " + str(round(score, 3)) +
              " < " + str(SIMILARITY_THRESHOLD) + ") → re-generate " +
              str(attempt) + "/" + str(MAX_RETRIES))
    return best_x, best_score
timeline   = []      # (numpy audio) টুকরোগুলো ক্রমানুসারে
report     = []
failed     = []
t_start    = time.time()
for seg in SEGMENTS:
    if "pause_ms" in seg:
        n = int(TARGET_SR * seg["pause_ms"] / 1000.0)
        timeline.append(np.zeros(n, dtype=np.float32))   # ← সঠিক sample rate!
        continue
    tag = seg["tag"]
    chunks = split_text(seg["text"])
    t0 = time.time()
    pieces, scores = [], []
    for i, chunk in enumerate(chunks):
        x, score = synth_verified(chunk, tag, i)
        if x is None:
            failed.append(tag + "#" + str(i))
            continue
        pieces.append(x)
        scores.append(score)
        if i < len(chunks) - 1:
            pieces.append(np.zeros(int(TARGET_SR * 0.16), dtype=np.float32))
    if not pieces:
        print("  ❌ " + tag + " — কিছুই তৈরি হয়নি")
        continue
    seg_audio = np.concatenate(pieces)
    seg_path  = os.path.join(seg_dir, tag + ".wav")
    wav_write(seg_path, seg_audio, TARGET_SR)
    timeline.append(seg_audio)
    s = min(scores) if scores else 1.0
    report.append((tag, s, len(seg_audio) / TARGET_SR))
    mark = "✅" if s >= SIMILARITY_THRESHOLD else "⚠️"
    print("  " + mark + " " + tag.ljust(20) +
          " │ " + str(round(len(seg_audio) / TARGET_SR, 1)).rjust(5) + "s" +
          " │ match " + str(round(s, 3)) +
          " │ " + str(round(time.time() - t0, 1)) + "s" +
          " │ " + ("🔗clone" if MODE == "clone" else "🔒seed"))
# ── STEP 3: MASTER + CONCAT (সব একই SR, একই লাউডনেস) ─────────────────────────
print()
print("─" * 74)
print("🔗 Concatenating (single sample-rate, single loudness)...")
final = np.concatenate(timeline) if timeline else np.zeros(1, dtype=np.float32)
final = normalize_rms(final, TARGET_RMS_DBFS + 2.0)   # সামান্য গরম master
peak = float(np.max(np.abs(final)) + 1e-9)
if peak > PEAK_CEILING:
    final = final * (PEAK_CEILING / peak)
wav_write(FINAL_OUTPUT_PATH, final, TARGET_SR)
dur = len(final) / TARGET_SR
size_kb = os.path.getsize(FINAL_OUTPUT_PATH) // 1024
print("  🎬 FINAL: " + os.path.basename(FINAL_OUTPUT_PATH) +
      "  (" + str(size_kb) + " KB, " + str(round(dur, 1)) + "s @ " + str(TARGET_SR) + " Hz)")
# ── REPORT ───────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("📋 VOICE CONSISTENCY REPORT")
print("=" * 74)
if report:
    worst = min(r[1] for r in report)
    avg   = sum(r[1] for r in report) / len(report)
    for tag, s, d in report:
        bar = "█" * int(max(0.0, min(1.0, s)) * 24)
        print("  " + tag.ljust(20) + " " + str(round(s, 3)) + "  " + bar)
    print("  " + "-" * 60)
    print("  average match : " + str(round(avg, 3)))
    print("  worst match   : " + str(round(worst, 3)))
    if worst >= SIMILARITY_THRESHOLD:
        print("  ✅ পুরো ট্র্যাকে একজনই কথা বলছে — শুরু থেকে শেষ পর্যন্ত।")
    else:
        print("  ⚠️ কিছু segment এখনো drift করছে। করণীয়:")
        print("     • CUSTOM_VOICE_PATH এ ৮–১২s পরিষ্কার single-speaker ক্লিপ দিন")
        print("     • CUSTOM_VOICE_TEXT এ ক্লিপের হুবহু transcript লিখুন")
        print("     • MAX_CHARS_PER_CALL কমিয়ে 200 করুন")
        print("     • guidance_scale 2.0 → 2.4 করুন (বেশি reference adherence)")
if failed:
    print("  ❌ failed: " + ", ".join(failed))
print("  ⏱️ total: " + str(round(time.time() - t_start, 1)) + "s")
print("=" * 74)
# ── DOWNLOAD ─────────────────────────────────────────────────────────────────
if os.path.exists(FINAL_OUTPUT_PATH):
    with open(FINAL_OUTPUT_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(FINAL_OUTPUT_PATH)
    mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    display(HTML(
        '<audio controls src="data:audio/wav;base64,' + b64 + '" style="width:100%;margin:8px 0"></audio>'
        '<a download="' + dl + '" href="data:audio/wav;base64,' + b64 + '">'
        '<button style="padding:14px 28px;background:linear-gradient(135deg,#667eea,#764ba2);'
        'color:white;border:none;border-radius:8px;cursor:pointer;font-weight:bold;font-size:16px;">'
        '⬇️ Download ' + dl + ' (' + str(round(mb, 1)) + ' MB)</button></a>'))
else:
    print("❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V9.0 — ONE VOICE, START TO FINISH  (Absolute Voice Lock)
# =============================================================================
# সমস্যা: "একটু পর পর ভিন্ন মানুষ কথা বলছে"
# লক্ষ্য : শুরুতে যিনি বলছেন, শেষ পর্যন্ত তিনিই বলবেন।
#
# ─────────────────────────────────────────────────────────────────────────────
# V8.6 তে গলা বদলে যাওয়ার ৫টা আসল কারণ (সবগুলো এখানে ফিক্স করা হয়েছে)
# ─────────────────────────────────────────────────────────────────────────────
# 1) MIXED METHODS  : কিছু segment clone endpoint দিয়ে, কিছু JSON preset দিয়ে
#                     তৈরি হচ্ছিল। দুই ইঞ্জিন = দুই মানুষ। এখন একটাই mode,
#                     পুরো রান জুড়ে — preflight probe দিয়ে আগেই ঠিক করা হয়।
# 2) PER-SEGMENT PARAMS : speed 0.82→0.92, guidance_scale 1.8→2.2,
#                     postprocess_output on/off — এগুলো timbre/formant বদলে
#                     দেয়, তাই প্রতিটা segment আলাদা লোক শোনায়।
#                     এখন সব segment-এ হুবহু একই VOICE dict।
# 3) SAMPLE-RATE BUG : make_silence() 22050 Hz লিখত, কিন্তু OmniVoice সাধারণত
#                     24000 Hz দেয়। concat_wavs() প্রথম ফাইলের params নিত →
#                     বাকি সব ক্লিপ ভুল রেটে বাজত = pitch shift = অন্য মানুষ।
#                     এখন সব কিছু TARGET_SR এ resample করে জোড়া হয়।
# 4) LOUDNESS JUMP  : segment-ভেদে RMS আলাদা হলে কান "নতুন বক্তা" ধরে নেয়।
#                     এখন প্রতিটা segment একই RMS (-20 dBFS) এ normalize।
# 5) NO VERIFICATION : খারাপ generation ধরা পড়ত না। এখন প্রতিটা segment-এর
#                     spectral+pitch fingerprint anchor-এর সাথে মিলিয়ে দেখা হয়,
#                     না মিললে অটো re-generate (MAX_RETRIES বার)।
#
# 📖 নীতি: "Reference audio = voice identity. Parameters = must stay frozen."
# =============================================================================
import os, sys, time, json, math, wave, glob, base64, shutil, subprocess
import urllib.request, urllib.error
import numpy as np
from IPython.display import HTML, display
# ═════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION  — এই ব্লকটাই শুধু ছোঁবেন
# ═════════════════════════════════════════════════════════════════════════════
HOST            = "http://localhost:3900"
API_URL         = HOST + "/v1/audio/speech"
CLONE_URL       = HOST + "/v1/audio/speech/clone"
HEALTH_URL      = HOST + "/health"
OPENAPI_URL     = HOST + "/openapi.json"
BASE_OUTPUT_DIR = "/kaggle/working/outputs/v9_one_voice"
FILENAME        = "v9_one_voice"
# আপনার ভয়েস স্যাম্পল (৫–১৫ সেকেন্ড, একজনের গলা, নয়েজ-মুক্ত, স্পষ্ট)
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
# reference clip-এর হুবহু transcript জানা থাকলে লিখুন (Whisper-এর ভুল এড়ায়,
# ফলে প্রতিবার একই conditioning → গলা আরও স্থির)। না জানলে "" রাখুন।
CUSTOM_VOICE_TEXT = ""
PRESET_VOICE    = "onyx"   # শুধু fallback / anchor তৈরির জন্য
SEED            = 1234
# ── 🔒 FROZEN VOICE PARAMETERS ───────────────────────────────────────────────
# এই ডিক্ট প্রতিটা segment-এ হুবহু একইভাবে যাবে। কখনো per-segment override
# করবেন না — এটাই V8.6-এর সবচেয়ে বড় ভুল ছিল।
# আবেগ (emotion) নিয়ন্ত্রণ করুন শুধু: শব্দচয়ন, কমা/ড্যাশ/ellipsis, আর pause
# length দিয়ে — parameter দিয়ে নয়।
VOICE = {
    "speed":              0.88,
    "guidance_scale":     2.0,
    "num_step":           32,
    "class_temperature":  0.3,
    "postprocess_output": True,
}
# ── Audio pipeline ───────────────────────────────────────────────────────────
TARGET_SR            = 24000    # সব কিছু এখানে resample হবে
TARGET_RMS_DBFS      = -20.0    # সব segment একই লাউডনেসে
PEAK_CEILING         = 0.97
FADE_MS              = 12       # click/pop ঠেকাতে
# ── Voice-drift guard ────────────────────────────────────────────────────────
VERIFY_VOICE         = True
SIMILARITY_THRESHOLD = 0.86     # 0.80 = ঢিলা, 0.90 = কড়া
MAX_RETRIES          = 3
MAX_CHARS_PER_CALL   = 300      # লম্বা টেক্সট ভাঙা হয়, প্যারামিটার একই থাকে
# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — শুধু text + pause. কোনো per-segment voice parameter নেই।
# ═════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text":
        "What if the greatest invention in human history... "
        "wasn't a machine that traveled through space, "
        "but one that crossed possibilities?"},
    {"pause_ms": 450},
    {"tag": "02_doubt", "text":
        "Another universe? Seriously? "
        "Scientists kept working. Everyone else kept doubting."},
    {"pause_ms": 350},
    {"tag": "03_wonder", "text":
        "And suddenly, the impossible became real. "
        "We saw dinosaurs, still walking, beneath blood-red skies. "
        "We found another Earth, where humanity was born on Mars."},
    {"pause_ms": 700},
    {"tag": "04_shift", "text":
        "But me... no. None of those worlds mattered. Not one. "
        "I was searching... for someone."},
    {"pause_ms": 800},
    {"tag": "05_grief", "text":
        "Three years ago... cancer... stole my mother. "
        "No warning. No mercy. No second chance."},
    {"pause_ms": 1000},
    {"tag": "06_hospital", "text":
        "I watched the hospital monitor... become... silent. "
        "I held her hand, hoping, just hoping, "
        "she would squeeze mine... one... last... time."},
    {"pause_ms": 1400},
    {"tag": "07_she_never_did", "text":
        "She never did."},
    {"pause_ms": 1000},
    {"tag": "08_pain", "text":
        "I know she will never answer. I know that. "
        "But I still call. "
        "Because sometimes... hope hurts more than reality."},
    {"pause_ms": 800},
    {"tag": "09_plea", "text":
        "Listen. Take me to the universe... where my mother... never died."},
    {"pause_ms": 1200},
    {"tag": "10_warmth", "text":
        "There she was. Alive. Smiling. Making breakfast. "
        "Humming the exact same song she used to sing... "
        "every Sunday morning."},
    {"pause_ms": 1000},
    {"tag": "11_ending", "text":
        "She did not know I was not her son. "
        "I was a broken man... borrowing someone else's miracle."},
]
# calibration line — custom voice না থাকলে এটা দিয়ে anchor তৈরি হয়
ANCHOR_TEXT = ("This is my natural speaking voice, calm and steady, "
               "and I will keep this exact same voice from the first word "
               "to the very last one.")
# ═════════════════════════════════════════════════════════════════════════════
# 🧰 LOW-LEVEL AUDIO UTILITIES  (numpy only — কোনো heavy dependency নেই)
# ═════════════════════════════════════════════════════════════════════════════
def have_ffmpeg():
    return shutil.which("ffmpeg") is not None
def wav_read(path):
    """WAV → (float32 mono [-1,1], sample_rate)"""
    with wave.open(path, "rb") as wf:
        nch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
        raw = wf.readframes(n)
    if sw == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sw == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError("Unsupported sample width: " + str(sw))
    if nch > 1:
        x = x.reshape(-1, nch).mean(axis=1)
    return x.astype(np.float32), sr
def wav_write(path, x, sr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    x = np.clip(x, -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm)
def resample(x, sr_in, sr_out):
    """High-quality-enough linear resampler (mono)."""
    if sr_in == sr_out or len(x) == 0:
        return x
    n_out = int(round(len(x) * float(sr_out) / float(sr_in)))
    t_in  = np.arange(len(x), dtype=np.float64)
    t_out = np.linspace(0.0, len(x) - 1.0, n_out)
    return np.interp(t_out, t_in, x).astype(np.float32)
def decode_to_wav(src, dst, sr=TARGET_SR):
    """যেকোনো audio (mp3/m4a/flac/ogg/wav) → mono WAV @ sr"""
    if have_ffmpeg():
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
               "-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", dst]
        if subprocess.run(cmd).returncode == 0 and os.path.exists(dst):
            return dst
    if src.lower().endswith(".wav"):
        x, s = wav_read(src)
        wav_write(dst, resample(x, s, sr), sr)
        return dst
    raise RuntimeError("ffmpeg নেই এবং reference WAV নয়: " + src)
def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))
def normalize_rms(x, target_dbfs=TARGET_RMS_DBFS, ceiling=PEAK_CEILING):
    cur = rms(x)
    if cur < 1e-6:
        return x
    gain = (10.0 ** (target_dbfs / 20.0)) / cur
    y = x * gain
    peak = float(np.max(np.abs(y)) + 1e-12)
    if peak > ceiling:
        y = y * (ceiling / peak)
    return y.astype(np.float32)
def apply_fades(x, sr, ms=FADE_MS):
    n = min(int(sr * ms / 1000.0), len(x) // 2)
    if n <= 0:
        return x
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x.copy()
    x[:n]  *= ramp
    x[-n:] *= ramp[::-1]
    return x
def trim_silence(x, sr, thresh_db=-42.0, pad_ms=60):
    """শুরু/শেষের নীরবতা কাটে — reference clip পরিষ্কার করার জন্য।"""
    win = max(1, int(sr * 0.02))
    if len(x) < win * 3:
        return x
    frames = x[:len(x) - len(x) % win].reshape(-1, win)
    energy = 20.0 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)
    loud = np.where(energy > thresh_db)[0]
    if len(loud) == 0:
        return x
    pad = int(sr * pad_ms / 1000.0)
    a = max(0, loud[0] * win - pad)
    b = min(len(x), (loud[-1] + 1) * win + pad)
    return x[a:b]
def best_window(x, sr, seconds=9.0):
    """সবচেয়ে energetic টানা অংশ বেছে নেয় (reference-এর জন্য আদর্শ)।"""
    want = int(sr * seconds)
    if len(x) <= want:
        return x
    hop = int(sr * 0.25)
    best_i, best_e = 0, -1.0
    for i in range(0, len(x) - want, hop):
        e = rms(x[i:i + want])
        if e > best_e:
            best_e, best_i = e, i
    return x[best_i:best_i + want]
# ═════════════════════════════════════════════════════════════════════════════
# 🧬 SPEAKER FINGERPRINT  (spectral envelope + pitch) — drift ধরার জন্য
# ═════════════════════════════════════════════════════════════════════════════
def spectral_fingerprint(x, sr, n_bands=40):
    n, hop = 1024, 512
    if len(x) < n * 2:
        return None
    w = np.hanning(n).astype(np.float32)
    acc, cnt = None, 0
    for i in range(0, len(x) - n, hop):
        seg = x[i:i + n]
        if rms(seg) < 0.005:            # নীরব ফ্রেম বাদ
            continue
        spec = np.abs(np.fft.rfft(seg * w))
        acc = spec if acc is None else acc + spec
        cnt += 1
    if cnt < 4:
        return None
    S = acc / cnt
    edges = np.linspace(0, len(S), n_bands + 1).astype(int)
    bands = np.array([S[edges[i]:edges[i + 1]].mean() + 1e-9 for i in range(n_bands)])
    v = np.log(bands)
    v = v - v.mean()
    nrm = np.linalg.norm(v) + 1e-9
    return (v / nrm).astype(np.float32)
def median_f0(x, sr, fmin=60.0, fmax=320.0):
    n, hop = 2048, 1024
    vals = []
    for i in range(0, max(1, len(x) - n), hop):
        seg = x[i:i + n]
        if len(seg) < n or rms(seg) < 0.01:
            continue
        seg = seg - seg.mean()
        ac = np.correlate(seg, seg, mode="full")[n - 1:]
        lo, hi = int(sr / fmax), int(sr / fmin)
        if hi >= len(ac):
            continue
        k = int(np.argmax(ac[lo:hi])) + lo
        if ac[k] > 0.3 * (ac[0] + 1e-9):
            vals.append(sr / float(k))
    return float(np.median(vals)) if len(vals) >= 3 else 0.0
def voice_profile(x, sr):
    return {"spec": spectral_fingerprint(x, sr), "f0": median_f0(x, sr)}
def voice_similarity(a, b):
    """0..1 — 1 মানে হুবহু একই বক্তা।"""
    if a is None or b is None or a["spec"] is None or b["spec"] is None:
        return 1.0
    cos = float(np.dot(a["spec"], b["spec"]))
    cos = max(0.0, min(1.0, (cos + 1.0) / 2.0 * 1.0 if cos < 0 else cos))
    if a["f0"] > 0 and b["f0"] > 0:
        ratio = min(a["f0"], b["f0"]) / max(a["f0"], b["f0"])
        pitch = max(0.0, 1.0 - (1.0 - ratio) * 2.5)     # 20% pitch drift = 0.5
    else:
        pitch = cos
    return 0.72 * cos + 0.28 * pitch
# ═════════════════════════════════════════════════════════════════════════════
# 🌐 SERVER + ENDPOINT
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False
def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None
def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! আগে Step 2 রান করুন।")
        return False
    print("  📂 " + studio)
    log = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log, stderr=log, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print("  ✅ Server ready! (" + str((i + 1) * 2) + "s)")
            return True
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False
def discover_clone_endpoint():
    """clone route আছে কিনা openapi.json থেকে দেখি।"""
    try:
        with urllib.request.urlopen(urllib.request.Request(OPENAPI_URL), timeout=10) as r:
            spec = json.loads(r.read().decode())
        for path, methods in sorted(spec.get("paths", {}).items()):
            if "clone" in path and "post" in methods:
                return path
    except Exception as e:
        print("  ⚠️ openapi.json পড়া যায়নি: " + str(e))
    return None
# ═════════════════════════════════════════════════════════════════════════════
# 📡 REQUESTS
# ═════════════════════════════════════════════════════════════════════════════
def post_multipart(url, fields, file_field, file_path, timeout=600):
    boundary = "----OmniVoiceLock" + str(int(time.time() * 1000))
    body = b""
    for k, v in fields.items():
        body += ("--" + boundary + "\r\n").encode()
        body += ('Content-Disposition: form-data; name="' + k + '"\r\n\r\n').encode()
        body += (str(v) + "\r\n").encode()
    fname = os.path.basename(file_path)
    ext = os.path.splitext(fname)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")
    body += ("--" + boundary + "\r\n").encode()
    body += ('Content-Disposition: form-data; name="' + file_field +
             '"; filename="' + fname + '"\r\n').encode()
    body += ("Content-Type: " + mime + "\r\n\r\n").encode()
    with open(file_path, "rb") as f:
        body += f.read()
    body += b"\r\n"
    body += ("--" + boundary + "--\r\n").encode()
    req = urllib.request.Request(
        url, data=body, method="POST",
        headers={"Content-Type": "multipart/form-data; boundary=" + boundary})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()
def clone_fields(text, minimal=False):
    """প্রতিটা call-এ হুবহু একই parameter set — এটাই voice lock-এর মূল।"""
    f = {"text": text, "input": text, "seed": SEED, "speed": VOICE["speed"]}
    if CUSTOM_VOICE_TEXT:
        f["ref_text"] = CUSTOM_VOICE_TEXT
    if minimal:
        return {k: v for k, v in f.items() if k in ("text", "input", "ref_text")}
    f["guidance_scale"] = VOICE["guidance_scale"]
    f["num_step"] = VOICE["num_step"]
    f["response_format"] = "wav"
    return f
def gen_clone(text, ref_wav, out_path, clone_ep):
    url = HOST + clone_ep if clone_ep else CLONE_URL
    for minimal in (False, True):     # অতিরিক্ত field-এ 422 হলে minimal-এ retry
        try:
            data = post_multipart(url, clone_fields(text, minimal), "ref_audio", ref_wav)
            with open(out_path, "wb") as f:
                f.write(data)
            return True
        except urllib.error.HTTPError as e:
            msg = ""
            try:
                msg = e.read().decode()[:180]
            except Exception:
                pass
            if e.code in (400, 422) and not minimal:
                continue
            print("    ❌ clone HTTP " + str(e.code) + ": " + msg)
            return False
        except Exception as e:
            print("    ❌ clone error: " + str(e))
            return False
    return False
def gen_json(text, out_path, voice=None):
    """Fallback — clone route না থাকলে। এখানেও প্যারামিটার হুবহু frozen।"""
    payload = {
        "model": "tts-1-hd",
        "voice": voice or PRESET_VOICE,
        "input": text,
        "response_format": "wav",
        "speed": VOICE["speed"],
        "num_step": VOICE["num_step"],
        "guidance_scale": VOICE["guidance_scale"],
        "class_temperature": VOICE["class_temperature"],
        "postprocess_output": VOICE["postprocess_output"],
        "seed": SEED,
    }
    req = urllib.request.Request(API_URL, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            data = r.read()
        with open(out_path, "wb") as f:
            f.write(data)
        return True
    except Exception as e:
        print("    ❌ json error: " + str(e))
        return False
# ═════════════════════════════════════════════════════════════════════════════
# ✂️  TEXT CHUNKING — লম্বা লাইন ভাঙে, কিন্তু voice setting একই থাকে
# ═════════════════════════════════════════════════════════════════════════════
def split_text(text, limit=MAX_CHARS_PER_CALL):
    text = " ".join(text.split())
    if len(text) <= limit:
        return [text]
    parts, buf = [], ""
    for piece in text.replace("? ", "?|").replace("! ", "!|").replace(". ", ".|").split("|"):
        if len(buf) + len(piece) + 1 <= limit:
            buf = (buf + " " + piece).strip()
        else:
            if buf:
                parts.append(buf)
            buf = piece.strip()
    if buf:
        parts.append(buf)
    return parts
# ═════════════════════════════════════════════════════════════════════════════
# 🚀 PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    v = 1
    while os.path.exists(os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")):
        v += 1
    return v, os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")
def auto_discover_voice(given):
    if given and os.path.exists(given):
        return given
    print("  ⚠️ দেওয়া পাথে ফাইল নেই: " + str(given))
    found = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                found.append(os.path.join(root, f))
    if not found:
        print("  ❌ /kaggle/input/ এ কোনো অডিও নেই → anchor TTS দিয়ে বানানো হবে")
        return None
    found.sort()
    print("  📋 পাওয়া গেছে " + str(len(found)) + " টি অডিও, ব্যবহার: " + found[0])
    return found[0]
print("=" * 74)
print("🔧 SERVER CHECK")
print("=" * 74)
if server_alive():
    print("  ✅ Server running")
elif not start_server():
    raise RuntimeError("Server failed. Step 2 & 3 আগে চালান।")
clone_ep = discover_clone_endpoint()
print("  🔗 Clone endpoint: " + str(clone_ep or "(default " + CLONE_URL + ")"))
RUN_VERSION, FINAL_OUTPUT_PATH = next_version(BASE_OUTPUT_DIR, FILENAME)
seg_dir  = os.path.join(BASE_OUTPUT_DIR, "segments_v" + str(RUN_VERSION))
work_dir = os.path.join(BASE_OUTPUT_DIR, "work")
os.makedirs(seg_dir, exist_ok=True)
os.makedirs(work_dir, exist_ok=True)
# ── STEP 1: ANCHOR (একটাই গলা, পুরো ট্র্যাকের ভিত্তি) ────────────────────────
print()
print("=" * 74)
print("🎙️  STEP 1 — ANCHOR VOICE তৈরি")
print("=" * 74)
anchor_wav = os.path.join(work_dir, "anchor.wav")
src_voice  = auto_discover_voice(CUSTOM_VOICE_PATH)
MODE = None      # "clone" বা "preset" — একবার ঠিক হলে সারা রান একই থাকবে
if src_voice:
    tmp = os.path.join(work_dir, "_ref_raw.wav")
    decode_to_wav(src_voice, tmp, TARGET_SR)
    x, sr = wav_read(tmp)
    x = best_window(trim_silence(x, sr), sr, 9.0)
    x = apply_fades(normalize_rms(x), sr)
    wav_write(anchor_wav, x, sr)
    print("  ✅ Custom reference: " + src_voice)
    print("     → anchor.wav  " + str(round(len(x) / sr, 1)) + "s @ " + str(sr) + " Hz")
    MODE = "clone"
else:
    print("  🔁 custom voice নেই → preset '" + PRESET_VOICE + "' দিয়ে anchor বানাচ্ছি...")
    raw = os.path.join(work_dir, "_anchor_raw.wav")
    if gen_json(ANCHOR_TEXT, raw):
        x, sr = wav_read(raw)
        x = resample(x, sr, TARGET_SR)
        x = apply_fades(normalize_rms(best_window(trim_silence(x, TARGET_SR), TARGET_SR, 9.0)), TARGET_SR)
        wav_write(anchor_wav, x, TARGET_SR)
        print("  ✅ anchor.wav তৈরি হয়েছে preset voice থেকে")
        MODE = "clone"
    else:
        print("  ⚠️ anchor বানানো যায়নি → পুরো রান preset+seed mode এ চলবে")
        MODE = "preset"
# ── STEP 2: PREFLIGHT PROBE — mode একবারেই লক ─────────────────────────────────
print()
print("=" * 74)
print("🧪 STEP 2 — PREFLIGHT (mode লক করা হচ্ছে, মাঝপথে আর বদলাবে না)")
print("=" * 74)
if MODE == "clone":
    probe = os.path.join(work_dir, "_probe.wav")
    ok = gen_clone("Testing one single voice.", anchor_wav, probe, clone_ep)
    if ok and os.path.getsize(probe) > 2000:
        print("  ✅ CLONE mode কাজ করছে → সব segment reference-locked হবে")
    else:
        MODE = "preset"
        print("  ⚠️ clone endpoint অচল → পুরো রান preset+seed mode এ (তবুও একই গলা,")
        print("     কারণ সব segment একই voice + একই frozen parameters ব্যবহার করবে)")
else:
    print("  ▶ preset+seed mode")
# anchor profile — drift মাপার রেফারেন্স
anchor_profile = None
if VERIFY_VOICE and os.path.exists(anchor_wav):
    ax, asr = wav_read(anchor_wav)
    anchor_profile = voice_profile(ax, asr)
print()
print("=" * 74)
print("🎭  V9.0 — ONE VOICE, START TO FINISH")
print("=" * 74)
print("  🔊 Mode        : " + MODE.upper())
print("  🎙️ Reference   : " + (anchor_wav if MODE == "clone" else PRESET_VOICE))
print("  🔒 Frozen      : speed=" + str(VOICE["speed"]) +
      "  gs=" + str(VOICE["guidance_scale"]) +
      "  steps=" + str(VOICE["num_step"]) + "  seed=" + str(SEED))
print("  📈 Verify      : " + ("on (threshold " + str(SIMILARITY_THRESHOLD) + ")" if VERIFY_VOICE else "off"))
print("  🎬 Segments    : " + str(len([s for s in SEGMENTS if "text" in s])))
print("  📁 Output      : " + os.path.basename(FINAL_OUTPUT_PATH))
print("=" * 74)
print()
def synth_once(text, out_path):
    ok = (gen_clone(text, anchor_wav, out_path, clone_ep) if MODE == "clone"
          else gen_json(text, out_path))
    if not ok or not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
        return None
    x, sr = wav_read(out_path)
    x = resample(x, sr, TARGET_SR)
    x = trim_silence(x, TARGET_SR, thresh_db=-45.0, pad_ms=40)
    x = normalize_rms(x)                # ← loudness jump বন্ধ
    x = apply_fades(x, TARGET_SR)       # ← click/pop বন্ধ
    return x
def synth_verified(text, tag, idx):
    """generate → fingerprint মিলিয়ে দেখা → না মিললে retry। সেরা attempt রাখে।"""
    best_x, best_score = None, -1.0
    for attempt in range(1, MAX_RETRIES + 1):
        raw = os.path.join(work_dir, "_tmp_" + tag + "_" + str(idx) + "_" + str(attempt) + ".wav")
        x = synth_once(text, raw)
        if x is None:
            print("    ↻ attempt " + str(attempt) + " ব্যর্থ, আবার চেষ্টা...")
            time.sleep(1.5)
            continue
        if not VERIFY_VOICE or anchor_profile is None:
            return x, 1.0
        score = voice_similarity(anchor_profile, voice_profile(x, TARGET_SR))
        if score > best_score:
            best_x, best_score = x, score
        if score >= SIMILARITY_THRESHOLD:
            return x, score
        print("    ⚠️ voice drift ধরা পড়েছে (score " + str(round(score, 3)) +
              " < " + str(SIMILARITY_THRESHOLD) + ") → re-generate " +
              str(attempt) + "/" + str(MAX_RETRIES))
    return best_x, best_score
timeline   = []      # (numpy audio) টুকরোগুলো ক্রমানুসারে
report     = []
failed     = []
t_start    = time.time()
for seg in SEGMENTS:
    if "pause_ms" in seg:
        n = int(TARGET_SR * seg["pause_ms"] / 1000.0)
        timeline.append(np.zeros(n, dtype=np.float32))   # ← সঠিক sample rate!
        continue
    tag = seg["tag"]
    chunks = split_text(seg["text"])
    t0 = time.time()
    pieces, scores = [], []
    for i, chunk in enumerate(chunks):
        x, score = synth_verified(chunk, tag, i)
        if x is None:
            failed.append(tag + "#" + str(i))
            continue
        pieces.append(x)
        scores.append(score)
        if i < len(chunks) - 1:
            pieces.append(np.zeros(int(TARGET_SR * 0.16), dtype=np.float32))
    if not pieces:
        print("  ❌ " + tag + " — কিছুই তৈরি হয়নি")
        continue
    seg_audio = np.concatenate(pieces)
    seg_path  = os.path.join(seg_dir, tag + ".wav")
    wav_write(seg_path, seg_audio, TARGET_SR)
    timeline.append(seg_audio)
    s = min(scores) if scores else 1.0
    report.append((tag, s, len(seg_audio) / TARGET_SR))
    mark = "✅" if s >= SIMILARITY_THRESHOLD else "⚠️"
    print("  " + mark + " " + tag.ljust(20) +
          " │ " + str(round(len(seg_audio) / TARGET_SR, 1)).rjust(5) + "s" +
          " │ match " + str(round(s, 3)) +
          " │ " + str(round(time.time() - t0, 1)) + "s" +
          " │ " + ("🔗clone" if MODE == "clone" else "🔒seed"))
# ── STEP 3: MASTER + CONCAT (সব একই SR, একই লাউডনেস) ─────────────────────────
print()
print("─" * 74)
print("🔗 Concatenating (single sample-rate, single loudness)...")
final = np.concatenate(timeline) if timeline else np.zeros(1, dtype=np.float32)
final = normalize_rms(final, TARGET_RMS_DBFS + 2.0)   # সামান্য গরম master
peak = float(np.max(np.abs(final)) + 1e-9)
if peak > PEAK_CEILING:
    final = final * (PEAK_CEILING / peak)
wav_write(FINAL_OUTPUT_PATH, final, TARGET_SR)
dur = len(final) / TARGET_SR
size_kb = os.path.getsize(FINAL_OUTPUT_PATH) // 1024
print("  🎬 FINAL: " + os.path.basename(FINAL_OUTPUT_PATH) +
      "  (" + str(size_kb) + " KB, " + str(round(dur, 1)) + "s @ " + str(TARGET_SR) + " Hz)")
# ── REPORT ───────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("📋 VOICE CONSISTENCY REPORT")
print("=" * 74)
if report:
    worst = min(r[1] for r in report)
    avg   = sum(r[1] for r in report) / len(report)
    for tag, s, d in report:
        bar = "█" * int(max(0.0, min(1.0, s)) * 24)
        print("  " + tag.ljust(20) + " " + str(round(s, 3)) + "  " + bar)
    print("  " + "-" * 60)
    print("  average match : " + str(round(avg, 3)))
    print("  worst match   : " + str(round(worst, 3)))
    if worst >= SIMILARITY_THRESHOLD:
        print("  ✅ পুরো ট্র্যাকে একজনই কথা বলছে — শুরু থেকে শেষ পর্যন্ত।")
    else:
        print("  ⚠️ কিছু segment এখনো drift করছে। করণীয়:")
        print("     • CUSTOM_VOICE_PATH এ ৮–১২s পরিষ্কার single-speaker ক্লিপ দিন")
        print("     • CUSTOM_VOICE_TEXT এ ক্লিপের হুবহু transcript লিখুন")
        print("     • MAX_CHARS_PER_CALL কমিয়ে 200 করুন")
        print("     • guidance_scale 2.0 → 2.4 করুন (বেশি reference adherence)")
if failed:
    print("  ❌ failed: " + ", ".join(failed))
print("  ⏱️ total: " + str(round(time.time() - t_start, 1)) + "s")
print("=" * 74)
# ── DOWNLOAD ─────────────────────────────────────────────────────────────────
if os.path.exists(FINAL_OUTPUT_PATH):
    with open(FINAL_OUTPUT_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(FINAL_OUTPUT_PATH)
    mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    display(HTML(
        '<audio controls src="data:audio/wav;base64,' + b64 + '" style="width:100%;margin:8px 0"></audio>'
        '<a download="' + dl + '" href="data:audio/wav;base64,' + b64 + '">'
        '<button style="padding:14px 28px;background:linear-gradient(135deg,#667eea,#764ba2);'
        'color:white;border:none;border-radius:8px;cursor:pointer;font-weight:bold;font-size:16px;">'
        '⬇️ Download ' + dl + ' (' + str(round(mb, 1)) + ' MB)</button></a>'))
else:
    print("❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V8.7 — TRUE VOICE LOCK: One Voice from Start to Finish
# -----------------------------------------------------------------------------
# 🔍 PROBLEM (তোমার সমস্যা):
#    "একটু পর পর ভিন্ন মানুষ কথা বলছিল" — segment-এ segment-ে voice বদলে
#    যাচ্ছিল, মনে হচ্ছিল অন্য মানুষ কথা বলছে।
#
# 🎯 ROOT CAUSE (আসল কারণ):
#    আগের version-এ যেকোনো segment-এ clone call fail করলেই সেটা চুপচাপ
#    "onyx" preset voice-এ চলে যেত। ফলে শুধু ঐ একটা segment-এ voice বদলে
#    গিয়ে অন্য মানুষের মতো শোনাতো — এটাই "different people speaking" এর কারণ।
#
# ✅ V8.7 FIXES (এই version-এ যা ঠিক করা হয়েছে):
#    1. ONE ANCHOR    — প্রথম speaker-এর voice = পুরো script-এর fixed reference
#    2. METHOD LOCK   — clone না json, একবার ঠিক করে নিই; মাঝে বদলাবে না
#    3. RETRY, NO MIX — transient fail-এ voice switch নয়, retry করবে
#    4. NO MIXING     — কখনো single segment-এ অন্য voice আসবে না
#    5. FIXED SEED    — সব call-এ একই seed
# =============================================================================

import os, time, json, urllib.request, urllib.error, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_7_voice_locked"
API_URL    = "http://localhost:3900/v1/audio/speech"
CLONE_URL  = "http://localhost:3900/v1/audio/speech/clone"
HEALTH_URL = "http://localhost:3900/health"

# ─── VOICE LOCK CONFIG ───────────────────────────────────────────────────────
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
PRESET_VOICE  = "onyx"        # anchor বানাতে (custom voice না থাকলে)
SEED          = 42                       # FIXED for ALL segments
MAX_RETRIES   = 4                 # প্রতি segment-এ retry (voice switch এড়াতে)
FILENAME      = "v8_7_true_voice_lock"


# ════════════════════════════════════════════════════════════════════════════
# 🔍 AUTO-DISCOVER: voice ফাইল খুঁজে বের করা
# ════════════════════════════════════════════════════════════════════════════
def auto_discover_voice(given_path):
    """দেওয়া পাথে ফাইল না থাকলে /kaggle/input/ এ সব audio ফাইল খুঁজে দেখায়।"""
    if given_path and os.path.exists(given_path):
        return given_path
    if not given_path:
        return None

    print(f"  ⚠️ দেওয়া পাথে ফাইল নেই: {given_path}")
    print(f"  🔍 /kaggle/input/ এ audio ফাইল খুঁজছি...")

    audio_files = []
    search_root = "/kaggle/input"
    if os.path.exists(search_root):
        for root, dirs, files in os.walk(search_root):
            for f in files:
                if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))

    if not audio_files:
        print("  ❌ /kaggle/input/ এ কোনো audio ফাইল পাওয়া যায়নি")
        return None

    print(f"  📋 {len(audio_files)} টি audio ফাইল পাওয়া গেছে:")
    for i, af in enumerate(audio_files):
        print(f"     {i+1}. {af} ({os.path.getsize(af)//1024} KB)")

    chosen = audio_files[0]
    print(f"  ✅ ব্যবহার করা হচ্ছে: {chosen}")
    return chosen


# ════════════════════════════════════════════════════════════════════════════
# 📁 AUTO-VERSIONING
# ════════════════════════════════════════════════════════════════════════════
def get_next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    version = 1
    while True:
        candidate = os.path.join(base_dir, f"{base_name}_v{version}.wav")
        if not os.path.exists(candidate):
            return version, candidate
        version += 1

RUN_VERSION, FINAL_OUTPUT_PATH = get_next_version(BASE_OUTPUT_DIR, FILENAME)
print(f"📁 Auto-version: v{RUN_VERSION}  →  {os.path.basename(FINAL_OUTPUT_PATH)}\n")


# ════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! আগে Step 2 রান করো।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()


# ════════════════════════════════════════════════════════════════════════════
# 🔍 ENDPOINT DISCOVERY
# ════════════════════════════════════════════════════════════════════════════
clone_ep = None

def discover_clone_endpoint():
    """সার্ভারে /clone endpoint আছে কিনা চেক করি।"""
    try:
        req = urllib.request.Request("http://localhost:3900/openapi.json")
        with urllib.request.urlopen(req, timeout=10) as resp:
            spec = json.loads(resp.read().decode())
            paths = spec.get("paths", {})
            for path in paths:
                if "clone" in path and "post" in paths[path]:
                    print(f"  🔗 Clone endpoint পাওয়া গেছে: {path}")
                    return path
            print("  🔒 কোনো clone endpoint নেই — JSON + seed fallback ব্যবহার হবে")
            return None
    except Exception as e:
        print(f"  ⚠️ OpenAPI spec পড়া যায়নি: {e}")
        return None

print("=" * 70)
print("🔍 ENDPOINT DISCOVERY")
print("=" * 70)
clone_ep = discover_clone_endpoint()
print()


# ════════════════════════════════════════════════════════════════════════════
# 🎵 CORE FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ কোনো WAV ফাইল নেই!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_with_clone(text, ref_audio_path, output_path, speed=0.88,
                   guidance_scale=2.0, seed=42):
    """🔗 /clone endpoint — multipart/form-data. Reference audio থেকে voice clone।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    boundary = "----OmniVoiceClone" + str(int(time.time() * 1000))
    body = b""

    body += f"--{boundary}\r\n".encode()
    body += b'Content-Disposition: form-data; name="text"\r\n\r\n'
    body += f"{text}\r\n".encode()

    body += f"--{boundary}\r\n".encode()
    body += b'Content-Disposition: form-data; name="speed"\r\n\r\n'
    body += f"{speed}\r\n".encode()

    filename = os.path.basename(ref_audio_path)
    ext = os.path.splitext(filename)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")

    body += f"--{boundary}\r\n".encode()
    body += f'Content-Disposition: form-data; name="ref_audio"; filename="{filename}"\r\n'.encode()
    body += f"Content-Type: {mime}\r\n\r\n".encode()
    with open(ref_audio_path, "rb") as f:
        body += f.read()
    body += b"\r\n"
    body += f"--{boundary}--\r\n".encode()

    url = f"http://localhost:3900{clone_ep}" if clone_ep else CLONE_URL
    req = urllib.request.Request(
        url, data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
        method="POST")

    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:34s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s │ 🔗clone")
            return True
    except urllib.error.HTTPError as e:
        error_body = e.read().decode() if hasattr(e, "read") else str(e)
        print(f"  ❌ Clone failed (HTTP {e.code}): {error_body[:160]}")
        return False
    except Exception as e:
        print(f"  ❌ Clone error: {e}")
        return False

def gen_with_json(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                  class_temperature=0.3, postprocess_output=True, seed=42):
    """🔒 Fallback: /v1/audio/speech — JSON (seed lock)। সব segment-ে একই voice।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:34s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s │ 🔒seed")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False


# ════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — SEGMENTS
# ════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text": "What if the greatest invention in human history... wasn't a machine that traveled through space — but one that crossed possibilities?", "speed": 0.9, "guidance_scale": 2.2},
    {"silence_ms": 400},
    {"tag": "02_doubt", "text": "Another universe? Seriously? Scientists kept working. Everyone else kept doubting.", "speed": 0.92, "guidance_scale": 2},
    {"silence_ms": 300},
    {"tag": "03_wonder", "text": "And suddenly — the impossible became real. We saw dinosaurs, still walking, beneath blood-red skies. We found another Earth where humanity was born on Mars.", "speed": 0.9, "guidance_scale": 2.2},
    {"silence_ms": 700},
    {"tag": "04_shift", "text": "But me... no. None of those worlds mattered. Not one. I was searching... for someone.", "speed": 0.85, "guidance_scale": 2, "postprocess_output": False},
    {"silence_ms": 800},
    {"tag": "05_grief", "text": "Three years ago... cancer... stole my mother. No warning. No mercy. No second chance.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1000},
    {"tag": "06_hospital", "text": "I watched the hospital monitor... become... silent. I held her hand — hoping — just hoping — she'd squeeze mine... one... last... time.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1500},
    {"tag": "07_she_never_did", "text": "She never did.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1000},
    {"tag": "08_pain", "text": "I know she'll never answer. I know that. But I still call. Because sometimes... hope hurts more than reality.", "speed": 0.85, "guidance_scale": 2.2},
    {"silence_ms": 800},
    {"tag": "09_plea", "text": "Listen. Take me to the universe... where my mother... never died.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1200},
    {"tag": "10_warmth", "text": "There she was. Alive. Smiling. Making breakfast. Humming the exact same song she used to sing... every Sunday morning.", "speed": 0.88, "guidance_scale": 2},
    {"silence_ms": 1000},
    {"tag": "11_ending", "text": "She didn't know I wasn't her son. I was a broken man... borrowing someone else's miracle.", "speed": 0.85, "guidance_scale": 2, "postprocess_output": False},
]


# ════════════════════════════════════════════════════════════════════════════
# 🚀 V8.7 — TRUE VOICE LOCK  (একটাই গলা, শুরু → শেষ)
# ════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)
speech_segs = [s for s in SEGMENTS if "text" in s]

# ─── 1) ANCHOR ঠিক করা (fixed reference voice) ─────────────────────────────
# anchor একবার সেট হলে আর বদলাবে না। এটাই voice lock-এর মূল।
anchor = auto_discover_voice(CUSTOM_VOICE_PATH)   # None হতে পারে
method = None                                      # 'clone' | 'json'

print("=" * 70)
print("🎭  V8.7 — TRUE VOICE LOCK  (One Voice, Start → Finish)")
print("=" * 70)
print(f"🎙️  Anchor file  : {anchor or '(প্রথম segment থেকে তৈরি হবে)'}")
print(f"🔗  Clone EP     : {clone_ep or CLONE_URL}")
print(f"🔢  Fixed seed   : {SEED}")
print(f"🔁  Retries/seg  : {MAX_RETRIES}")
print(f"🎬  Speech segs  : {len(speech_segs)}")
print("=" * 70)
print()

# ─── 2) METHOD একবার ঠিক করা (probe with segment[0]) ───────────────────────
# প্রথম segment দিয়ে পরীক্ষা। যেটা চলবে সেটাই পুরো script-এ চলবে — মাঝে বদলাবে না।
first = speech_segs[0]
probe_out = os.path.join(seg_dir, first["tag"] + ".wav")

if anchor:
    if gen_with_clone(first["text"], anchor, probe_out,
                      first.get("speed", 0.88), first.get("guidance_scale", 2.0), SEED):
        method = "clone"
        print("🎉  Clone কাজ করছে → পুরো script-এ এই anchor দিয়ে clone হবে\n")

if method != "clone":
    method = "json"
    gen_with_json(first["text"], PRESET_VOICE, probe_out,
                  first.get("speed", 0.88), first.get("guidance_scale", 2.0),
                  0.3, first.get("postprocess_output", True), SEED)
    print("🔒  Clone unavailable → পুরো script-এ JSON + fixed seed ব্যবহার হবে\n")

# custom voice না থাকলে প্রথম segment-ই anchor হয়ে যায়
if method == "clone" and not anchor:
    anchor = probe_out
    print(f"🔄  প্রথম segment anchor হিসেবে সেট হলো: {first['tag']}\n")


# ─── 3) gen_locked: method অনুযায়ী generate, fail হলে retry, voice বদলায় না ─
def gen_locked(seg):
    text  = seg["text"]
    out   = os.path.join(seg_dir, seg["tag"] + ".wav")
    speed = seg.get("speed", 0.88)
    gs    = seg.get("guidance_scale", 2.0)
    pp    = seg.get("postprocess_output", True)

    if method == "clone" and anchor:
        for attempt in range(MAX_RETRIES):
            if gen_with_clone(text, anchor, out, speed, gs, SEED):
                return out
            print(f"     ↻ retry {attempt + 1}/{MAX_RETRIES}  (voice switch নয়, retry...)")
            time.sleep(2)
        # সব retry fail → clone mode-এ JSON-এ যাই না (mixing এড়াতে); জায়গা ফাঁকা রাখি
        print(f"     ⚠️  {seg['tag']} clone করা গেল না — জায়গা ফাঁকা রাখা হলো (voice mix নয়)")
        return None
    else:
        for attempt in range(MAX_RETRIES):
            if gen_with_json(text, PRESET_VOICE, out, speed, gs, 0.3, pp, SEED):
                return out
            print(f"     ↻ retry {attempt + 1}/{MAX_RETRIES} ...")
            time.sleep(2)
        return None


# ─── 4) সব segment generate (প্রথমটা probe-এ তৈরি, বাকিগুলো locked method) ──
wav_order = []
generated = 0
failed = []
sil_idx = 0
first_done = False

for seg in SEGMENTS:
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    out_path = os.path.join(seg_dir, seg["tag"] + ".wav")
    if not first_done:
        first_done = True
        # probe-এ আগেই তৈরি হয়ে গেছে — আবার generate করি না (voice একই থাকে)
        res = out_path if os.path.exists(out_path) else gen_locked(seg)
    else:
        res = gen_locked(seg)

    if res and os.path.exists(res):
        wav_order.append(res)
        generated += 1
    else:
        failed.append(seg["tag"])

# ─── Concatenate ─────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = FINAL_OUTPUT_PATH
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Skipped (no voice mix): {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")

# ─── Voice Consistency Report ────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("📋 VOICE CONSISTENCY REPORT:")
if method == "clone":
    print("  🔗 Method: Reference Audio Cloning (BEST — one voice locked)")
    print(f"  🎙️ Anchor : {anchor}")
else:
    print(f"  🔒 Method: Seed-lock fallback (one preset voice, fixed seed)")
    print(f"  🎙️ Voice : {PRESET_VOICE}, Seed: {SEED}")
print("  ✅ No segment switched to a different voice.")
print(f"{'=' * 70}")

# ─── Direct Download (base64 — no 404) ───────────────────────────────────────
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024 * 1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print("\n⬇️  নিচের বাটনে ক্লিক করো:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V9.0 — ONE VOICE, START TO FINISH  (Absolute Voice Lock)
# =============================================================================
# সমস্যা: "একটু পর পর ভিন্ন মানুষ কথা বলছে"
# লক্ষ্য : শুরুতে যিনি বলছেন, শেষ পর্যন্ত তিনিই বলবেন।
#
# ─────────────────────────────────────────────────────────────────────────────
# V8.6 তে গলা বদলে যাওয়ার ৫টা আসল কারণ (সবগুলো এখানে ফিক্স করা হয়েছে)
# ─────────────────────────────────────────────────────────────────────────────
# 1) MIXED METHODS  : কিছু segment clone endpoint দিয়ে, কিছু JSON preset দিয়ে
#                     তৈরি হচ্ছিল। দুই ইঞ্জিন = দুই মানুষ। এখন একটাই mode,
#                     পুরো রান জুড়ে — preflight probe দিয়ে আগেই ঠিক করা হয়।
# 2) PER-SEGMENT PARAMS : speed 0.82→0.92, guidance_scale 1.8→2.2,
#                     postprocess_output on/off — এগুলো timbre/formant বদলে
#                     দেয়, তাই প্রতিটা segment আলাদা লোক শোনায়।
#                     এখন সব segment-এ হুবহু একই VOICE dict।
# 3) SAMPLE-RATE BUG : make_silence() 22050 Hz লিখত, কিন্তু OmniVoice সাধারণত
#                     24000 Hz দেয়। concat_wavs() প্রথম ফাইলের params নিত →
#                     বাকি সব ক্লিপ ভুল রেটে বাজত = pitch shift = অন্য মানুষ।
#                     এখন সব কিছু TARGET_SR এ resample করে জোড়া হয়।
# 4) LOUDNESS JUMP  : segment-ভেদে RMS আলাদা হলে কান "নতুন বক্তা" ধরে নেয়।
#                     এখন প্রতিটা segment একই RMS (-20 dBFS) এ normalize।
# 5) NO VERIFICATION : খারাপ generation ধরা পড়ত না। এখন প্রতিটা segment-এর
#                     spectral+pitch fingerprint anchor-এর সাথে মিলিয়ে দেখা হয়,
#                     না মিললে অটো re-generate (MAX_RETRIES বার)।
#
# 📖 নীতি: "Reference audio = voice identity. Parameters = must stay frozen."
# =============================================================================

import os, sys, time, json, math, wave, glob, base64, shutil, subprocess
import urllib.request, urllib.error
import numpy as np
from IPython.display import HTML, display

# ═════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION  — এই ব্লকটাই শুধু ছোঁবেন
# ═════════════════════════════════════════════════════════════════════════════
HOST            = "http://localhost:3900"
API_URL         = HOST + "/v1/audio/speech"
CLONE_URL       = HOST + "/v1/audio/speech/clone"
HEALTH_URL      = HOST + "/health"
OPENAPI_URL     = HOST + "/openapi.json"

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v9_one_voice"
FILENAME        = "v9_one_voice"

# আপনার ভয়েস স্যাম্পল (৫–১৫ সেকেন্ড, একজনের গলা, নয়েজ-মুক্ত, স্পষ্ট)
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
# reference clip-এর হুবহু transcript জানা থাকলে লিখুন (Whisper-এর ভুল এড়ায়,
# ফলে প্রতিবার একই conditioning → গলা আরও স্থির)। না জানলে "" রাখুন।
CUSTOM_VOICE_TEXT = ""

PRESET_VOICE    = "onyx"   # শুধু fallback / anchor তৈরির জন্য
SEED            = 1234

# ── 🔒 FROZEN VOICE PARAMETERS ───────────────────────────────────────────────
# এই ডিক্ট প্রতিটা segment-এ হুবহু একইভাবে যাবে। কখনো per-segment override
# করবেন না — এটাই V8.6-এর সবচেয়ে বড় ভুল ছিল।
# আবেগ (emotion) নিয়ন্ত্রণ করুন শুধু: শব্দচয়ন, কমা/ড্যাশ/ellipsis, আর pause
# length দিয়ে — parameter দিয়ে নয়।
VOICE = {
    "speed":              0.88,
    "guidance_scale":     2.0,
    "num_step":           32,
    "class_temperature":  0.3,
    "postprocess_output": True,
}

# ── Audio pipeline ───────────────────────────────────────────────────────────
TARGET_SR            = 24000    # সব কিছু এখানে resample হবে
TARGET_RMS_DBFS      = -20.0    # সব segment একই লাউডনেসে
PEAK_CEILING         = 0.97
FADE_MS              = 12       # click/pop ঠেকাতে

# ── Voice-drift guard ────────────────────────────────────────────────────────
VERIFY_VOICE         = True
SIMILARITY_THRESHOLD = 0.86     # 0.80 = ঢিলা, 0.90 = কড়া
MAX_RETRIES          = 3
MAX_CHARS_PER_CALL   = 300      # লম্বা টেক্সট ভাঙা হয়, প্যারামিটার একই থাকে


# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — শুধু text + pause. কোনো per-segment voice parameter নেই।
# ═════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text":
        "What if the greatest invention in human history... "
        "wasn't a machine that traveled through space, "
        "but one that crossed possibilities?"},
    {"pause_ms": 450},

    {"tag": "02_doubt", "text":
        "Another universe? Seriously? "
        "Scientists kept working. Everyone else kept doubting."},
    {"pause_ms": 350},

    {"tag": "03_wonder", "text":
        "And suddenly, the impossible became real. "
        "We saw dinosaurs, still walking, beneath blood-red skies. "
        "We found another Earth, where humanity was born on Mars."},
    {"pause_ms": 700},

    {"tag": "04_shift", "text":
        "But me... no. None of those worlds mattered. Not one. "
        "I was searching... for someone."},
    {"pause_ms": 800},

    {"tag": "05_grief", "text":
        "Three years ago... cancer... stole my mother. "
        "No warning. No mercy. No second chance."},
    {"pause_ms": 1000},

    {"tag": "06_hospital", "text":
        "I watched the hospital monitor... become... silent. "
        "I held her hand, hoping, just hoping, "
        "she would squeeze mine... one... last... time."},
    {"pause_ms": 1400},

    {"tag": "07_she_never_did", "text":
        "She never did."},
    {"pause_ms": 1000},

    {"tag": "08_pain", "text":
        "I know she will never answer. I know that. "
        "But I still call. "
        "Because sometimes... hope hurts more than reality."},
    {"pause_ms": 800},

    {"tag": "09_plea", "text":
        "Listen. Take me to the universe... where my mother... never died."},
    {"pause_ms": 1200},

    {"tag": "10_warmth", "text":
        "There she was. Alive. Smiling. Making breakfast. "
        "Humming the exact same song she used to sing... "
        "every Sunday morning."},
    {"pause_ms": 1000},

    {"tag": "11_ending", "text":
        "She did not know I was not her son. "
        "I was a broken man... borrowing someone else's miracle."},
]

# calibration line — custom voice না থাকলে এটা দিয়ে anchor তৈরি হয়
ANCHOR_TEXT = ("This is my natural speaking voice, calm and steady, "
               "and I will keep this exact same voice from the first word "
               "to the very last one.")


# ═════════════════════════════════════════════════════════════════════════════
# 🧰 LOW-LEVEL AUDIO UTILITIES  (numpy only — কোনো heavy dependency নেই)
# ═════════════════════════════════════════════════════════════════════════════
def have_ffmpeg():
    return shutil.which("ffmpeg") is not None

def wav_read(path):
    """WAV → (float32 mono [-1,1], sample_rate)"""
    with wave.open(path, "rb") as wf:
        nch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
        raw = wf.readframes(n)
    if sw == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sw == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError("Unsupported sample width: " + str(sw))
    if nch > 1:
        x = x.reshape(-1, nch).mean(axis=1)
    return x.astype(np.float32), sr

def wav_write(path, x, sr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    x = np.clip(x, -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm)

def resample(x, sr_in, sr_out):
    """High-quality-enough linear resampler (mono)."""
    if sr_in == sr_out or len(x) == 0:
        return x
    n_out = int(round(len(x) * float(sr_out) / float(sr_in)))
    t_in  = np.arange(len(x), dtype=np.float64)
    t_out = np.linspace(0.0, len(x) - 1.0, n_out)
    return np.interp(t_out, t_in, x).astype(np.float32)

def decode_to_wav(src, dst, sr=TARGET_SR):
    """যেকোনো audio (mp3/m4a/flac/ogg/wav) → mono WAV @ sr"""
    if have_ffmpeg():
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
               "-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", dst]
        if subprocess.run(cmd).returncode == 0 and os.path.exists(dst):
            return dst
    if src.lower().endswith(".wav"):
        x, s = wav_read(src)
        wav_write(dst, resample(x, s, sr), sr)
        return dst
    raise RuntimeError("ffmpeg নেই এবং reference WAV নয়: " + src)

def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))

def normalize_rms(x, target_dbfs=TARGET_RMS_DBFS, ceiling=PEAK_CEILING):
    cur = rms(x)
    if cur < 1e-6:
        return x
    gain = (10.0 ** (target_dbfs / 20.0)) / cur
    y = x * gain
    peak = float(np.max(np.abs(y)) + 1e-12)
    if peak > ceiling:
        y = y * (ceiling / peak)
    return y.astype(np.float32)

def apply_fades(x, sr, ms=FADE_MS):
    n = min(int(sr * ms / 1000.0), len(x) // 2)
    if n <= 0:
        return x
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x.copy()
    x[:n]  *= ramp
    x[-n:] *= ramp[::-1]
    return x

def trim_silence(x, sr, thresh_db=-42.0, pad_ms=60):
    """শুরু/শেষের নীরবতা কাটে — reference clip পরিষ্কার করার জন্য।"""
    win = max(1, int(sr * 0.02))
    if len(x) < win * 3:
        return x
    frames = x[:len(x) - len(x) % win].reshape(-1, win)
    energy = 20.0 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)
    loud = np.where(energy > thresh_db)[0]
    if len(loud) == 0:
        return x
    pad = int(sr * pad_ms / 1000.0)
    a = max(0, loud[0] * win - pad)
    b = min(len(x), (loud[-1] + 1) * win + pad)
    return x[a:b]

def best_window(x, sr, seconds=9.0):
    """সবচেয়ে energetic টানা অংশ বেছে নেয় (reference-এর জন্য আদর্শ)।"""
    want = int(sr * seconds)
    if len(x) <= want:
        return x
    hop = int(sr * 0.25)
    best_i, best_e = 0, -1.0
    for i in range(0, len(x) - want, hop):
        e = rms(x[i:i + want])
        if e > best_e:
            best_e, best_i = e, i
    return x[best_i:best_i + want]


# ═════════════════════════════════════════════════════════════════════════════
# 🧬 SPEAKER FINGERPRINT  (spectral envelope + pitch) — drift ধরার জন্য
# ═════════════════════════════════════════════════════════════════════════════
def spectral_fingerprint(x, sr, n_bands=40):
    n, hop = 1024, 512
    if len(x) < n * 2:
        return None
    w = np.hanning(n).astype(np.float32)
    acc, cnt = None, 0
    for i in range(0, len(x) - n, hop):
        seg = x[i:i + n]
        if rms(seg) < 0.005:            # নীরব ফ্রেম বাদ
            continue
        spec = np.abs(np.fft.rfft(seg * w))
        acc = spec if acc is None else acc + spec
        cnt += 1
    if cnt < 4:
        return None
    S = acc / cnt
    edges = np.linspace(0, len(S), n_bands + 1).astype(int)
    bands = np.array([S[edges[i]:edges[i + 1]].mean() + 1e-9 for i in range(n_bands)])
    v = np.log(bands)
    v = v - v.mean()
    nrm = np.linalg.norm(v) + 1e-9
    return (v / nrm).astype(np.float32)

def median_f0(x, sr, fmin=60.0, fmax=320.0):
    n, hop = 2048, 1024
    vals = []
    for i in range(0, max(1, len(x) - n), hop):
        seg = x[i:i + n]
        if len(seg) < n or rms(seg) < 0.01:
            continue
        seg = seg - seg.mean()
        ac = np.correlate(seg, seg, mode="full")[n - 1:]
        lo, hi = int(sr / fmax), int(sr / fmin)
        if hi >= len(ac):
            continue
        k = int(np.argmax(ac[lo:hi])) + lo
        if ac[k] > 0.3 * (ac[0] + 1e-9):
            vals.append(sr / float(k))
    return float(np.median(vals)) if len(vals) >= 3 else 0.0

def voice_profile(x, sr):
    return {"spec": spectral_fingerprint(x, sr), "f0": median_f0(x, sr)}

def voice_similarity(a, b):
    """0..1 — 1 মানে হুবহু একই বক্তা।"""
    if a is None or b is None or a["spec"] is None or b["spec"] is None:
        return 1.0
    cos = float(np.dot(a["spec"], b["spec"]))
    cos = max(0.0, min(1.0, (cos + 1.0) / 2.0 * 1.0 if cos < 0 else cos))
    if a["f0"] > 0 and b["f0"] > 0:
        ratio = min(a["f0"], b["f0"]) / max(a["f0"], b["f0"])
        pitch = max(0.0, 1.0 - (1.0 - ratio) * 2.5)     # 20% pitch drift = 0.5
    else:
        pitch = cos
    return 0.72 * cos + 0.28 * pitch


# ═════════════════════════════════════════════════════════════════════════════
# 🌐 SERVER + ENDPOINT
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! আগে Step 2 রান করুন।")
        return False
    print("  📂 " + studio)
    log = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log, stderr=log, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print("  ✅ Server ready! (" + str((i + 1) * 2) + "s)")
            return True
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

def discover_clone_endpoint():
    """clone route আছে কিনা openapi.json থেকে দেখি।"""
    try:
        with urllib.request.urlopen(urllib.request.Request(OPENAPI_URL), timeout=10) as r:
            spec = json.loads(r.read().decode())
        for path, methods in sorted(spec.get("paths", {}).items()):
            if "clone" in path and "post" in methods:
                return path
    except Exception as e:
        print("  ⚠️ openapi.json পড়া যায়নি: " + str(e))
    return None


# ═════════════════════════════════════════════════════════════════════════════
# 📡 REQUESTS
# ═════════════════════════════════════════════════════════════════════════════
def post_multipart(url, fields, file_field, file_path, timeout=600):
    boundary = "----OmniVoiceLock" + str(int(time.time() * 1000))
    body = b""
    for k, v in fields.items():
        body += ("--" + boundary + "\r\n").encode()
        body += ('Content-Disposition: form-data; name="' + k + '"\r\n\r\n').encode()
        body += (str(v) + "\r\n").encode()
    fname = os.path.basename(file_path)
    ext = os.path.splitext(fname)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")
    body += ("--" + boundary + "\r\n").encode()
    body += ('Content-Disposition: form-data; name="' + file_field +
             '"; filename="' + fname + '"\r\n').encode()
    body += ("Content-Type: " + mime + "\r\n\r\n").encode()
    with open(file_path, "rb") as f:
        body += f.read()
    body += b"\r\n"
    body += ("--" + boundary + "--\r\n").encode()
    req = urllib.request.Request(
        url, data=body, method="POST",
        headers={"Content-Type": "multipart/form-data; boundary=" + boundary})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()

def clone_fields(text, minimal=False):
    """প্রতিটা call-এ হুবহু একই parameter set — এটাই voice lock-এর মূল।"""
    f = {"text": text, "input": text, "seed": SEED, "speed": VOICE["speed"]}
    if CUSTOM_VOICE_TEXT:
        f["ref_text"] = CUSTOM_VOICE_TEXT
    if minimal:
        return {k: v for k, v in f.items() if k in ("text", "input", "ref_text")}
    f["guidance_scale"] = VOICE["guidance_scale"]
    f["num_step"] = VOICE["num_step"]
    f["response_format"] = "wav"
    return f

def gen_clone(text, ref_wav, out_path, clone_ep):
    url = HOST + clone_ep if clone_ep else CLONE_URL
    for minimal in (False, True):     # অতিরিক্ত field-এ 422 হলে minimal-এ retry
        try:
            data = post_multipart(url, clone_fields(text, minimal), "ref_audio", ref_wav)
            with open(out_path, "wb") as f:
                f.write(data)
            return True
        except urllib.error.HTTPError as e:
            msg = ""
            try:
                msg = e.read().decode()[:180]
            except Exception:
                pass
            if e.code in (400, 422) and not minimal:
                continue
            print("    ❌ clone HTTP " + str(e.code) + ": " + msg)
            return False
        except Exception as e:
            print("    ❌ clone error: " + str(e))
            return False
    return False

def gen_json(text, out_path, voice=None):
    """Fallback — clone route না থাকলে। এখানেও প্যারামিটার হুবহু frozen।"""
    payload = {
        "model": "tts-1-hd",
        "voice": voice or PRESET_VOICE,
        "input": text,
        "response_format": "wav",
        "speed": VOICE["speed"],
        "num_step": VOICE["num_step"],
        "guidance_scale": VOICE["guidance_scale"],
        "class_temperature": VOICE["class_temperature"],
        "postprocess_output": VOICE["postprocess_output"],
        "seed": SEED,
    }
    req = urllib.request.Request(API_URL, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            data = r.read()
        with open(out_path, "wb") as f:
            f.write(data)
        return True
    except Exception as e:
        print("    ❌ json error: " + str(e))
        return False


# ═════════════════════════════════════════════════════════════════════════════
# ✂️  TEXT CHUNKING — লম্বা লাইন ভাঙে, কিন্তু voice setting একই থাকে
# ═════════════════════════════════════════════════════════════════════════════
def split_text(text, limit=MAX_CHARS_PER_CALL):
    text = " ".join(text.split())
    if len(text) <= limit:
        return [text]
    parts, buf = [], ""
    for piece in text.replace("? ", "?|").replace("! ", "!|").replace(". ", ".|").split("|"):
        if len(buf) + len(piece) + 1 <= limit:
            buf = (buf + " " + piece).strip()
        else:
            if buf:
                parts.append(buf)
            buf = piece.strip()
    if buf:
        parts.append(buf)
    return parts


# ═════════════════════════════════════════════════════════════════════════════
# 🚀 PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    v = 1
    while os.path.exists(os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")):
        v += 1
    return v, os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")

def auto_discover_voice(given):
    if given and os.path.exists(given):
        return given
    print("  ⚠️ দেওয়া পাথে ফাইল নেই: " + str(given))
    found = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                found.append(os.path.join(root, f))
    if not found:
        print("  ❌ /kaggle/input/ এ কোনো অডিও নেই → anchor TTS দিয়ে বানানো হবে")
        return None
    found.sort()
    print("  📋 পাওয়া গেছে " + str(len(found)) + " টি অডিও, ব্যবহার: " + found[0])
    return found[0]


print("=" * 74)
print("🔧 SERVER CHECK")
print("=" * 74)
if server_alive():
    print("  ✅ Server running")
elif not start_server():
    raise RuntimeError("Server failed. Step 2 & 3 আগে চালান।")

clone_ep = discover_clone_endpoint()
print("  🔗 Clone endpoint: " + str(clone_ep or "(default " + CLONE_URL + ")"))

RUN_VERSION, FINAL_OUTPUT_PATH = next_version(BASE_OUTPUT_DIR, FILENAME)
seg_dir  = os.path.join(BASE_OUTPUT_DIR, "segments_v" + str(RUN_VERSION))
work_dir = os.path.join(BASE_OUTPUT_DIR, "work")
os.makedirs(seg_dir, exist_ok=True)
os.makedirs(work_dir, exist_ok=True)

# ── STEP 1: ANCHOR (একটাই গলা, পুরো ট্র্যাকের ভিত্তি) ────────────────────────
print()
print("=" * 74)
print("🎙️  STEP 1 — ANCHOR VOICE তৈরি")
print("=" * 74)

anchor_wav = os.path.join(work_dir, "anchor.wav")
src_voice  = auto_discover_voice(CUSTOM_VOICE_PATH)
MODE = None      # "clone" বা "preset" — একবার ঠিক হলে সারা রান একই থাকবে

if src_voice:
    tmp = os.path.join(work_dir, "_ref_raw.wav")
    decode_to_wav(src_voice, tmp, TARGET_SR)
    x, sr = wav_read(tmp)
    x = best_window(trim_silence(x, sr), sr, 9.0)
    x = apply_fades(normalize_rms(x), sr)
    wav_write(anchor_wav, x, sr)
    print("  ✅ Custom reference: " + src_voice)
    print("     → anchor.wav  " + str(round(len(x) / sr, 1)) + "s @ " + str(sr) + " Hz")
    MODE = "clone"
else:
    print("  🔁 custom voice নেই → preset '" + PRESET_VOICE + "' দিয়ে anchor বানাচ্ছি...")
    raw = os.path.join(work_dir, "_anchor_raw.wav")
    if gen_json(ANCHOR_TEXT, raw):
        x, sr = wav_read(raw)
        x = resample(x, sr, TARGET_SR)
        x = apply_fades(normalize_rms(best_window(trim_silence(x, TARGET_SR), TARGET_SR, 9.0)), TARGET_SR)
        wav_write(anchor_wav, x, TARGET_SR)
        print("  ✅ anchor.wav তৈরি হয়েছে preset voice থেকে")
        MODE = "clone"
    else:
        print("  ⚠️ anchor বানানো যায়নি → পুরো রান preset+seed mode এ চলবে")
        MODE = "preset"

# ── STEP 2: PREFLIGHT PROBE — mode একবারেই লক ─────────────────────────────────
print()
print("=" * 74)
print("🧪 STEP 2 — PREFLIGHT (mode লক করা হচ্ছে, মাঝপথে আর বদলাবে না)")
print("=" * 74)

if MODE == "clone":
    probe = os.path.join(work_dir, "_probe.wav")
    ok = gen_clone("Testing one single voice.", anchor_wav, probe, clone_ep)
    if ok and os.path.getsize(probe) > 2000:
        print("  ✅ CLONE mode কাজ করছে → সব segment reference-locked হবে")
    else:
        MODE = "preset"
        print("  ⚠️ clone endpoint অচল → পুরো রান preset+seed mode এ (তবুও একই গলা,")
        print("     কারণ সব segment একই voice + একই frozen parameters ব্যবহার করবে)")
else:
    print("  ▶ preset+seed mode")

# anchor profile — drift মাপার রেফারেন্স
anchor_profile = None
if VERIFY_VOICE and os.path.exists(anchor_wav):
    ax, asr = wav_read(anchor_wav)
    anchor_profile = voice_profile(ax, asr)

print()
print("=" * 74)
print("🎭  V9.0 — ONE VOICE, START TO FINISH")
print("=" * 74)
print("  🔊 Mode        : " + MODE.upper())
print("  🎙️ Reference   : " + (anchor_wav if MODE == "clone" else PRESET_VOICE))
print("  🔒 Frozen      : speed=" + str(VOICE["speed"]) +
      "  gs=" + str(VOICE["guidance_scale"]) +
      "  steps=" + str(VOICE["num_step"]) + "  seed=" + str(SEED))
print("  📈 Verify      : " + ("on (threshold " + str(SIMILARITY_THRESHOLD) + ")" if VERIFY_VOICE else "off"))
print("  🎬 Segments    : " + str(len([s for s in SEGMENTS if "text" in s])))
print("  📁 Output      : " + os.path.basename(FINAL_OUTPUT_PATH))
print("=" * 74)
print()


def synth_once(text, out_path):
    ok = (gen_clone(text, anchor_wav, out_path, clone_ep) if MODE == "clone"
          else gen_json(text, out_path))
    if not ok or not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
        return None
    x, sr = wav_read(out_path)
    x = resample(x, sr, TARGET_SR)
    x = trim_silence(x, TARGET_SR, thresh_db=-45.0, pad_ms=40)
    x = normalize_rms(x)                # ← loudness jump বন্ধ
    x = apply_fades(x, TARGET_SR)       # ← click/pop বন্ধ
    return x


def synth_verified(text, tag, idx):
    """generate → fingerprint মিলিয়ে দেখা → না মিললে retry। সেরা attempt রাখে।"""
    best_x, best_score = None, -1.0
    for attempt in range(1, MAX_RETRIES + 1):
        raw = os.path.join(work_dir, "_tmp_" + tag + "_" + str(idx) + "_" + str(attempt) + ".wav")
        x = synth_once(text, raw)
        if x is None:
            print("    ↻ attempt " + str(attempt) + " ব্যর্থ, আবার চেষ্টা...")
            time.sleep(1.5)
            continue
        if not VERIFY_VOICE or anchor_profile is None:
            return x, 1.0
        score = voice_similarity(anchor_profile, voice_profile(x, TARGET_SR))
        if score > best_score:
            best_x, best_score = x, score
        if score >= SIMILARITY_THRESHOLD:
            return x, score
        print("    ⚠️ voice drift ধরা পড়েছে (score " + str(round(score, 3)) +
              " < " + str(SIMILARITY_THRESHOLD) + ") → re-generate " +
              str(attempt) + "/" + str(MAX_RETRIES))
    return best_x, best_score


timeline   = []      # (numpy audio) টুকরোগুলো ক্রমানুসারে
report     = []
failed     = []
t_start    = time.time()

for seg in SEGMENTS:
    if "pause_ms" in seg:
        n = int(TARGET_SR * seg["pause_ms"] / 1000.0)
        timeline.append(np.zeros(n, dtype=np.float32))   # ← সঠিক sample rate!
        continue

    tag = seg["tag"]
    chunks = split_text(seg["text"])
    t0 = time.time()
    pieces, scores = [], []

    for i, chunk in enumerate(chunks):
        x, score = synth_verified(chunk, tag, i)
        if x is None:
            failed.append(tag + "#" + str(i))
            continue
        pieces.append(x)
        scores.append(score)
        if i < len(chunks) - 1:
            pieces.append(np.zeros(int(TARGET_SR * 0.16), dtype=np.float32))

    if not pieces:
        print("  ❌ " + tag + " — কিছুই তৈরি হয়নি")
        continue

    seg_audio = np.concatenate(pieces)
    seg_path  = os.path.join(seg_dir, tag + ".wav")
    wav_write(seg_path, seg_audio, TARGET_SR)
    timeline.append(seg_audio)

    s = min(scores) if scores else 1.0
    report.append((tag, s, len(seg_audio) / TARGET_SR))
    mark = "✅" if s >= SIMILARITY_THRESHOLD else "⚠️"
    print("  " + mark + " " + tag.ljust(20) +
          " │ " + str(round(len(seg_audio) / TARGET_SR, 1)).rjust(5) + "s" +
          " │ match " + str(round(s, 3)) +
          " │ " + str(round(time.time() - t0, 1)) + "s" +
          " │ " + ("🔗clone" if MODE == "clone" else "🔒seed"))

# ── STEP 3: MASTER + CONCAT (সব একই SR, একই লাউডনেস) ─────────────────────────
print()
print("─" * 74)
print("🔗 Concatenating (single sample-rate, single loudness)...")

final = np.concatenate(timeline) if timeline else np.zeros(1, dtype=np.float32)
final = normalize_rms(final, TARGET_RMS_DBFS + 2.0)   # সামান্য গরম master
peak = float(np.max(np.abs(final)) + 1e-9)
if peak > PEAK_CEILING:
    final = final * (PEAK_CEILING / peak)
wav_write(FINAL_OUTPUT_PATH, final, TARGET_SR)

dur = len(final) / TARGET_SR
size_kb = os.path.getsize(FINAL_OUTPUT_PATH) // 1024
print("  🎬 FINAL: " + os.path.basename(FINAL_OUTPUT_PATH) +
      "  (" + str(size_kb) + " KB, " + str(round(dur, 1)) + "s @ " + str(TARGET_SR) + " Hz)")

# ── REPORT ───────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("📋 VOICE CONSISTENCY REPORT")
print("=" * 74)
if report:
    worst = min(r[1] for r in report)
    avg   = sum(r[1] for r in report) / len(report)
    for tag, s, d in report:
        bar = "█" * int(max(0.0, min(1.0, s)) * 24)
        print("  " + tag.ljust(20) + " " + str(round(s, 3)) + "  " + bar)
    print("  " + "-" * 60)
    print("  average match : " + str(round(avg, 3)))
    print("  worst match   : " + str(round(worst, 3)))
    if worst >= SIMILARITY_THRESHOLD:
        print("  ✅ পুরো ট্র্যাকে একজনই কথা বলছে — শুরু থেকে শেষ পর্যন্ত।")
    else:
        print("  ⚠️ কিছু segment এখনো drift করছে। করণীয়:")
        print("     • CUSTOM_VOICE_PATH এ ৮–১২s পরিষ্কার single-speaker ক্লিপ দিন")
        print("     • CUSTOM_VOICE_TEXT এ ক্লিপের হুবহু transcript লিখুন")
        print("     • MAX_CHARS_PER_CALL কমিয়ে 200 করুন")
        print("     • guidance_scale 2.0 → 2.4 করুন (বেশি reference adherence)")
if failed:
    print("  ❌ failed: " + ", ".join(failed))
print("  ⏱️ total: " + str(round(time.time() - t_start, 1)) + "s")
print("=" * 74)

# ── DOWNLOAD ─────────────────────────────────────────────────────────────────
if os.path.exists(FINAL_OUTPUT_PATH):
    with open(FINAL_OUTPUT_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(FINAL_OUTPUT_PATH)
    mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    display(HTML(
        '<audio controls src="data:audio/wav;base64,' + b64 + '" style="width:100%;margin:8px 0"></audio>'
        '<a download="' + dl + '" href="data:audio/wav;base64,' + b64 + '">'
        '<button style="padding:14px 28px;background:linear-gradient(135deg,#667eea,#764ba2);'
        'color:white;border:none;border-radius:8px;cursor:pointer;font-weight:bold;font-size:16px;">'
        '⬇️ Download ' + dl + ' (' + str(round(mb, 1)) + ' MB)</button></a>'))
else:
    print("❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V9.5 — GENDER-LOCKED: একটাই গলা, একটাই gender, শুরু থেকে শেষ পর্যন্ত
# =============================================================================
# সমস্যা: "একটু পর পর ভিন্ন মানুষ, এমনকি Male → Female flip!"
#
# কেন Male → Female হচ্ছে? ৫টা আসল কারণ (V9.0-র পরেও থেকে যাওয়া)
# ─────────────────────────────────────────────────────────────────────────────
# 1) Reference clip-এ একাধিক মানুষ আছে
#    → clone endpoint segment-ভেদে ভিন্ন speaker-এর অংশ emphasize করছে
#
# 2) Reference clip-এর শুরুতে নীরবতা/মিউজিক/অন্য গলা
#    → কিছু TTS reference-এর "প্রথম ৩ সেকেন্ড" থেকে voice identity নেয়
#    → ভিন্ন segment, ভিন্ন alignment, ভিন্ন গলা
#
# 3) Reference clip-ই ভুল gender-এর
#    → clone হুবহু সেই gender-এই copy করছে
#    → আপনি যেটা চান সেটা নাও হতে পারে
#
# 4) Fallback JSON call-এ "preset voice" বদলে যাচ্ছে
#    → V8.6-এ কিছু segment "onyx" (male), কিছু অন্য preset = gender flip
#
# 5) Reference clip খুব লম্বা/অস্থির
#    → ৬০ সেকেন্ডের ক্লিপে হাসি, কাশি, অন্য মানুষ — segment-ভেদে ভিন্ন
#      conditioning → ভিন্ন gender পর্যন্ত
#
# V9.5 সমাধান:
#   • anchor clip-এর auto gender + single-speaker verification
#   • GENDER_LOCK config: "male" / "female" / "auto"
#   • per-segment gender check — mismatch হলে auto fallback preset-এ
#   • full-track gender report
# =============================================================================

import os, sys, time, json, math, wave, base64, shutil, subprocess
import urllib.request, urllib.error
import numpy as np
from IPython.display import HTML, display

# ═════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION — এখানেই সব ঠিক করুন
# ═════════════════════════════════════════════════════════════════════════════
HOST            = "http://localhost:3900"
API_URL         = HOST + "/v1/audio/speech"
CLONE_URL       = HOST + "/v1/audio/speech/clone"
HEALTH_URL      = HOST + "/health"
OPENAPI_URL     = HOST + "/openapi.json"

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v9_5_gender_locked"
FILENAME        = "v9_5_gender_locked"

# ── 🔑 সবচেয়ে গুরুত্বপূর্ণ: আপনার voice sample ─────────────────────────────
# নিয়ম:
#   ✅ ৮–১২ সেকেন্ডের একটানা কথা, একজনেরই গলা
#   ✅ একই emotion / একই pace
#   ✅ পরিষ্কার, কোনো ব্যাকগ্রাউন্ড মিউজিক/নয়েজ নেই
#   ❌ হাসি, কাশি, অন্য মানুষের কণ্ঠ, মিউজিক — কিছুই না
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
# জানা থাকলে হুবহু transcript দিন (Whisper-এর ভুল এড়াতে)
CUSTOM_VOICE_TEXT = ""

# ── 🔒 GENDER LOCK — এটা ঠিক করলে Male→Female flip বন্ধ হবে ───────────────
# "male"   → anchor female হলেও preset "onyx" দিয়ে force male (fallback)
# "female" → anchor male হলেও preset "nova" দিয়ে force female
# "auto"   → anchor যা সেটাই — কিন্তু mismatch হলে warning দেবে
GENDER_LOCK = "auto"     # 👈 এটা "male" বা "female" করে দেখুন সমস্যা কাটে কিনা

# gender অনুযায়ী fallback preset (clone ব্যর্থ/ভুল gender হলে ব্যবহার হবে)
PRESET_MALE   = "onyx"   # deep male
PRESET_FEMALE = "nova"   # warm female
PRESET_NEUTRAL= "onyx"

SEED          = 1234

# ── 🔒 FROZEN VOICE PARAMETERS — প্রতিটা segment-এ হুবহু একই ────────────────
VOICE = {
    "speed":              0.88,
    "guidance_scale":     2.0,
    "num_step":           32,
    "class_temperature":  0.3,
    "postprocess_output": True,
}

# ── Audio pipeline ───────────────────────────────────────────────────────────
TARGET_SR            = 24000
TARGET_RMS_DBFS      = -20.0
PEAK_CEILING         = 0.97
FADE_MS              = 12

# ── Gender & voice checks ────────────────────────────────────────────────────
# F0 thresholds (Hz) — Praat standard
MALE_F0_MAX          = 165.0
FEMALE_F0_MIN        = 165.0
# per-segment F0 deviation from anchor — এর বেশি হলে gender flip
MAX_F0_DEVIATION_HZ  = 25.0
# reference clip-এর F0 std — এর বেশি হলে "multiple speakers" সন্দেহ
MAX_ANCHOR_F0_STD    = 22.0

VERIFY_VOICE         = True
SIMILARITY_THRESHOLD = 0.86
MAX_RETRIES          = 3
MAX_CHARS_PER_CALL   = 280
ANCHOR_DURATION_S    = 8.0    # reference clip-এর আদর্শ দৈর্ঘ্য


# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT
# ═════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text":
        "What if the greatest invention in human history... "
        "wasn't a machine that traveled through space, "
        "but one that crossed possibilities?"},
    {"pause_ms": 450},
    {"tag": "02_doubt", "text":
        "Another universe? Seriously? "
        "Scientists kept working. Everyone else kept doubting."},
    {"pause_ms": 350},
    {"tag": "03_wonder", "text":
        "And suddenly, the impossible became real. "
        "We saw dinosaurs, still walking, beneath blood-red skies. "
        "We found another Earth, where humanity was born on Mars."},
    {"pause_ms": 700},
    {"tag": "04_shift", "text":
        "But me... no. None of those worlds mattered. Not one. "
        "I was searching... for someone."},
    {"pause_ms": 800},
    {"tag": "05_grief", "text":
        "Three years ago... cancer... stole my mother. "
        "No warning. No mercy. No second chance."},
    {"pause_ms": 1000},
    {"tag": "06_hospital", "text":
        "I watched the hospital monitor... become... silent. "
        "I held her hand, hoping, just hoping, "
        "she would squeeze mine... one... last... time."},
    {"pause_ms": 1400},
    {"tag": "07_she_never_did", "text": "She never did."},
    {"pause_ms": 1000},
    {"tag": "08_pain", "text":
        "I know she will never answer. I know that. "
        "But I still call. "
        "Because sometimes... hope hurts more than reality."},
    {"pause_ms": 800},
    {"tag": "09_plea", "text":
        "Listen. Take me to the universe... where my mother... never died."},
    {"pause_ms": 1200},
    {"tag": "10_warmth", "text":
        "There she was. Alive. Smiling. Making breakfast. "
        "Humming the exact same song she used to sing... "
        "every Sunday morning."},
    {"pause_ms": 1000},
    {"tag": "11_ending", "text":
        "She did not know I was not her son. "
        "I was a broken man... borrowing someone else's miracle."},
]

ANCHOR_TEXT = ("This is my natural speaking voice, calm and steady, "
               "and I will keep this exact same voice, same gender, "
               "from the first word to the very last one.")


# ═════════════════════════════════════════════════════════════════════════════
# 🧰 AUDIO UTILITIES
# ═════════════════════════════════════════════════════════════════════════════
def have_ffmpeg():
    return shutil.which("ffmpeg") is not None

def wav_read(path):
    with wave.open(path, "rb") as wf:
        nch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
        raw = wf.readframes(n)
    if sw == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sw == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError("Unsupported sample width: " + str(sw))
    if nch > 1:
        x = x.reshape(-1, nch).mean(axis=1)
    return x.astype(np.float32), sr

def wav_write(path, x, sr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    x = np.clip(x, -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(pcm)

def resample(x, sr_in, sr_out):
    if sr_in == sr_out or len(x) == 0:
        return x
    n_out = int(round(len(x) * float(sr_out) / float(sr_in)))
    t_in  = np.arange(len(x), dtype=np.float64)
    t_out = np.linspace(0.0, len(x) - 1.0, n_out)
    return np.interp(t_out, t_in, x).astype(np.float32)

def decode_to_wav(src, dst, sr=TARGET_SR):
    if have_ffmpeg():
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
               "-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", dst]
        if subprocess.run(cmd).returncode == 0 and os.path.exists(dst):
            return dst
    if src.lower().endswith(".wav"):
        x, s = wav_read(src)
        wav_write(dst, resample(x, s, sr), sr)
        return dst
    raise RuntimeError("ffmpeg নেই এবং reference WAV নয়: " + src)

def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))

def normalize_rms(x, target_dbfs=TARGET_RMS_DBFS, ceiling=PEAK_CEILING):
    cur = rms(x)
    if cur < 1e-6:
        return x
    gain = (10.0 ** (target_dbfs / 20.0)) / cur
    y = x * gain
    peak = float(np.max(np.abs(y)) + 1e-12)
    if peak > ceiling:
        y = y * (ceiling / peak)
    return y.astype(np.float32)

def apply_fades(x, sr, ms=FADE_MS):
    n = min(int(sr * ms / 1000.0), len(x) // 2)
    if n <= 0:
        return x
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x.copy()
    x[:n]  *= ramp
    x[-n:] *= ramp[::-1]
    return x

def trim_silence(x, sr, thresh_db=-42.0, pad_ms=60):
    win = max(1, int(sr * 0.02))
    if len(x) < win * 3:
        return x
    frames = x[:len(x) - len(x) % win].reshape(-1, win)
    energy = 20.0 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)
    loud = np.where(energy > thresh_db)[0]
    if len(loud) == 0:
        return x
    pad = int(sr * pad_ms / 1000.0)
    a = max(0, loud[0] * win - pad)
    b = min(len(x), (loud[-1] + 1) * win + pad)
    return x[a:b]

def sliding_energy(x, win):
    if len(x) < win:
        return np.array([rms(x)])
    frames = x[:len(x) - len(x) % win].reshape(-1, win)
    return np.sqrt(np.mean(frames ** 2, axis=1))


# ═════════════════════════════════════════════════════════════════════════════
# 🧬 PITCH & GENDER
# ═════════════════════════════════════════════════════════════════════════════
def frame_f0s(x, sr, fmin=55.0, fmax=400.0, n=2048, hop=1024, voiced_th=0.30):
    out = []
    for i in range(0, max(1, len(x) - n), hop):
        seg = x[i:i + n]
        if len(seg) < n or rms(seg) < 0.012:
            continue
        seg = seg - seg.mean()
        ac = np.correlate(seg, seg, mode="full")[n - 1:]
        lo, hi = int(sr / fmax), int(sr / fmin)
        if hi >= len(ac) or hi <= lo:
            continue
        k = int(np.argmax(ac[lo:hi])) + lo
        if ac[k] > voiced_th * (ac[0] + 1e-9):
            out.append(sr / float(k))
    return np.array(out, dtype=np.float32)

def median_f0(x, sr):
    f = frame_f0s(x, sr)
    return float(np.median(f)) if len(f) > 0 else 0.0

def f0_stats(x, sr):
    f = frame_f0s(x, sr)
    if len(f) == 0:
        return {"mean": 0.0, "median": 0.0, "std": 0.0, "n": 0}
    return {
        "mean": float(np.mean(f)),
        "median": float(np.median(f)),
        "std": float(np.std(f)),
        "n": len(f),
    }

def detect_gender(x, sr):
    s = f0_stats(x, sr)
    if s["n"] < 5:
        return "uncertain", s
    med = s["median"]
    if med >= FEMALE_F0_MIN + 15:
        return "female", s
    if med <= MALE_F0_MAX - 15:
        return "male", s
    return "uncertain", s

def check_single_speaker(x, sr):
    """F0 std খুব বেশি হলে → multiple speakers সন্দেহ"""
    s = f0_stats(x, sr)
    ok = s["std"] < MAX_ANCHOR_F0_STD
    return ok, s


# ═════════════════════════════════════════════════════════════════════════════
# ✂️  SMART REFERENCE WINDOW
# ═════════════════════════════════════════════════════════════════════════════
def best_speech_window(x, sr, seconds=ANCHOR_DURATION_S):
    """সবচেয়ে energetic + সবচেয়ে stable-pitch টানা অংশ বেছে নেয়"""
    want = int(sr * seconds)
    if len(x) <= want:
        return x
    hop = int(sr * 0.25)
    best_i, best_score = 0, -1.0
    for i in range(0, len(x) - want, hop):
        w = x[i:i + want]
        e = rms(w)
        s = f0_stats(w, sr)
        if s["n"] < 4:
            continue
        # energy বেশি + pitch stable = ভালো reference
        score = e * (1.0 / (1.0 + s["std"] / 25.0))
        if score > best_score:
            best_score, best_i = score, i
    return x[best_i:best_i + want]


# ═════════════════════════════════════════════════════════════════════════════
# 🧬 VOICE SIMILARITY (from V9.0)
# ═════════════════════════════════════════════════════════════════════════════
def spectral_fingerprint(x, sr, n_bands=40):
    n, hop = 1024, 512
    if len(x) < n * 2:
        return None
    w = np.hanning(n).astype(np.float32)
    acc, cnt = None, 0
    for i in range(0, len(x) - n, hop):
        seg = x[i:i + n]
        if rms(seg) < 0.005:
            continue
        spec = np.abs(np.fft.rfft(seg * w))
        acc = spec if acc is None else acc + spec
        cnt += 1
    if cnt < 4:
        return None
    S = acc / cnt
    edges = np.linspace(0, len(S), n_bands + 1).astype(int)
    bands = np.array([S[edges[i]:edges[i + 1]].mean() + 1e-9 for i in range(n_bands)])
    v = np.log(bands); v = v - v.mean()
    return (v / (np.linalg.norm(v) + 1e-9)).astype(np.float32)

def voice_profile(x, sr):
    return {"spec": spectral_fingerprint(x, sr), "f0": median_f0(x, sr)}

def voice_similarity(a, b):
    if a is None or b is None or a["spec"] is None or b["spec"] is None:
        return 1.0
    cos = float(np.dot(a["spec"], b["spec"]))
    cos = max(0.0, cos)
    if a["f0"] > 0 and b["f0"] > 0:
        ratio = min(a["f0"], b["f0"]) / max(a["f0"], b["f0"])
        pitch = max(0.0, 1.0 - (1.0 - ratio) * 2.5)
    else:
        pitch = cos
    return 0.72 * cos + 0.28 * pitch


# ═════════════════════════════════════════════════════════════════════════════
# 🌐 SERVER
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি!"); return False
    log = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log, stderr=log, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print("  ✅ Server ready! (" + str((i+1)*2) + "s)"); return True
    return False

def discover_clone_endpoint():
    try:
        with urllib.request.urlopen(urllib.request.Request(OPENAPI_URL), timeout=10) as r:
            spec = json.loads(r.read().decode())
        for path, methods in sorted(spec.get("paths", {}).items()):
            if "clone" in path and "post" in methods:
                return path
    except Exception:
        pass
    return None


# ═════════════════════════════════════════════════════════════════════════════
# 📡 REQUESTS
# ═════════════════════════════════════════════════════════════════════════════
def post_multipart(url, fields, file_field, file_path, timeout=600):
    boundary = "----OmniVoiceLock" + str(int(time.time() * 1000))
    body = b""
    for k, v in fields.items():
        body += ("--" + boundary + "\r\n").encode()
        body += ('Content-Disposition: form-data; name="' + k + '"\r\n\r\n').encode()
        body += (str(v) + "\r\n").encode()
    fname = os.path.basename(file_path)
    ext = os.path.splitext(fname)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")
    body += ("--" + boundary + "\r\n").encode()
    body += ('Content-Disposition: form-data; name="' + file_field +
             '"; filename="' + fname + '"\r\n').encode()
    body += ("Content-Type: " + mime + "\r\n\r\n").encode()
    with open(file_path, "rb") as f:
        body += f.read()
    body += b"\r\n--" + boundary.encode() + b"--\r\n"
    req = urllib.request.Request(
        url, data=body, method="POST",
        headers={"Content-Type": "multipart/form-data; boundary=" + boundary})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def clone_fields(text, minimal=False):
    f = {"text": text, "input": text, "seed": SEED, "speed": VOICE["speed"]}
    if CUSTOM_VOICE_TEXT:
        f["ref_text"] = CUSTOM_VOICE_TEXT
    if minimal:
        return {k: v for k, v in f.items() if k in ("text", "input", "ref_text")}
    f["guidance_scale"] = VOICE["guidance_scale"]
    f["num_step"] = VOICE["num_step"]
    f["response_format"] = "wav"
    return f

def gen_clone(text, ref_wav, out_path, clone_ep, voice_preset=None):
    """voice_preset দিলে clone না করে JSON preset ব্যবহার হবে (gender mismatch fix)"""
    if voice_preset:
        return gen_json(text, out_path, voice_preset)
    url = HOST + clone_ep if clone_ep else CLONE_URL
    for minimal in (False, True):
        try:
            data = post_multipart(url, clone_fields(text, minimal), "ref_audio", ref_wav)
            with open(out_path, "wb") as f:
                f.write(data)
            return True
        except urllib.error.HTTPError as e:
            msg = ""
            try: msg = e.read().decode()[:180]
            except Exception: pass
            if e.code in (400, 422) and not minimal:
                continue
            print("    ❌ clone HTTP " + str(e.code) + ": " + msg); return False
        except Exception as e:
            print("    ❌ clone error: " + str(e)); return False
    return False

def gen_json(text, out_path, voice):
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": VOICE["speed"],
        "num_step": VOICE["num_step"],
        "guidance_scale": VOICE["guidance_scale"],
        "class_temperature": VOICE["class_temperature"],
        "postprocess_output": VOICE["postprocess_output"],
        "seed": SEED,
    }
    req = urllib.request.Request(API_URL, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            data = r.read()
        with open(out_path, "wb") as f:
            f.write(data)
        return True
    except Exception as e:
        print("    ❌ json error: " + str(e)); return False


def split_text(text, limit=MAX_CHARS_PER_CALL):
    text = " ".join(text.split())
    if len(text) <= limit:
        return [text]
    parts, buf = [], ""
    for piece in text.replace("? ", "?|").replace("! ", "!|").replace(". ", ".|").split("|"):
        if len(buf) + len(piece) + 1 <= limit:
            buf = (buf + " " + piece).strip()
        else:
            if buf: parts.append(buf)
            buf = piece.strip()
    if buf: parts.append(buf)
    return parts


# ═════════════════════════════════════════════════════════════════════════════
# 🚀 PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    v = 1
    while os.path.exists(os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")):
        v += 1
    return v, os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")

def auto_discover_voice(given):
    if given and os.path.exists(given):
        return given
    print("  ⚠️ দেওয়া পাথে ফাইল নেই: " + str(given))
    found = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                found.append(os.path.join(root, f))
    if not found:
        print("  ❌ /kaggle/input/ এ কোনো অডিও নেই")
        return None
    found.sort()
    print("  📋 পাওয়া গেছে " + str(len(found)) + " টি অডিও, ব্যবহার: " + found[0])
    return found[0]


print("=" * 74)
print("🔧 SERVER CHECK")
print("=" * 74)
if server_alive():
    print("  ✅ Server running")
elif not start_server():
    raise RuntimeError("Server failed.")

clone_ep = discover_clone_endpoint()
print("  🔗 Clone endpoint: " + str(clone_ep or CLONE_URL))

RUN_VERSION, FINAL_OUTPUT_PATH = next_version(BASE_OUTPUT_DIR, FILENAME)
seg_dir  = os.path.join(BASE_OUTPUT_DIR, "segments_v" + str(RUN_VERSION))
work_dir = os.path.join(BASE_OUTPUT_DIR, "work")
os.makedirs(seg_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: ANCHOR + GENDER/ONE-SPEAKER CHECK
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("🎙️  STEP 1 — ANCHOR + GENDER/SPEAKER VERIFICATION")
print("=" * 74)

anchor_wav = os.path.join(work_dir, "anchor.wav")
src_voice  = auto_discover_voice(CUSTOM_VOICE_PATH)

anchor_x = None
anchor_sr = TARGET_SR
anchor_gender = "uncertain"
anchor_f0_stats = None
multiple_speakers = False

if src_voice:
    tmp = os.path.join(work_dir, "_ref_raw.wav")
    decode_to_wav(src_voice, tmp, TARGET_SR)
    raw_x, raw_sr = wav_read(tmp)
    raw_dur = len(raw_x) / raw_sr
    print("  📥 reference loaded: " + str(round(raw_dur, 1)) + "s")

    # pre-window stats (পুরো clip কেমন?)
    full_stats = f0_stats(raw_x, raw_sr)
    full_gender, _ = detect_gender(raw_x, raw_sr)
    single_ok, _ = check_single_speaker(raw_x, raw_sr)

    if not single_ok:
        multiple_speakers = True
        print("  ⚠️ ⚠️ ⚠️  WARNING: reference clip-এ একাধিক মানুষ বলে মনে হচ্ছে!")
        print("     F0 std = " + str(round(full_stats["std"], 1)) +
              " Hz (threshold " + str(MAX_ANCHOR_F0_STD) + ")")
        print("     👉 এটাই Male→Female flip-এর প্রধান কারণ হতে পারে")
        print("     👉 করণীয়: ৮–১২ সেকেন্ডের একক speaker clip ব্যবহার করুন")

    # সবচেয়ে clean, stable window নাও
    win = best_speech_window(trim_silence(raw_x, raw_sr), raw_sr, ANCHOR_DURATION_S)
    anchor_x = apply_fades(normalize_rms(win), raw_sr)
    anchor_sr = raw_sr
    anchor_gender, anchor_f0_stats = detect_gender(anchor_x, anchor_sr)

    wav_write(anchor_wav, anchor_x, anchor_sr)
    print("  ✅ anchor.wav (" + str(round(len(anchor_x) / anchor_sr, 1)) +
          "s) → detected gender: " + anchor_gender.upper())
    print("     F0 median=" + str(round(anchor_f0_stats["median"], 1)) +
          " Hz, std=" + str(round(anchor_f0_stats["std"], 1)) + " Hz")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: GENDER LOCK — চূড়ান্ত mode ঠিক করা
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("🔒 STEP 2 — GENDER LOCK")
print("=" * 74)

MODE          = "clone"
FORCE_PRESET  = None   # None হলে clone, সেট থাকলে পুরো ট্র্যাক এই preset-এ

if GENDER_LOCK in ("male", "female"):
    desired = GENDER_LOCK
    if anchor_x is not None and anchor_gender != "uncertain" and anchor_gender != desired:
        print("  ⚠️ GENDER MISMATCH DETECTED!")
        print("     আপনার reference clip: " + anchor_gender.upper())
        print("     আপনার চাওয়া (GENDER_LOCK): " + desired.upper())
        print("     👉 clone ব্যবহার করলে পুরো ট্র্যাক " + anchor_gender +
              " হবে — যা আপনি চান না।")
        FORCE_PRESET = PRESET_MALE if desired == "male" else PRESET_FEMALE
        MODE = "preset"
        print("     ✅ তাই পুরো ট্র্যাক forced preset '" + FORCE_PRESET +
              "' (" + desired + ") দিয়ে তৈরি হবে।")
    else:
        FORCE_PRESET = PRESET_MALE if desired == "male" else PRESET_FEMALE
        print("  ✅ GENDER_LOCK=" + desired + " → forced preset: " + FORCE_PRESET)
        MODE = "preset"
elif anchor_x is None:
    FORCE_PRESET = PRESET_NEUTRAL
    MODE = "preset"
    print("  ⚠️ reference নেই → preset '" + FORCE_PRESET + "' ব্যবহার হবে")
else:
    print("  ▶ auto mode — anchor gender (" + anchor_gender + ") ব্যবহার হবে")

# preflight probe (clone mode-এ)
if MODE == "clone" and anchor_x is not None:
    probe = os.path.join(work_dir, "_probe.wav")
    ok = gen_clone("Testing voice lock.", anchor_wav, probe, clone_ep, None)
    if ok and os.path.getsize(probe) > 2000:
        px, psr = wav_read(probe)
        pg, pstats = detect_gender(resample(px, psr, TARGET_SR), TARGET_SR)
        print("  🧪 probe gender: " + pg +
              " (F0=" + str(round(pstats["median"], 1)) + " Hz)")
        if pg != "uncertain" and anchor_gender != "uncertain" and pg != anchor_gender:
            print("  ❌ clone gender FLIP করছে! preset fallback চালু হচ্ছে।")
            desired = GENDER_LOCK if GENDER_LOCK in ("male", "female") else anchor_gender
            FORCE_PRESET = PRESET_MALE if desired == "male" else PRESET_FEMALE
            MODE = "preset"
            print("     → forced preset: " + FORCE_PRESET)
    else:
        FORCE_PRESET = PRESET_NEUTRAL
        MODE = "preset"
        print("  ⚠️ clone অচল → preset '" + FORCE_PRESET + "'")

anchor_profile = None
if anchor_x is not None:
    anchor_profile = voice_profile(anchor_x, anchor_sr)

print()
print("=" * 74)
print("🎭  V9.5 — ONE VOICE, ONE GENDER")
print("=" * 74)
print("  🔊 Mode        : " + MODE.upper() +
      (" (forced preset '" + FORCE_PRESET + "')" if FORCE_PRESET else ""))
print("  🎙️ Anchor      : " + (anchor_wav if MODE == "clone" else "(none)"))
print("  🧬 Anchor gndr : " + anchor_gender)
print("  🔒 GENDER_LOCK : " + GENDER_LOCK)
print("  🧊 Frozen      : speed=" + str(VOICE["speed"]) +
      "  gs=" + str(VOICE["guidance_scale"]) +
      "  seed=" + str(SEED))
print("  🎬 Segments    : " + str(len([s for s in SEGMENTS if "text" in s])))
print("  📁 Output      : " + os.path.basename(FINAL_OUTPUT_PATH))
if multiple_speakers:
    print()
    print("  ⚠️ ⚠️  আপনার reference clip-এ একাধিক মানুষ সন্দেহ!")
    print("        Male→Female সমস্যা সম্ভবত এখান থেকেই আসছে।")
print("=" * 74)
print()


def synth_once(text, out_path):
    ok = gen_clone(text, anchor_wav, out_path, clone_ep, FORCE_PRESET) \
         if MODE == "clone" else gen_json(text, out_path, FORCE_PRESET)
    if not ok or not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
        return None
    x, sr = wav_read(out_path)
    x = resample(x, sr, TARGET_SR)
    x = trim_silence(x, TARGET_SR, thresh_db=-45.0, pad_ms=40)
    x = normalize_rms(x)
    x = apply_fades(x, TARGET_SR)
    return x


def synth_verified(text, tag, idx):
    best_x, best_score = None, -1.0
    for attempt in range(1, MAX_RETRIES + 1):
        raw = os.path.join(work_dir, "_tmp_" + tag + "_" + str(idx) + "_" + str(attempt) + ".wav")
        x = synth_once(text, raw)
        if x is None:
            time.sleep(1.5); continue
        if not VERIFY_VOICE or anchor_profile is None:
            return x, 1.0, median_f0(x, TARGET_SR)
        score = voice_similarity(anchor_profile, voice_profile(x, TARGET_SR))
        f0 = median_f0(x, TARGET_SR)
        # gender mismatch check
        if anchor_f0_stats and f0 > 0:
            deviation = abs(f0 - anchor_f0_stats["median"])
            if deviation > MAX_F0_DEVIATION_HZ:
                score = min(score, 0.80)   # penalty
        if score > best_score:
            best_x, best_score = x, score
        if score >= SIMILARITY_THRESHOLD:
            return x, score, median_f0(x, TARGET_SR)
        print("    ⚠️ drift (score " + str(round(score, 3)) +
              ") → retry " + str(attempt) + "/" + str(MAX_RETRIES))
    return best_x, best_score, (median_f0(best_x, TARGET_SR) if best_x is not None else 0.0)


timeline   = []
report     = []
failed     = []
t_start    = time.time()

for seg in SEGMENTS:
    if "pause_ms" in seg:
        n = int(TARGET_SR * seg["pause_ms"] / 1000.0)
        timeline.append(np.zeros(n, dtype=np.float32)); continue

    tag = seg["tag"]
    chunks = split_text(seg["text"])
    t0 = time.time()
    pieces, scores, f0s = [], [], []

    for i, chunk in enumerate(chunks):
        x, score, f0 = synth_verified(chunk, tag, i)
        if x is None:
            failed.append(tag + "#" + str(i)); continue
        pieces.append(x); scores.append(score); f0s.append(f0)
        if i < len(chunks) - 1:
            pieces.append(np.zeros(int(TARGET_SR * 0.16), dtype=np.float32))

    if not pieces:
        print("  ❌ " + tag); continue

    seg_audio = np.concatenate(pieces)
    seg_path  = os.path.join(seg_dir, tag + ".wav")
    wav_write(seg_path, seg_audio, TARGET_SR)
    timeline.append(seg_audio)

    s = min(scores) if scores else 1.0
    f0_seg = float(np.median([f for f in f0s if f > 0])) if any(f > 0 for f in f0s) else 0.0

    # gender flip detection
    gender_flag = ""
    if anchor_f0_stats and f0_seg > 0:
        dev = abs(f0_seg - anchor_f0_stats["median"])
        if dev > MAX_F0_DEVIATION_HZ:
            gender_flag = " ⚠️ GENDER DRIFT (" + str(round(dev, 0)) + " Hz)"

    report.append((tag, s, len(seg_audio) / TARGET_SR, f0_seg, gender_flag))
    mark = "✅" if s >= SIMILARITY_THRESHOLD and not gender_flag else "⚠️"
    print("  " + mark + " " + tag.ljust(20) +
          " │ " + str(round(len(seg_audio) / TARGET_SR, 1)).rjust(5) + "s" +
          " │ match " + str(round(s, 3)) +
          " │ F0 " + str(round(f0_seg, 0)).rjust(3) + "Hz" +
          " │ " + str(round(time.time() - t0, 1)) + "s" +
          (" │ " + MODE.upper()) + gender_flag)

# ─────────────────────────────────────────────────────────────────────────────
# MASTER
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 74)
print("🔗 Concatenating...")

final = np.concatenate(timeline) if timeline else np.zeros(1, dtype=np.float32)
final = normalize_rms(final, TARGET_RMS_DBFS + 2.0)
peak = float(np.max(np.abs(final)) + 1e-9)
if peak > PEAK_CEILING:
    final = final * (PEAK_CEILING / peak)
wav_write(FINAL_OUTPUT_PATH, final, TARGET_SR)

dur = len(final) / TARGET_SR
size_kb = os.path.getsize(FINAL_OUTPUT_PATH) // 1024
print("  🎬 FINAL: " + os.path.basename(FINAL_OUTPUT_PATH) +
      " (" + str(size_kb) + " KB, " + str(round(dur, 1)) + "s)")


# ─────────────────────────────────────────────────────────────────────────────
# REPORT
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("📋 VOICE + GENDER CONSISTENCY REPORT")
print("=" * 74)
if report:
    print("  " + "segment".ljust(22) + "match   F0(Hz)  notes")
    print("  " + "-" * 64)
    gender_flips = 0
    for tag, s, d, f0, flag in report:
        bar = "█" * int(max(0.0, min(1.0, s)) * 18)
        line = "  " + tag.ljust(22) + str(round(s, 3)).ljust(8) + str(round(f0, 0)).rjust(3).ljust(8) + bar
        if flag:
            line += "   " + flag
            gender_flips += 1
        print(line)
    print("  " + "-" * 64)
    avg   = sum(r[1] for r in report) / len(report)
    f0s   = [r[3] for r in report if r[3] > 0]
    f0rng = (max(f0s) - min(f0s)) if f0s else 0
    print("  average match    : " + str(round(avg, 3)))
    print("  F0 range (track) : " + str(round(f0rng, 0)) + " Hz " +
          ("✅ tight" if f0rng < 40 else "⚠️ wide"))
    print("  gender flips     : " + str(gender_flips))
    if gender_flips == 0 and avg >= SIMILARITY_THRESHOLD:
        print("  ✅ পুরো ট্র্যাকে একজনই, একই gender-এ, শুরু থেকে শেষ।")
    elif gender_flips > 0:
        print("  ⚠️ কিছু segment-এ gender drift। করণীয়:")
        print("     1) GENDER_LOCK = \"" + ("male" if (anchor_gender == "female") else "female") +
              "\" করে দেখুন (hard preset lock)")
        print("     2) reference clip বদলান — ৮–১২s একক speaker")
        print("     3) CUSTOM_VOICE_TEXT দিন (Whisper-এর ভুল এড়াতে)")
if failed:
    print("  ❌ failed: " + ", ".join(failed))
print("  ⏱️ total: " + str(round(time.time() - t_start, 1)) + "s")
print("=" * 74)


# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(FINAL_OUTPUT_PATH):
    with open(FINAL_OUTPUT_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(FINAL_OUTPUT_PATH)
    mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    display(HTML(
        '<audio controls src="data:audio/wav;base64,' + b64 + '" style="width:100%;margin:8px 0"></audio>'
        '<a download="' + dl + '" href="data:audio/wav;base64,' + b64 + '">'
        '<button style="padding:14px 28px;background:linear-gradient(135deg,#667eea,#764ba2);'
        'color:white;border:none;border-radius:8px;cursor:pointer;font-weight:bold;font-size:16px;">'
        '⬇️ Download ' + dl + ' (' + str(round(mb, 1)) + ' MB)</button></a>'))
else:
    print("❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V8.7 — TRUE VOICE LOCK: One Voice from Start to Finish
# -----------------------------------------------------------------------------
# 🔍 PROBLEM (তোমার সমস্যা):
#    "একটু পর পর ভিন্ন মানুষ কথা বলছিল" — segment-এ segment-ে voice বদলে
#    যাচ্ছিল, মনে হচ্ছিল অন্য মানুষ কথা বলছে।
#
# 🎯 ROOT CAUSE — MALE → FEMALE FLIP (আসল কারণ):
#    OmniVoice-এর model diffusion-based (F5-TTS style)। Zero-shot clone-এ
#    reference audio-র পাশাপাশি সেই audio-র TRANSCRIPT (ref_text) লাগে।
#    ref_text না দিলে model নিজে auto-transcribe করে — সেটা ভুল হলে speaker
#    identity align হয় না, আর model প্রতি segment-এ নতুন random speaker
#    sample করে → এক segment MALE তো পরেরটা FEMALE। এটাই তোমার সমস্যা।
#
#    সঙ্গে আগের bug তো ছিলই: clone fail করলে চুপচাপ "onyx" preset-এ
#    চলে যেত (voice mix)।
#
# ✅ V8.7 FIXES (এই version-এ যা ঠিক করা হয়েছে):
#    1. REF_TEXT 🔑   — প্রতিটা clone call-এ reference-এর transcript পাঠাই
#                       (male→female flip বন্ধ — সবচেয়ে গুরুত্বপূর্ণ fix)
#    2. ONE ANCHOR    — প্রথম speaker-এর voice = পুরো script-এর fixed reference
#    3. METHOD LOCK   — clone না json, একবার ঠিক করে নিই; মাঝে বদলাবে না
#    4. RETRY, NO MIX — transient fail-এ voice switch নয়, retry করবে
#    5. NO MIXING     — কখনো single segment-এ অন্য voice আসবে না
#    6. FIXED SEED    — সব call-এ একই seed
# =============================================================================

import os, time, json, urllib.request, urllib.error, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v8_7_voice_locked"
API_URL    = "http://localhost:3900/v1/audio/speech"
CLONE_URL  = "http://localhost:3900/v1/audio/speech/clone"
HEALTH_URL = "http://localhost:3900/health"

# ─── VOICE LOCK CONFIG ───────────────────────────────────────────────────────
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
CUSTOM_REF_TEXT   = ""   # 🔑 sample-এ ঠিক কী বলা আছে (খালি = auto, ঝুঁকিপূর্ণ!)
PRESET_VOICE  = "onyx"        # anchor বানাতে (custom voice না থাকলে)
SEED          = 42                       # FIXED for ALL segments
MAX_RETRIES   = 4                 # প্রতি segment-এ retry (voice switch এড়াতে)
FILENAME      = "v8_7_true_voice_lock"


# ════════════════════════════════════════════════════════════════════════════
# 🔍 AUTO-DISCOVER: voice ফাইল খুঁজে বের করা
# ════════════════════════════════════════════════════════════════════════════
def auto_discover_voice(given_path):
    """দেওয়া পাথে ফাইল না থাকলে /kaggle/input/ এ সব audio ফাইল খুঁজে দেখায়।"""
    if given_path and os.path.exists(given_path):
        return given_path
    if not given_path:
        return None

    print(f"  ⚠️ দেওয়া পাথে ফাইল নেই: {given_path}")
    print(f"  🔍 /kaggle/input/ এ audio ফাইল খুঁজছি...")

    audio_files = []
    search_root = "/kaggle/input"
    if os.path.exists(search_root):
        for root, dirs, files in os.walk(search_root):
            for f in files:
                if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))

    if not audio_files:
        print("  ❌ /kaggle/input/ এ কোনো audio ফাইল পাওয়া যায়নি")
        return None

    print(f"  📋 {len(audio_files)} টি audio ফাইল পাওয়া গেছে:")
    for i, af in enumerate(audio_files):
        print(f"     {i+1}. {af} ({os.path.getsize(af)//1024} KB)")

    chosen = audio_files[0]
    print(f"  ✅ ব্যবহার করা হচ্ছে: {chosen}")
    return chosen


# ════════════════════════════════════════════════════════════════════════════
# 📁 AUTO-VERSIONING
# ════════════════════════════════════════════════════════════════════════════
def get_next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    version = 1
    while True:
        candidate = os.path.join(base_dir, f"{base_name}_v{version}.wav")
        if not os.path.exists(candidate):
            return version, candidate
        version += 1

RUN_VERSION, FINAL_OUTPUT_PATH = get_next_version(BASE_OUTPUT_DIR, FILENAME)
print(f"📁 Auto-version: v{RUN_VERSION}  →  {os.path.basename(FINAL_OUTPUT_PATH)}\n")


# ════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! আগে Step 2 রান করো।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()


# ════════════════════════════════════════════════════════════════════════════
# 🔍 ENDPOINT DISCOVERY
# ════════════════════════════════════════════════════════════════════════════
clone_ep = None

def discover_clone_endpoint():
    """সার্ভারে /clone endpoint আছে কিনা চেক করি।"""
    try:
        req = urllib.request.Request("http://localhost:3900/openapi.json")
        with urllib.request.urlopen(req, timeout=10) as resp:
            spec = json.loads(resp.read().decode())
            paths = spec.get("paths", {})
            for path in paths:
                if "clone" in path and "post" in paths[path]:
                    print(f"  🔗 Clone endpoint পাওয়া গেছে: {path}")
                    return path
            print("  🔒 কোনো clone endpoint নেই — JSON + seed fallback ব্যবহার হবে")
            return None
    except Exception as e:
        print(f"  ⚠️ OpenAPI spec পড়া যায়নি: {e}")
        return None

print("=" * 70)
print("🔍 ENDPOINT DISCOVERY")
print("=" * 70)
clone_ep = discover_clone_endpoint()
print()


# ════════════════════════════════════════════════════════════════════════════
# 🎵 CORE FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ কোনো WAV ফাইল নেই!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_with_clone(text, ref_audio_path, output_path, speed=0.88,
                   guidance_scale=2.0, seed=42, ref_text=""):
    """🔗 /clone endpoint — multipart/form-data. Reference audio থেকে voice clone।
    🔑 ref_text = reference audio-র transcript। এটাই speaker identity (male/female)
    lock রাখে — বাদ দিলে model প্রতি call-এ নতুন speaker sample করে।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    boundary = "----OmniVoiceClone" + str(int(time.time() * 1000))
    body = b""

    body += f"--{boundary}\r\n".encode()
    body += b'Content-Disposition: form-data; name="text"\r\n\r\n'
    body += f"{text}\r\n".encode()

    body += f"--{boundary}\r\n".encode()
    body += b'Content-Disposition: form-data; name="speed"\r\n\r\n'
    body += f"{speed}\r\n".encode()

    # 🔑 ref_text — reference audio-র transcript (identity lock)
    if ref_text:
        body += f"--{boundary}\r\n".encode()
        body += b'Content-Disposition: form-data; name="ref_text"\r\n\r\n'
        body += f"{ref_text}\r\n".encode()

    filename = os.path.basename(ref_audio_path)
    ext = os.path.splitext(filename)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")

    body += f"--{boundary}\r\n".encode()
    body += f'Content-Disposition: form-data; name="ref_audio"; filename="{filename}"\r\n'.encode()
    body += f"Content-Type: {mime}\r\n\r\n".encode()
    with open(ref_audio_path, "rb") as f:
        body += f.read()
    body += b"\r\n"
    body += f"--{boundary}--\r\n".encode()

    url = f"http://localhost:3900{clone_ep}" if clone_ep else CLONE_URL
    req = urllib.request.Request(
        url, data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
        method="POST")

    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:34s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s │ 🔗clone")
            return True
    except urllib.error.HTTPError as e:
        error_body = e.read().decode() if hasattr(e, "read") else str(e)
        print(f"  ❌ Clone failed (HTTP {e.code}): {error_body[:160]}")
        return False
    except Exception as e:
        print(f"  ❌ Clone error: {e}")
        return False

def gen_with_json(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                  class_temperature=0.3, postprocess_output=True, seed=42):
    """🔒 Fallback: /v1/audio/speech — JSON (seed lock)। সব segment-ে একই voice।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:34s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s │ 🔒seed")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False


# ════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — SEGMENTS
# ════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text": "What if the greatest invention in human history... wasn't a machine that traveled through space — but one that crossed possibilities?", "speed": 0.9, "guidance_scale": 2.2},
    {"silence_ms": 400},
    {"tag": "02_doubt", "text": "Another universe? Seriously? Scientists kept working. Everyone else kept doubting.", "speed": 0.92, "guidance_scale": 2},
    {"silence_ms": 300},
    {"tag": "03_wonder", "text": "And suddenly — the impossible became real. We saw dinosaurs, still walking, beneath blood-red skies. We found another Earth where humanity was born on Mars.", "speed": 0.9, "guidance_scale": 2.2},
    {"silence_ms": 700},
    {"tag": "04_shift", "text": "But me... no. None of those worlds mattered. Not one. I was searching... for someone.", "speed": 0.85, "guidance_scale": 2, "postprocess_output": False},
    {"silence_ms": 800},
    {"tag": "05_grief", "text": "Three years ago... cancer... stole my mother. No warning. No mercy. No second chance.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1000},
    {"tag": "06_hospital", "text": "I watched the hospital monitor... become... silent. I held her hand — hoping — just hoping — she'd squeeze mine... one... last... time.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1500},
    {"tag": "07_she_never_did", "text": "She never did.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1000},
    {"tag": "08_pain", "text": "I know she'll never answer. I know that. But I still call. Because sometimes... hope hurts more than reality.", "speed": 0.85, "guidance_scale": 2.2},
    {"silence_ms": 800},
    {"tag": "09_plea", "text": "Listen. Take me to the universe... where my mother... never died.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1200},
    {"tag": "10_warmth", "text": "There she was. Alive. Smiling. Making breakfast. Humming the exact same song she used to sing... every Sunday morning.", "speed": 0.88, "guidance_scale": 2},
    {"silence_ms": 1000},
    {"tag": "11_ending", "text": "She didn't know I wasn't her son. I was a broken man... borrowing someone else's miracle.", "speed": 0.85, "guidance_scale": 2, "postprocess_output": False},
]


# ════════════════════════════════════════════════════════════════════════════
# 🚀 V8.7 — TRUE VOICE LOCK  (একটাই গলা, শুরু → শেষ)
# ════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)
speech_segs = [s for s in SEGMENTS if "text" in s]

# ─── 1) ANCHOR ঠিক করা (fixed reference voice) ─────────────────────────────
# anchor একবার সেট হলে আর বদলাবে না। এটাই voice lock-এর মূল।
anchor = auto_discover_voice(CUSTOM_VOICE_PATH)   # None হতে পারে
method = None                                      # 'clone' | 'json'

# 🔑 anchor-এর transcript — প্রতিটা clone call-এ পাঠানো হবে (identity lock)
# custom voice → CUSTOM_REF_TEXT ; anchor = segment 01 হলে segment 01-এর text
anchor_ref_text = CUSTOM_REF_TEXT

print("=" * 70)
print("🎭  V8.7 — TRUE VOICE LOCK  (One Voice, Start → Finish)")
print("=" * 70)
print(f"🎙️  Anchor file  : {anchor or '(প্রথম segment থেকে তৈরি হবে)'}")
print(f"🔗  Clone EP     : {clone_ep or CLONE_URL}")
print(f"🔢  Fixed seed   : {SEED}")
print(f"🔁  Retries/seg  : {MAX_RETRIES}")
print(f"🎬  Speech segs  : {len(speech_segs)}")
print("=" * 70)
print()

# ─── 2) METHOD একবার ঠিক করা (probe with segment[0]) ───────────────────────
# প্রথম segment দিয়ে পরীক্ষা। যেটা চলবে সেটাই পুরো script-এ চলবে — মাঝে বদলাবে না।
first = speech_segs[0]
probe_out = os.path.join(seg_dir, first["tag"] + ".wav")

# CASE A: custom voice sample আছে → সেটাই anchor, তার transcript = CUSTOM_REF_TEXT
if anchor:
    if not anchor_ref_text.strip():
        print("  ⚠️  CUSTOM_REF_TEXT খালি! male→female flip হতে পারে।")
        print("      স্ক্রিপ্টের CUSTOM_REF_TEXT-এ sample-এর কথাগুলো লিখে দাও।\n")
    if gen_with_clone(first["text"], anchor, probe_out,
                      first.get("speed", 0.88), first.get("guidance_scale", 2.0),
                      SEED, ref_text=anchor_ref_text):
        method = "clone"
        print("🎉  Clone কাজ করছে → পুরো script-এ এই anchor + ref_text দিয়ে clone হবে\n")

# CASE B: custom voice নেই → segment 01 preset দিয়ে বানাও, সেটাই anchor হয়ে যাক
if method is None and not anchor:
    if gen_with_json(first["text"], PRESET_VOICE, probe_out,
                     first.get("speed", 0.88), first.get("guidance_scale", 2.0),
                     0.3, first.get("postprocess_output", True), SEED):
        # এখন segment 01-এর audio দিয়ে clone probe করো
        if gen_with_clone(speech_segs[1]["text"] if len(speech_segs) > 1 else first["text"],
                          probe_out,
                          os.path.join(seg_dir, "__chain_probe.wav"),
                          0.88, 2.0, SEED, ref_text=first["text"]):
            anchor = probe_out
            anchor_ref_text = first["text"]   # 🔑 anchor-এ যা বলা আছে সেটাই transcript
            method = "clone"
            print(f"🔄  Segment 01 → anchor সেট হলো: {first['tag']}")
            print("🎉  Chain-clone কাজ করছে → বাকি সব segment segment 01-এর গলায় হবে\n")
        else:
            method = "json"
            print("🔒  Clone chain কাজ করলো না → পুরো script JSON + fixed seed\n")
    else:
        method = "json"

# CASE C: custom voice ছিল কিন্তু clone fail → পুরোটা JSON (mix নয়)
if method is None:
    method = "json"
    gen_with_json(first["text"], PRESET_VOICE, probe_out,
                  first.get("speed", 0.88), first.get("guidance_scale", 2.0),
                  0.3, first.get("postprocess_output", True), SEED)
    print("🔒  Clone endpoint কাজ করলো না → পুরো script-এ JSON + fixed seed (voice mix নয়)\n")


# ─── 3) gen_locked: method অনুযায়ী generate, fail হলে retry, voice বদলায় না ─
def gen_locked(seg):
    text  = seg["text"]
    out   = os.path.join(seg_dir, seg["tag"] + ".wav")
    speed = seg.get("speed", 0.88)
    gs    = seg.get("guidance_scale", 2.0)
    pp    = seg.get("postprocess_output", True)

    if method == "clone" and anchor:
        for attempt in range(MAX_RETRIES):
            if gen_with_clone(text, anchor, out, speed, gs, SEED,
                              ref_text=anchor_ref_text):   # 🔑 identity lock
                return out
            print(f"     ↻ retry {attempt + 1}/{MAX_RETRIES}  (voice switch নয়, retry...)")
            time.sleep(2)
        # সব retry fail → clone mode-এ JSON-এ যাই না (mixing এড়াতে); জায়গা ফাঁকা রাখি
        print(f"     ⚠️  {seg['tag']} clone করা গেল না — জায়গা ফাঁকা রাখা হলো (voice mix নয়)")
        return None
    else:
        for attempt in range(MAX_RETRIES):
            if gen_with_json(text, PRESET_VOICE, out, speed, gs, 0.3, pp, SEED):
                return out
            print(f"     ↻ retry {attempt + 1}/{MAX_RETRIES} ...")
            time.sleep(2)
        return None


# ─── 4) সব segment generate (প্রথমটা probe-এ তৈরি, বাকিগুলো locked method) ──
wav_order = []
generated = 0
failed = []
sil_idx = 0
first_done = False

for seg in SEGMENTS:
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    out_path = os.path.join(seg_dir, seg["tag"] + ".wav")
    if not first_done:
        first_done = True
        # probe-এ আগেই তৈরি হয়ে গেছে — আবার generate করি না (voice একই থাকে)
        res = out_path if os.path.exists(out_path) else gen_locked(seg)
    else:
        res = gen_locked(seg)

    if res and os.path.exists(res):
        wav_order.append(res)
        generated += 1
    else:
        failed.append(seg["tag"])

# ─── Concatenate ─────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = FINAL_OUTPUT_PATH
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Skipped (no voice mix): {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")

# ─── Voice Consistency Report ────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("📋 VOICE CONSISTENCY REPORT:")
if method == "clone":
    print("  🔗 Method: Reference Audio Cloning (BEST — one voice locked)")
    print(f"  🎙️ Anchor : {anchor}")
else:
    print(f"  🔒 Method: Seed-lock fallback (one preset voice, fixed seed)")
    print(f"  🎙️ Voice : {PRESET_VOICE}, Seed: {SEED}")
print("  ✅ No segment switched to a different voice.")
print(f"{'=' * 70}")

# ─── Direct Download (base64 — no 404) ───────────────────────────────────────
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024 * 1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print("\n⬇️  নিচের বাটনে ক্লিক করো:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V10 — COMBINE MODE: পুরো স্ক্রিপ্ট সর্বনিম্ন API call-এ
# =============================================================================
# আপনার server-এ clone endpoint নেই (HTTP 404), seed voice lock করে না,
# reference clip-এ একাধিক speaker — এই ৩টা সমস্যার জন্যই একমাত্র সমাধান:
#
# 📌 পুরো স্ক্রিপ্ট ১-২ টা API call-এ submit করা।
#    যত কম call = তত কম voice বদলানোর সুযোগ।
#
# V10 vs V9.5 পার্থক্য:
#   ❌ anchor reference chaining সরানো (clone নেই)
#   ❌ speaker fingerprint verification সরানো (ভুল positive)
#   ❌ per-segment gender check সরানো (unreliable)
#   ✅ COMBINE_MODE: "all_in_one" / "split"
#   ✅ guidance_scale 2.0→3.0 (preset adherence বাড়াতে)
#   ✅ class_temperature = 0 (max determinism)
#   ✅ MAX_CHARS_PER_CALL = 1500 (পাওয়া গেলে পুরো script একবারে)
# =============================================================================

import os, sys, time, json, math, wave, base64, shutil, subprocess
import urllib.request, urllib.error
import numpy as np
from IPython.display import HTML, display

# ═════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════
HOST            = "http://localhost:3900"
API_URL         = HOST + "/v1/audio/speech"
HEALTH_URL      = HOST + "/health"
OPENAPI_URL     = HOST + "/openapi.json"

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v10_combine"
FILENAME        = "v10_combine"

# ── সবচেয়ে গুরুত্বপূর্ণ ──────────────────────────────────────────────────
# COMBINE_MODE:
#   "all_in_one" → পুরো স্ক্রিপ্ট ১টাই API call (দ্রুত, voice lock নিশ্চিত)
#   "split"      → MAX_CHARS_PER_CALL অনুযায়ী ভাগ (safe, কম error)
COMBINE_MODE = "all_in_one"      # 👈 এইটা চেষ্টা করুন আগে

# MAX_CHARS_PER_CALL = 0 মানে unlimited (শুধু all_in_one mode-এ কাজ করে)
MAX_CHARS_PER_CALL = 0           # 0 = unlimited (all_in_one)
# "split" mode-এ per-call limit:
SPLIT_CHARS      = 600

# ── Voice ──────────────────────────────────────────────────────────────────
VOICE = "onyx"                   # আপনার পছন্দের preset
SEED  = 1234

# Frozen params — সব segment-এ একই
SPEED             = 0.88
GUIDANCE_SCALE    = 3.0          # ↑ V9.5-এ ছিল 2.0, এখন 3.0 = বেশি voice lock
NUM_STEP          = 32
CLASS_TEMPERATURE = 0.0          # 0 = সম্পূর্ণ deterministic
POSTPROCESS       = True

# ── Audio pipeline ─────────────────────────────────────────────────────────
TARGET_SR         = 24000
TARGET_RMS_DBFS   = -20.0
PEAK_CEILING      = 0.97
FADE_MS           = 12

# ── Script pauses ──────────────────────────────────────────────────────────
# প্রতিটা pause_ms একটা নীরবতা segment হিসেবে যুক্ত হবে। এগুলো parameter নয়,
# শুধু timing fix করার জন্য।
PAUSE_BETWEEN_PARAGRAPHS = 800   # ms


# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — শুধু text + pause_ms। আর কিছু না।
# ═════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "pause_ms": 0, "text":
        "What if the greatest invention in human history "
        "wasn't a machine that traveled through space, "
        "but one that crossed possibilities?"},
    {"tag": "02_doubt", "pause_ms": 450, "text":
        "Another universe? Seriously? "
        "Scientists kept working. Everyone else kept doubting."},
    {"tag": "03_wonder", "pause_ms": 350, "text":
        "And suddenly, the impossible became real. "
        "We saw dinosaurs, still walking, beneath blood-red skies. "
        "We found another Earth, where humanity was born on Mars."},
    {"tag": "04_shift", "pause_ms": 700, "text":
        "But me... no. None of those worlds mattered. Not one. "
        "I was searching... for someone."},
    {"tag": "05_grief", "pause_ms": 800, "text":
        "Three years ago... cancer... stole my mother. "
        "No warning. No mercy. No second chance."},
    {"tag": "06_hospital", "pause_ms": 1000, "text":
        "I watched the hospital monitor... become... silent. "
        "I held her hand, hoping, just hoping, "
        "she would squeeze mine... one... last... time."},
    {"tag": "07_she_never_did", "pause_ms": 1400, "text":
        "She never did."},
    {"tag": "08_pain", "pause_ms": 1000, "text":
        "I know she will never answer. I know that. "
        "But I still call. "
        "Because sometimes... hope hurts more than reality."},
    {"tag": "09_plea", "pause_ms": 800, "text":
        "Listen. Take me to the universe... where my mother... never died."},
    {"tag": "10_warmth", "pause_ms": 1200, "text":
        "There she was. Alive. Smiling. Making breakfast. "
        "Humming the exact same song she used to sing... "
        "every Sunday morning."},
    {"tag": "11_ending", "pause_ms": 1000, "text":
        "She did not know I was not her son. "
        "I was a broken man... borrowing someone else's miracle."},
]


# ═════════════════════════════════════════════════════════════════════════════
# 🧰 AUDIO UTILITIES
# ═════════════════════════════════════════════════════════════════════════════
def wav_read(path):
    with wave.open(path, "rb") as wf:
        nch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
        raw = wf.readframes(n)
    if sw == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sw == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError("Unsupported sample width: " + str(sw))
    if nch > 1:
        x = x.reshape(-1, nch).mean(axis=1)
    return x.astype(np.float32), sr

def wav_write(path, x, sr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    x = np.clip(x, -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(pcm)

def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))

def normalize_rms(x, target_dbfs=TARGET_RMS_DBFS, ceiling=PEAK_CEILING):
    cur = rms(x)
    if cur < 1e-6: return x
    gain = (10.0 ** (target_dbfs / 20.0)) / cur
    y = x * gain
    peak = float(np.max(np.abs(y)) + 1e-12)
    if peak > ceiling: y = y * (ceiling / peak)
    return y.astype(np.float32)

def apply_fades(x, sr, ms=FADE_MS):
    n = min(int(sr * ms / 1000.0), len(x) // 2)
    if n <= 0: return x
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x.copy(); x[:n] *= ramp; x[-n:] *= ramp[::-1]
    return x

def silence(sr, ms):
    return np.zeros(int(sr * ms / 1000.0), dtype=np.float32)


# ═════════════════════════════════════════════════════════════════════════════
# 🌐 SERVER
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except: return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    s = find_studio_dir()
    if not s: print("  ❌ OmniVoice Studio পাওয়া যায়নি!"); return False
    log = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log, stderr=log, cwd=s)
    for i in range(30):
        time.sleep(2)
        if server_alive(): print("  ✅ Server ready!"); return True
    return False


# ═════════════════════════════════════════════════════════════════════════════
# 📡 API — শুধু JSON mode, কোনো clone নয়
# ═════════════════════════════════════════════════════════════════════════════
def gen_speech(text, voice, out_path):
    """JSON API call — একমাত্র পদ্ধতি (clone ব্যর্থ)"""
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": SPEED,
        "num_step": NUM_STEP,
        "guidance_scale": GUIDANCE_SCALE,
        "class_temperature": CLASS_TEMPERATURE,
        "postprocess_output": POSTPROCESS,
        "seed": SEED,
    }
    req = urllib.request.Request(
        API_URL, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            data = r.read()
        with open(out_path, "wb") as f: f.write(data)
        return True, len(data)
    except urllib.error.HTTPError as e:
        body = e.read().decode()[:200] if hasattr(e, 'read') else str(e)
        return False, f"HTTP {e.code}: {body}"
    except Exception as e:
        return False, str(e)


# ═════════════════════════════════════════════════════════════════════════════
# 🚀 PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    v = 1
    while os.path.exists(os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")): v += 1
    return v, os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")


print("=" * 74)
print("🔧 SERVER CHECK")
print("=" * 74)
if server_alive():
    print("  ✅ Server running")
elif not start_server():
    raise RuntimeError("Server failed.")

RUN_VERSION, FINAL_OUTPUT_PATH = next_version(BASE_OUTPUT_DIR, FILENAME)
seg_dir  = os.path.join(BASE_OUTPUT_DIR, "segments_v" + str(RUN_VERSION))
work_dir = os.path.join(BASE_OUTPUT_DIR, "work")
os.makedirs(seg_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE TEXT — COMBINE_MODE অনুযায়ী
# ─────────────────────────────────────────────────────────────────────────────
speech_segs = [s for s in SEGMENTS if s.get("text")]
all_texts = [s["text"] for s in speech_segs]
all_tags  = [s["tag"] for s in speech_segs]
all_pauses = [s.get("pause_ms", 0) for s in SEGMENTS if s.get("pause_ms") is not None]

if COMBINE_MODE == "all_in_one":
    # পুরো script একটাই স্ট্রিং
    full_text = " ".join(all_texts)
    chunks   = [(full_text, "FULL")]
    strategy = f"all_in_one (1 call, {len(full_text)} chars)"
elif COMBINE_MODE == "split":
    limit = SPLIT_CHARS
    chunks, buf, buf_tags, buf_pauses = [], "", [], []
    for i, (t, tag) in enumerate(zip(all_texts, all_tags)):
        pause = all_pauses[i] if i < len(all_pauses) else PAUSE_BETWEEN_PARAGRAPHS
        if len(buf) + len(t) + 2 > limit and buf:
            chunks.append((buf, "+".join(buf_tags)))
            buf, buf_tags = "", []
        buf += (" " + t) if buf else t
        buf_tags.append(tag)
    if buf:
        chunks.append((buf, "+".join(buf_tags)))
    strategy = f"split ({len(chunks)} calls, {SPLIT_CHARS} chars/call)"
else:
    raise ValueError("Unknown COMBINE_MODE: " + COMBINE_MODE)

print()
print("=" * 74)
print("🎭  V10 — COMBINE MODE")
print("=" * 74)
print("  🔊 Voice       :", VOICE)
print("  🔒 Seed        :", SEED)
print("  🎚️ Guidance    :", GUIDANCE_SCALE)
print("  🌡️ Class temp  :", CLASS_TEMPERATURE)
print("  📦 Strategy    :", strategy)
print("  🎬 Segments    :", len(speech_segs), "speech")
print("  📁 Output      :", os.path.basename(FINAL_OUTPUT_PATH))
print("=" * 74)
print()


# ─────────────────────────────────────────────────────────────────────────────
# GENERATE
# ─────────────────────────────────────────────────────────────────────────────
timeline  = []
generated = 0
failed    = []
t_start   = time.time()

for text, tag_label in chunks:
    t0 = time.time()
    raw_path = os.path.join(work_dir, f"_{tag_label}.wav")
    ok, extra = gen_speech(text, VOICE, raw_path)

    if not ok:
        failed.append(tag_label)
        print(f"  ❌ {tag_label:30s} — {extra}")
        continue

    x, sr = wav_read(raw_path)
    x = x.astype(np.float32)
    # normalize pipeline
    x = normalize_rms(x)
    x = apply_fades(x, sr)
    dur = len(x) / sr

    # save individual segment
    chunk_wav = os.path.join(seg_dir, f"{tag_label}.wav")
    wav_write(chunk_wav, x, sr)

    kb = extra // 1024 if isinstance(extra, int) else os.path.getsize(raw_path) // 1024
    print(f"  ✅ {tag_label:30s} │ {dur:5.1f}s │ {kb:5d} KB │ {time.time()-t0:4.1f}s")
    timeline.append((tag_label, x))
    generated += 1

    # pause after this chunk (except last)
    if len(timeline) < len(chunks):
        pause_ms = PAUSE_BETWEEN_PARAGRAPHS
        timeline.append(("_pause", silence(TARGET_SR, pause_ms)))


# ─────────────────────────────────────────────────────────────────────────────
# CONCATENATE + MASTER
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 74)
print("🔗 Concatenating...")

parts = []
for label, x in timeline:
    if label.startswith("_"):
        parts.append(x)
    else:
        # resample to TARGET_SR
        parts.append(x)  # ইতিমধ্যেই normalize করা হয়েছে

if not parts:
    print("  ❌ কিছুই তৈরি হয়নি!"); sys.exit(1)

final = np.concatenate(parts)
final = normalize_rms(final, TARGET_RMS_DBFS + 2.0)
peak = float(np.max(np.abs(final)) + 1e-9)
if peak > PEAK_CEILING:
    final = final * (PEAK_CEILING / peak)
wav_write(FINAL_OUTPUT_PATH, final, TARGET_SR)

dur = len(final) / TARGET_SR
size_kb = os.path.getsize(FINAL_OUTPUT_PATH) // 1024
print(f"  🎬 FINAL: {os.path.basename(FINAL_OUTPUT_PATH)} ({size_kb} KB, {dur:.1f}s)")


# ─────────────────────────────────────────────────────────────────────────────
# REPORT
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("📋 REPORT")
print("=" * 74)
print(f"  ✅ Generated  : {generated}/{len(chunks)} chunks")
if failed:
    print(f"  ❌ Failed     : {', '.join(failed)}")
print(f"  🎚️ Voice      : {VOICE}")
print(f"  🔒 Seed       : {SEED}")
print(f"  🎚️ Guidance   : {GUIDANCE_SCALE}")
print(f"  🌡️ Class temp : {CLASS_TEMPERATURE}")
print(f"  📦 Strategy   : {strategy}")
print(f"  ⏱️ Total      : {time.time()-t_start:.1f}s")
print(f"  📁 File       : {FINAL_OUTPUT_PATH}")

if COMBINE_MODE == "all_in_one":
    print()
    print("  💡 ALL_IN_ONE — পুরো স্ক্রিপ্ট একটাই API call.")
    print("     ভেতরে কোনো voice বদলানোর সুযোগ নেই।")
    print("     তবুও যদি Male→Female শোনা যায়,")
    print("     তাহলে কারণ OmniVoice server-ই inconsistent,")
    print("     আপনার script-এর সমস্যা না।")
elif COMBINE_MODE == "split":
    print()
    print("  💡 SPLIT MODE — {len(chunks)} টি call.")
    print(f"     প্রতিটা {SPLIT_CHARS} character। সব একই voice + seed + params।")
    print("     যদি voice বদলায়, তাহলে split কমিয়ে দিন")
    print("     বা all_in_one mode ব্যবহার করুন।")


# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(FINAL_OUTPUT_PATH):
    with open(FINAL_OUTPUT_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(FINAL_OUTPUT_PATH)
    mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    display(HTML(
        '<audio controls src="data:audio/wav;base64,' + b64 +
        '" style="width:100%;margin:8px 0"></audio>'
        '<a download="' + dl + '" href="data:audio/wav;base64,' + b64 + '">'
        '<button style="padding:14px 28px;background:linear-gradient(135deg,#667eea,#764ba2);'
        'color:white;border:none;border-radius:8px;cursor:pointer;font-weight:bold;font-size:16px;">'
        '⬇️ Download ' + dl + ' (' + str(round(mb, 1)) + ' MB)</button></a>'))
else:
    print("❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V10 — CLONE REVIVER + TRUE VOICE LOCK
# -----------------------------------------------------------------------------
# 🔍 তোমার log-এ যা ধরা পড়েছে:
#    ❌ Clone failed (HTTP 404): {"detail":"Not Found"}
#       → clone endpoint মরে আছে! তাই সব audio preset voice দিয়ে তৈরি
#         হয়েছিল, আর preset mode-এ gender drift (male→female) হয়।
#    ⚠️ reference clip-এ একাধিক মানুষ সন্দেহ (F0 std 55 Hz)
#
# ✅ V10 যা করে:
#    1. CLONE REVIVER — একাধিক endpoint URL + field-name combination try করে
#       কোনটা কাজ করে খুঁজে বের করে (trailing slash সহ)।
#    2. POSTMORTEM     — clone মরে থাকলে ঠিক কী try হয়েছে সেটা রিপোর্ট দেয়।
#    3. REF_TEXT 🔑    — প্রতিটা clone call-এ reference-এর transcript পাঠায়
#       (identity lock — gender flip বন্ধ)।
#    4. ONE ANCHOR     — প্রথম speaker-এর voice পুরো script-এ fixed।
#    5. METHOD LOCK    — clone না json একবার ঠিক হয়, মাঝে বদলায় না।
#    6. RETRY, NO MIX  — fail হলে voice switch নয়, retry।
#    7. FIXED SEED     — সব call-এ একই seed।
# =============================================================================

import os, time, json, urllib.request, urllib.error, wave, subprocess, base64
from IPython.display import HTML, display

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v10_clone_reviver"
API_URL    = "http://localhost:3900/v1/audio/speech"
HEALTH_URL = "http://localhost:3900/health"

# ─── VOICE LOCK CONFIG ───────────────────────────────────────────────────────
CUSTOM_VOICE_PATH = "/kaggle/input/sample-voice/njk-nyntrnr-upy_S4kV3QyM.mp3"
CUSTOM_REF_TEXT   = ""   # 🔑 sample-এ ঠিক কী বলা আছে
PRESET_VOICE  = "onyx"   # anchor বানাতে (custom voice না থাকলে)
SEED          = 42                 # FIXED for ALL segments
MAX_RETRIES   = 4           # প্রতি segment-এ retry (voice switch এড়াতে)
FILENAME      = "v10_clone_reviver"


# ════════════════════════════════════════════════════════════════════════════
# 🔍 AUTO-DISCOVER: voice ফাইল খুঁজে বের করা
# ════════════════════════════════════════════════════════════════════════════
def auto_discover_voice(given_path):
    if given_path and os.path.exists(given_path):
        return given_path
    if not given_path:
        return None
    print(f"  ⚠️ দেওয়া পাথে ফাইল নেই: {given_path}")
    print("  🔍 /kaggle/input/ এ audio ফাইল খুঁজছি...")
    audio_files = []
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.lower().endswith((".mp3", ".wav", ".flac", ".ogg", ".m4a")):
                    audio_files.append(os.path.join(root, f))
    if not audio_files:
        print("  ❌ /kaggle/input/ এ কোনো audio ফাইল পাওয়া যায়নি")
        return None
    print(f"  📋 {len(audio_files)} টি audio ফাইল পাওয়া গেছে:")
    for i, af in enumerate(audio_files):
        print(f"     {i+1}. {af} ({os.path.getsize(af)//1024} KB)")
    chosen = audio_files[0]
    print(f"  ✅ ব্যবহার হচ্ছে: {chosen}")
    return chosen


# ════════════════════════════════════════════════════════════════════════════
# 📁 AUTO-VERSIONING
# ════════════════════════════════════════════════════════════════════════════
def get_next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    version = 1
    while True:
        candidate = os.path.join(base_dir, f"{base_name}_v{version}.wav")
        if not os.path.exists(candidate):
            return version, candidate
        version += 1

RUN_VERSION, FINAL_OUTPUT_PATH = get_next_version(BASE_OUTPUT_DIR, FILENAME)
print(f"📁 Auto-version: v{RUN_VERSION}  →  {os.path.basename(FINAL_OUTPUT_PATH)}\n")


# ════════════════════════════════════════════════════════════════════════════
# 🔧 SERVER CHECK
# ════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    studio = find_studio_dir()
    if not studio:
        print("  ❌ OmniVoice Studio পাওয়া যায়নি! আগে Step 2 রান করো।")
        return False
    print(f"  📂 {studio}")
    log_file = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"],
                     stdout=log_file, stderr=log_file, cwd=studio)
    for i in range(30):
        time.sleep(2)
        if server_alive():
            print(f"  ✅ Server ready! ({(i+1)*2}s)")
            return True
        if i % 5 == 4:
            print(f"  ⏳ Waiting... ({(i+1)*2}s)")
    print("  ❌ Failed! Run: !cat /tmp/omnivoice.log")
    return False

print("=" * 70)
print("🔧 SERVER CHECK")
print("=" * 70)
if server_alive():
    print("  ✅ Server running")
else:
    print("  ⚠️  Restarting...")
    if not start_server():
        raise RuntimeError("Server failed. Run Step 2 & 3 first.")
print()


# ════════════════════════════════════════════════════════════════════════════
# 🔬 CLONE REVIVER — endpoint + field-name খুঁজে বের করা
# ════════════════════════════════════════════════════════════════════════════
CLONE_ACTIVE = {"url": None, "fields": None}   # probe-এ যেটা কাজ করবে সেটা এখানে
TRIED_LOG = []                                  # postmortem-এর জন্য

def get_openapi():
    try:
        req = urllib.request.Request("http://localhost:3900/openapi.json")
        with urllib.request.urlopen(req, timeout=10) as resp:
            return json.loads(resp.read().decode())
    except Exception as e:
        print(f"  ⚠️ OpenAPI পড়া যায়নি: {e}")
        return None

def find_clone_candidates():
    """OpenAPI থেকে clone/speech path বের করি + কয়েকটা সম্ভাব্য path যোগ করি।"""
    cands = []
    spec = get_openapi()
    if spec:
        for path, methods in spec.get("paths", {}).items():
            # শুধু যাদের নামে "clone" আছে — plain /speech endpoint বাদ
            if "post" in methods and "clone" in path:
                if path not in cands:
                    cands.append(path)
        print(f"  📋 OpenAPI থেকে পাওয়া clone candidate: {cands}")
    for extra in ["/v1/audio/speech/clone", "/v1/audio/clone"]:
        if extra not in cands:
            cands.append(extra)
    return cands

# প্রতিটা field-set = (text, ref_audio, ref_text, speed)-এর field name
FIELDSETS = [
    {"text": "text",  "ref_audio": "ref_audio",      "ref_text": "ref_text", "speed": "speed"},
    {"text": "input", "ref_audio": "ref_audio",      "ref_text": "ref_text", "speed": "speed"},
    {"text": "text",  "ref_audio": "audio",          "ref_text": "ref_text", "speed": "speed"},
    {"text": "text",  "ref_audio": "reference_audio","ref_text": "ref_text", "speed": "speed"},
]

def build_multipart(fields, text, ref_audio_path, ref_text, speed):
    boundary = "----OmniVoiceV10" + str(int(time.time() * 1000))
    body = b""

    def add_field(name, value):
        nonlocal body
        body += f"--{boundary}\r\n".encode()
        body += f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode()
        body += f"{value}\r\n".encode()

    add_field(fields["text"], text)
    add_field(fields["speed"], speed)
    if ref_text:
        add_field(fields["ref_text"], ref_text)

    filename = os.path.basename(ref_audio_path)
    ext = os.path.splitext(filename)[1].lower()
    mime = {".wav": "audio/wav", ".mp3": "audio/mpeg", ".flac": "audio/flac",
            ".ogg": "audio/ogg", ".m4a": "audio/mp4"}.get(ext, "audio/wav")
    body += f"--{boundary}\r\n".encode()
    body += (f'Content-Disposition: form-data; name="{fields["ref_audio"]}"; '
             f'filename="{filename}"\r\n').encode()
    body += f"Content-Type: {mime}\r\n\r\n".encode()
    with open(ref_audio_path, "rb") as f:
        body += f.read()
    body += b"\r\n"
    body += f"--{boundary}--\r\n".encode()
    return body, boundary

def attempt_clone(url, fields, text, ref_audio_path, out_path, ref_text, speed, seed):
    """একটা নির্দিষ্ট URL + field-set দিয়ে clone try। success হলে True।"""
    body, boundary = build_multipart(fields, text, ref_audio_path, ref_text, speed)
    req = urllib.request.Request(
        url, data=body,
        headers={"Content-Type": f"multipart/form-data; boundary={boundary}"},
        method="POST")
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(out_path, "wb") as f:
                f.write(content)
            TRIED_LOG.append((url, "OK"))
            return True
    except urllib.error.HTTPError as e:
        TRIED_LOG.append((url, f"HTTP {e.code}"))
        return False
    except Exception as e:
        TRIED_LOG.append((url, str(e)[:60]))
        return False

def clone_postmortem():
    print("\n" + "=" * 70)
    print("🔬 CLONE POSTMORTEM — clone endpoint কাজ করলো না")
    print("=" * 70)
    print("  যা try করা হয়েছে:")
    for url, status in TRIED_LOG:
        print(f"    POST {url:50s} → {status}")
    print("  সম্ভাব্য কারণ ও সমাধান:")
    print("    1) Server STALE — route আছে কিন্তু reload হয়নি।")
    print("       👉 Step 2 (server start) notebook আবার চালান, তারপর এটা চালান।")
    print("    2) এই build-এ voice-clone feature নেই।")
    print("  → আপাতত JSON + fixed-seed mode-এ যাচ্ছি (একই preset voice)")
    print("=" * 70 + "\n")


# ════════════════════════════════════════════════════════════════════════════
# 🎵 CORE FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════
def concat_wavs(wav_paths, output_path):
    params_set = False
    all_frames = b""
    for wp in wav_paths:
        if not os.path.exists(wp):
            continue
        with wave.open(wp, "rb") as wf:
            if not params_set:
                params = wf.getparams()
                params_set = True
            all_frames += wf.readframes(wf.getnframes())
    if not params_set:
        print("  ❌ কোনো WAV ফাইল নেই!")
        return None
    with wave.open(output_path, "wb") as out:
        out.setparams(params)
        out.writeframes(all_frames)
    kb = os.path.getsize(output_path) // 1024
    dur = len(all_frames) / (params.framerate * params.sampwidth * params.nchannels)
    print(f"  🎬 FINAL: {os.path.basename(output_path)} ({kb} KB, {dur:.1f}s)")
    return output_path

def make_silence(ms, path, sr=22050):
    n = int(sr * ms / 1000)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(b"\x00\x00" * n)

def gen_with_json(text, voice, output_path, speed=0.88, guidance_scale=2.0,
                  class_temperature=0.3, postprocess_output=True, seed=42):
    """🔒 Fallback: /v1/audio/speech — JSON (seed lock)। সব segment-ে একই voice।"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": speed,
        "num_step": 32,
        "guidance_scale": guidance_scale,
        "seed": seed,
    }
    if class_temperature > 0:
        payload["class_temperature"] = class_temperature
    if not postprocess_output:
        payload["postprocess_output"] = False

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(API_URL, data=data,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=300) as resp:
            content = resp.read()
            with open(output_path, "wb") as f:
                f.write(content)
            tag = os.path.basename(output_path).replace(".wav", "")
            print(f"  ✅ {tag:34s} │ {len(content)//1024:5d} KB │ {time.time()-t0:4.1f}s │ 🔒seed")
            return True
    except Exception as e:
        print(f"  ❌ {os.path.basename(output_path)}: {e}")
        return False


# ════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — SEGMENTS
# ════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "text": "What if the greatest invention in human history... wasn't a machine that traveled through space — but one that crossed possibilities?", "speed": 0.9, "guidance_scale": 2.2},
    {"silence_ms": 400},
    {"tag": "02_doubt", "text": "Another universe? Seriously? Scientists kept working. Everyone else kept doubting.", "speed": 0.92, "guidance_scale": 2},
    {"silence_ms": 300},
    {"tag": "03_wonder", "text": "And suddenly — the impossible became real. We saw dinosaurs, still walking, beneath blood-red skies. We found another Earth where humanity was born on Mars.", "speed": 0.9, "guidance_scale": 2.2},
    {"silence_ms": 700},
    {"tag": "04_shift", "text": "But me... no. None of those worlds mattered. Not one. I was searching... for someone.", "speed": 0.85, "guidance_scale": 2, "postprocess_output": False},
    {"silence_ms": 800},
    {"tag": "05_grief", "text": "Three years ago... cancer... stole my mother. No warning. No mercy. No second chance.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1000},
    {"tag": "06_hospital", "text": "I watched the hospital monitor... become... silent. I held her hand — hoping — just hoping — she'd squeeze mine... one... last... time.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1500},
    {"tag": "07_she_never_did", "text": "She never did.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1000},
    {"tag": "08_pain", "text": "I know she'll never answer. I know that. But I still call. Because sometimes... hope hurts more than reality.", "speed": 0.85, "guidance_scale": 2.2},
    {"silence_ms": 800},
    {"tag": "09_plea", "text": "Listen. Take me to the universe... where my mother... never died.", "speed": 0.82, "guidance_scale": 1.8, "postprocess_output": False},
    {"silence_ms": 1200},
    {"tag": "10_warmth", "text": "There she was. Alive. Smiling. Making breakfast. Humming the exact same song she used to sing... every Sunday morning.", "speed": 0.88, "guidance_scale": 2},
    {"silence_ms": 1000},
    {"tag": "11_ending", "text": "She didn't know I wasn't her son. I was a broken man... borrowing someone else's miracle.", "speed": 0.85, "guidance_scale": 2, "postprocess_output": False},
]


# ════════════════════════════════════════════════════════════════════════════
# 🚀 V10 — CLONE REVIVER + VOICE LOCK  (একটাই গলা, শুরু → শেষ)
# ════════════════════════════════════════════════════════════════════════════
total_start = time.time()
seg_dir = os.path.join(BASE_OUTPUT_DIR, "segments")
os.makedirs(seg_dir, exist_ok=True)
speech_segs = [s for s in SEGMENTS if "text" in s]

anchor = auto_discover_voice(CUSTOM_VOICE_PATH)   # None হতে পারে
anchor_ref_text = CUSTOM_REF_TEXT                  # 🔑 anchor-এর transcript
method = None                                      # 'clone' | 'json'

print("=" * 70)
print("🎭  V10 — CLONE REVIVER + TRUE VOICE LOCK")
print("=" * 70)
print(f"🎙️  Anchor file  : {anchor or '(প্রথম segment থেকে তৈরি হবে)'}")
print(f"🔢  Fixed seed   : {SEED}")
print(f"🔁  Retries/seg  : {MAX_RETRIES}")
print(f"🎬  Speech segs  : {len(speech_segs)}")
print("=" * 70)
print()

first = speech_segs[0]
probe_out = os.path.join(seg_dir, first["tag"] + ".wav")


def try_clone_variants(text, ref_path, out_path, ref_text, speed, gs):
    """সব endpoint URL × field-set combination try। প্রথম success-এ থেমে যায়।"""
    candidates = find_clone_candidates()
    for path in candidates:
        for url in ["http://localhost:3900" + path.rstrip("/"),
                    "http://localhost:3900" + path.rstrip("/") + "/"]:
            for fields in FIELDSETS:
                if attempt_clone(url, fields, text, ref_path, out_path,
                                 ref_text, speed, SEED):
                    CLONE_ACTIVE["url"] = url
                    CLONE_ACTIVE["fields"] = fields
                    return True
    return False


# ─── ANCHOR + METHOD ঠিক করা ──────────────────────────────────────────────
if anchor:
    if not anchor_ref_text.strip():
        print("  ⚠️  CUSTOM_REF_TEXT খালি! gender flip হতে পারে —")
        print("      স্ক্রিপ্টের CUSTOM_REF_TEXT-এ sample-এর কথাগুলো লিখে দাও।\n")
    if try_clone_variants(first["text"], anchor, probe_out, anchor_ref_text,
                          first.get("speed", 0.88), first.get("guidance_scale", 2.0)):
        method = "clone"
        print(f"🎉  Clone REVIVED! কাজ করছে: {CLONE_ACTIVE['url']}")
        print(f"    field-set: {CLONE_ACTIVE['fields']}")
        print("    → পুরো script এই anchor + ref_text দিয়ে clone হবে\n")
    else:
        clone_postmortem()
        method = "json"
        gen_with_json(first["text"], PRESET_VOICE, probe_out,
                      first.get("speed", 0.88), first.get("guidance_scale", 2.0),
                      0.3, first.get("postprocess_output", True), SEED)

if method is None:
    # custom voice নেই → segment 01 preset দিয়ে বানাও, সেটাই anchor
    if gen_with_json(first["text"], PRESET_VOICE, probe_out,
                     first.get("speed", 0.88), first.get("guidance_scale", 2.0),
                     0.3, first.get("postprocess_output", True), SEED):
        chain_out = os.path.join(seg_dir, "__chain_probe.wav")
        if try_clone_variants(first["text"], probe_out, chain_out,
                              first["text"], 0.88, 2.0):
            anchor = probe_out
            anchor_ref_text = first["text"]   # 🔑 anchor-এ যা বলা আছে সেটাই transcript
            method = "clone"
            print(f"🔄  Segment 01 → anchor সেট হলো: {first['tag']}")
            print(f"🎉  Clone REVIVED: {CLONE_ACTIVE['url']}\n")
        else:
            clone_postmortem()
            method = "json"
    else:
        method = "json"


# ─── gen_locked: probed URL দিয়ে generate; fail হলে retry, voice বদলায় না ───
def gen_locked(seg):
    text  = seg["text"]
    out   = os.path.join(seg_dir, seg["tag"] + ".wav")
    speed = seg.get("speed", 0.88)
    gs    = seg.get("guidance_scale", 2.0)
    pp    = seg.get("postprocess_output", True)

    if method == "clone" and anchor and CLONE_ACTIVE["url"]:
        for attempt in range(MAX_RETRIES):
            if attempt_clone(CLONE_ACTIVE["url"], CLONE_ACTIVE["fields"],
                             text, anchor, out, anchor_ref_text, speed, SEED):
                tag = seg["tag"]
                print(f"  ✅ {tag:34s} │ 🔗clone (identity locked)")
                return out
            print(f"     ↻ retry {attempt + 1}/{MAX_RETRIES}  (voice switch নয়, retry...)")
            time.sleep(2)
        print(f"     ⚠️  {seg['tag']} clone করা গেল না — জায়গা ফাঁকা (voice mix নয়)")
        return None
    else:
        for attempt in range(MAX_RETRIES):
            if gen_with_json(text, PRESET_VOICE, out, speed, gs, 0.3, pp, SEED):
                return out
            print(f"     ↻ retry {attempt + 1}/{MAX_RETRIES} ...")
            time.sleep(2)
        return None


# ─── সব segment generate (প্রথমটা probe-এ তৈরি, বাকিগুলো locked) ──────────
wav_order = []
generated = 0
failed = []
sil_idx = 0
first_done = False

for seg in SEGMENTS:
    if "silence_ms" in seg:
        sil_path = os.path.join(seg_dir, f"sil_{sil_idx:02d}.wav")
        make_silence(seg["silence_ms"], sil_path)
        wav_order.append(sil_path)
        sil_idx += 1
        continue

    out_path = os.path.join(seg_dir, seg["tag"] + ".wav")
    if not first_done:
        first_done = True
        res = out_path if os.path.exists(out_path) else gen_locked(seg)
    else:
        res = gen_locked(seg)

    if res and os.path.exists(res):
        wav_order.append(res)
        generated += 1
    else:
        failed.append(seg["tag"])

# ─── Concatenate ─────────────────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("🔗 Concatenating...")
final_path = FINAL_OUTPUT_PATH
result = concat_wavs(wav_order, final_path)

total_time = time.time() - total_start
print(f"\n{'=' * 70}")
print(f"🎉  DONE! {generated}/{len(speech_segs)} segments → 1 file")
if failed:
    print(f"  ⚠️  Skipped (no voice mix): {', '.join(failed)}")
print(f"  ⏱️  {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  📁 {final_path}")

# ─── Voice Consistency Report ────────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("📋 VOICE CONSISTENCY REPORT:")
if method == "clone":
    print("  🔗 Method: Reference Audio Cloning (identity locked)")
    print(f"  🎙️ Anchor : {anchor}")
    print(f"  🔑 ref_text: {'yes' if anchor_ref_text.strip() else 'EMPTY ⚠️'}")
    print(f"  🌐 Endpoint: {CLONE_ACTIVE['url']}")
else:
    print(f"  🔒 Method: Seed-lock fallback (one preset voice, fixed seed)")
    print(f"  🎙️ Voice : {PRESET_VOICE}, Seed: {SEED}")
print("  ✅ No segment switched to a different voice.")
print(f"{'=' * 70}")

# ─── Direct Download (base64 — no 404) ───────────────────────────────────────
if result and os.path.exists(final_path):
    with open(final_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(final_path)
    mb = os.path.getsize(final_path) / (1024 * 1024)
    html = f'''<a download="{dl}" href="data:audio/wav;base64,{b64}">
    <button style="padding:14px 28px; background:linear-gradient(135deg,#667eea,#764ba2);
    color:white; border:none; border-radius:8px; cursor:pointer; font-weight:bold;
    font-size:16px; margin:10px 0;">
    ⬇️ Download {dl} ({mb:.1f} MB)
    </button></a>'''
    print("\n⬇️  নিচের বাটনে ক্লিক করো:")
    display(HTML(html))
else:
    print("\n❌ ফাইল তৈরি হয়নি।")


In [ ]:
# =============================================================================
# 🎭 V10 — COMBINE MODE: পুরো স্ক্রিপ্ট সর্বনিম্ন API call-এ
# =============================================================================
# আপনার server-এ clone endpoint নেই (HTTP 404), seed voice lock করে না,
# reference clip-এ একাধিক speaker — এই ৩টা সমস্যার জন্যই একমাত্র সমাধান:
#
# 📌 পুরো স্ক্রিপ্ট ১-২ টা API call-এ submit করা।
#    যত কম call = তত কম voice বদলানোর সুযোগ।
#
# V10 vs V9.5 পার্থক্য:
#   ❌ anchor reference chaining সরানো (clone নেই)
#   ❌ speaker fingerprint verification সরানো (ভুল positive)
#   ❌ per-segment gender check সরানো (unreliable)
#   ✅ COMBINE_MODE: "all_in_one" / "split"
#   ✅ guidance_scale 2.0→3.0 (preset adherence বাড়াতে)
#   ✅ class_temperature = 0 (max determinism)
#   ✅ MAX_CHARS_PER_CALL = 1500 (পাওয়া গেলে পুরো script একবারে)
# =============================================================================

import os, sys, time, json, math, wave, base64, shutil, subprocess
import urllib.request, urllib.error
import numpy as np
from IPython.display import HTML, display

# ═════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURATION
# ═════════════════════════════════════════════════════════════════════════════
HOST            = "http://localhost:3900"
API_URL         = HOST + "/v1/audio/speech"
HEALTH_URL      = HOST + "/health"
OPENAPI_URL     = HOST + "/openapi.json"

BASE_OUTPUT_DIR = "/kaggle/working/outputs/v10_combine"
FILENAME        = "v10_combine"

# ── সবচেয়ে গুরুত্বপূর্ণ ──────────────────────────────────────────────────
# COMBINE_MODE:
#   "all_in_one" → পুরো স্ক্রিপ্ট ১টাই API call (দ্রুত, voice lock নিশ্চিত)
#   "split"      → MAX_CHARS_PER_CALL অনুযায়ী ভাগ (safe, কম error)
COMBINE_MODE = "all_in_one"      # 👈 এইটা চেষ্টা করুন আগে

# MAX_CHARS_PER_CALL = 0 মানে unlimited (শুধু all_in_one mode-এ কাজ করে)
MAX_CHARS_PER_CALL = 0           # 0 = unlimited (all_in_one)
# "split" mode-এ per-call limit:
SPLIT_CHARS      = 600

# ── Voice ──────────────────────────────────────────────────────────────────
VOICE = "onyx"                   # আপনার পছন্দের preset
SEED  = 1234

# Frozen params — সব segment-এ একই
SPEED             = 0.88
GUIDANCE_SCALE    = 3.0          # ↑ V9.5-এ ছিল 2.0, এখন 3.0 = বেশি voice lock
NUM_STEP          = 32
CLASS_TEMPERATURE = 0.0          # 0 = সম্পূর্ণ deterministic
POSTPROCESS       = True

# ── Audio pipeline ─────────────────────────────────────────────────────────
TARGET_SR         = 24000
TARGET_RMS_DBFS   = -20.0
PEAK_CEILING      = 0.97
FADE_MS           = 12

# ── Script pauses ──────────────────────────────────────────────────────────
# প্রতিটা pause_ms একটা নীরবতা segment হিসেবে যুক্ত হবে। এগুলো parameter নয়,
# শুধু timing fix করার জন্য।
PAUSE_BETWEEN_PARAGRAPHS = 800   # ms


# ═════════════════════════════════════════════════════════════════════════════
# 🎬 SCRIPT — শুধু text + pause_ms। আর কিছু না।
# ═════════════════════════════════════════════════════════════════════════════
SEGMENTS = [
    {"tag": "01_opening", "pause_ms": 0, "text":
        "What if the greatest invention in human history "
        "wasn't a machine that traveled through space, "
        "but one that crossed possibilities?"},
    {"tag": "02_doubt", "pause_ms": 450, "text":
        "Another universe? Seriously? "
        "Scientists kept working. Everyone else kept doubting."},
    {"tag": "03_wonder", "pause_ms": 350, "text":
        "And suddenly, the impossible became real. "
        "We saw dinosaurs, still walking, beneath blood-red skies. "
        "We found another Earth, where humanity was born on Mars."},
    {"tag": "04_shift", "pause_ms": 700, "text":
        "But me... no. None of those worlds mattered. Not one. "
        "I was searching... for someone."},
    {"tag": "05_grief", "pause_ms": 800, "text":
        "Three years ago... cancer... stole my mother. "
        "No warning. No mercy. No second chance."},
    {"tag": "06_hospital", "pause_ms": 1000, "text":
        "I watched the hospital monitor... become... silent. "
        "I held her hand, hoping, just hoping, "
        "she would squeeze mine... one... last... time."},
    {"tag": "07_she_never_did", "pause_ms": 1400, "text":
        "She never did."},
    {"tag": "08_pain", "pause_ms": 1000, "text":
        "I know she will never answer. I know that. "
        "But I still call. "
        "Because sometimes... hope hurts more than reality."},
    {"tag": "09_plea", "pause_ms": 800, "text":
        "Listen. Take me to the universe... where my mother... never died."},
    {"tag": "10_warmth", "pause_ms": 1200, "text":
        "There she was. Alive. Smiling. Making breakfast. "
        "Humming the exact same song she used to sing... "
        "every Sunday morning."},
    {"tag": "11_ending", "pause_ms": 1000, "text":
        "She did not know I was not her son. "
        "I was a broken man... borrowing someone else's miracle."},
]


# ═════════════════════════════════════════════════════════════════════════════
# 🧰 AUDIO UTILITIES
# ═════════════════════════════════════════════════════════════════════════════
def wav_read(path):
    with wave.open(path, "rb") as wf:
        nch, sw, sr, n = wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getnframes()
        raw = wf.readframes(n)
    if sw == 2:
        x = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sw == 1:
        x = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sw == 4:
        x = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError("Unsupported sample width: " + str(sw))
    if nch > 1:
        x = x.reshape(-1, nch).mean(axis=1)
    return x.astype(np.float32), sr

def wav_write(path, x, sr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    x = np.clip(x, -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(pcm)

def rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))

def normalize_rms(x, target_dbfs=TARGET_RMS_DBFS, ceiling=PEAK_CEILING):
    cur = rms(x)
    if cur < 1e-6: return x
    gain = (10.0 ** (target_dbfs / 20.0)) / cur
    y = x * gain
    peak = float(np.max(np.abs(y)) + 1e-12)
    if peak > ceiling: y = y * (ceiling / peak)
    return y.astype(np.float32)

def apply_fades(x, sr, ms=FADE_MS):
    n = min(int(sr * ms / 1000.0), len(x) // 2)
    if n <= 0: return x
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x.copy(); x[:n] *= ramp; x[-n:] *= ramp[::-1]
    return x

def silence(sr, ms):
    return np.zeros(int(sr * ms / 1000.0), dtype=np.float32)


# ═════════════════════════════════════════════════════════════════════════════
# 🌐 SERVER
# ═════════════════════════════════════════════════════════════════════════════
def server_alive():
    try:
        with urllib.request.urlopen(urllib.request.Request(HEALTH_URL), timeout=5) as r:
            return r.status == 200
    except: return False

def find_studio_dir():
    for root, dirs, files in os.walk("/kaggle/working"):
        if "backend" in dirs and os.path.exists(os.path.join(root, "backend", "main.py")):
            return root
    return None

def start_server():
    s = find_studio_dir()
    if not s: print("  ❌ OmniVoice Studio পাওয়া যায়নি!"); return False
    log = open("/tmp/omnivoice.log", "w")
    subprocess.Popen(["uv", "run", "python", "backend/main.py"], stdout=log, stderr=log, cwd=s)
    for i in range(30):
        time.sleep(2)
        if server_alive(): print("  ✅ Server ready!"); return True
    return False


# ═════════════════════════════════════════════════════════════════════════════
# 📡 API — শুধু JSON mode, কোনো clone নয়
# ═════════════════════════════════════════════════════════════════════════════
def gen_speech(text, voice, out_path):
    """JSON API call — একমাত্র পদ্ধতি (clone ব্যর্থ)"""
    payload = {
        "model": "tts-1-hd",
        "voice": voice,
        "input": text,
        "response_format": "wav",
        "speed": SPEED,
        "num_step": NUM_STEP,
        "guidance_scale": GUIDANCE_SCALE,
        "class_temperature": CLASS_TEMPERATURE,
        "postprocess_output": POSTPROCESS,
        "seed": SEED,
    }
    req = urllib.request.Request(
        API_URL, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            data = r.read()
        with open(out_path, "wb") as f: f.write(data)
        return True, len(data)
    except urllib.error.HTTPError as e:
        body = e.read().decode()[:200] if hasattr(e, 'read') else str(e)
        return False, f"HTTP {e.code}: {body}"
    except Exception as e:
        return False, str(e)


# ═════════════════════════════════════════════════════════════════════════════
# 🚀 PIPELINE
# ═════════════════════════════════════════════════════════════════════════════
def next_version(base_dir, base_name):
    os.makedirs(base_dir, exist_ok=True)
    v = 1
    while os.path.exists(os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")): v += 1
    return v, os.path.join(base_dir, base_name + "_v" + str(v) + ".wav")


print("=" * 74)
print("🔧 SERVER CHECK")
print("=" * 74)
if server_alive():
    print("  ✅ Server running")
elif not start_server():
    raise RuntimeError("Server failed.")

RUN_VERSION, FINAL_OUTPUT_PATH = next_version(BASE_OUTPUT_DIR, FILENAME)
seg_dir  = os.path.join(BASE_OUTPUT_DIR, "segments_v" + str(RUN_VERSION))
work_dir = os.path.join(BASE_OUTPUT_DIR, "work")
os.makedirs(seg_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE TEXT — COMBINE_MODE অনুযায়ী
# ─────────────────────────────────────────────────────────────────────────────
speech_segs = [s for s in SEGMENTS if s.get("text")]
all_texts = [s["text"] for s in speech_segs]
all_tags  = [s["tag"] for s in speech_segs]
all_pauses = [s.get("pause_ms", 0) for s in SEGMENTS if s.get("pause_ms") is not None]

if COMBINE_MODE == "all_in_one":
    # পুরো script একটাই স্ট্রিং
    full_text = " ".join(all_texts)
    chunks   = [(full_text, "FULL")]
    strategy = f"all_in_one (1 call, {len(full_text)} chars)"
elif COMBINE_MODE == "split":
    limit = SPLIT_CHARS
    chunks, buf, buf_tags, buf_pauses = [], "", [], []
    for i, (t, tag) in enumerate(zip(all_texts, all_tags)):
        pause = all_pauses[i] if i < len(all_pauses) else PAUSE_BETWEEN_PARAGRAPHS
        if len(buf) + len(t) + 2 > limit and buf:
            chunks.append((buf, "+".join(buf_tags)))
            buf, buf_tags = "", []
        buf += (" " + t) if buf else t
        buf_tags.append(tag)
    if buf:
        chunks.append((buf, "+".join(buf_tags)))
    strategy = f"split ({len(chunks)} calls, {SPLIT_CHARS} chars/call)"
else:
    raise ValueError("Unknown COMBINE_MODE: " + COMBINE_MODE)

print()
print("=" * 74)
print("🎭  V10 — COMBINE MODE")
print("=" * 74)
print("  🔊 Voice       :", VOICE)
print("  🔒 Seed        :", SEED)
print("  🎚️ Guidance    :", GUIDANCE_SCALE)
print("  🌡️ Class temp  :", CLASS_TEMPERATURE)
print("  📦 Strategy    :", strategy)
print("  🎬 Segments    :", len(speech_segs), "speech")
print("  📁 Output      :", os.path.basename(FINAL_OUTPUT_PATH))
print("=" * 74)
print()


# ─────────────────────────────────────────────────────────────────────────────
# GENERATE
# ─────────────────────────────────────────────────────────────────────────────
timeline  = []
generated = 0
failed    = []
t_start   = time.time()

for text, tag_label in chunks:
    t0 = time.time()
    raw_path = os.path.join(work_dir, f"_{tag_label}.wav")
    ok, extra = gen_speech(text, VOICE, raw_path)

    if not ok:
        failed.append(tag_label)
        print(f"  ❌ {tag_label:30s} — {extra}")
        continue

    x, sr = wav_read(raw_path)
    x = x.astype(np.float32)
    # normalize pipeline
    x = normalize_rms(x)
    x = apply_fades(x, sr)
    dur = len(x) / sr

    # save individual segment
    chunk_wav = os.path.join(seg_dir, f"{tag_label}.wav")
    wav_write(chunk_wav, x, sr)

    kb = extra // 1024 if isinstance(extra, int) else os.path.getsize(raw_path) // 1024
    print(f"  ✅ {tag_label:30s} │ {dur:5.1f}s │ {kb:5d} KB │ {time.time()-t0:4.1f}s")
    timeline.append((tag_label, x))
    generated += 1

    # pause after this chunk (except last)
    if len(timeline) < len(chunks):
        pause_ms = PAUSE_BETWEEN_PARAGRAPHS
        timeline.append(("_pause", silence(TARGET_SR, pause_ms)))


# ─────────────────────────────────────────────────────────────────────────────
# CONCATENATE + MASTER
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 74)
print("🔗 Concatenating...")

parts = []
for label, x in timeline:
    if label.startswith("_"):
        parts.append(x)
    else:
        # resample to TARGET_SR
        parts.append(x)  # ইতিমধ্যেই normalize করা হয়েছে

if not parts:
    print("  ❌ কিছুই তৈরি হয়নি!"); sys.exit(1)

final = np.concatenate(parts)
final = normalize_rms(final, TARGET_RMS_DBFS + 2.0)
peak = float(np.max(np.abs(final)) + 1e-9)
if peak > PEAK_CEILING:
    final = final * (PEAK_CEILING / peak)
wav_write(FINAL_OUTPUT_PATH, final, TARGET_SR)

dur = len(final) / TARGET_SR
size_kb = os.path.getsize(FINAL_OUTPUT_PATH) // 1024
print(f"  🎬 FINAL: {os.path.basename(FINAL_OUTPUT_PATH)} ({size_kb} KB, {dur:.1f}s)")


# ─────────────────────────────────────────────────────────────────────────────
# REPORT
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 74)
print("📋 REPORT")
print("=" * 74)
print(f"  ✅ Generated  : {generated}/{len(chunks)} chunks")
if failed:
    print(f"  ❌ Failed     : {', '.join(failed)}")
print(f"  🎚️ Voice      : {VOICE}")
print(f"  🔒 Seed       : {SEED}")
print(f"  🎚️ Guidance   : {GUIDANCE_SCALE}")
print(f"  🌡️ Class temp : {CLASS_TEMPERATURE}")
print(f"  📦 Strategy   : {strategy}")
print(f"  ⏱️ Total      : {time.time()-t_start:.1f}s")
print(f"  📁 File       : {FINAL_OUTPUT_PATH}")

if COMBINE_MODE == "all_in_one":
    print()
    print("  💡 ALL_IN_ONE — পুরো স্ক্রিপ্ট একটাই API call.")
    print("     ভেতরে কোনো voice বদলানোর সুযোগ নেই।")
    print("     তবুও যদি Male→Female শোনা যায়,")
    print("     তাহলে কারণ OmniVoice server-ই inconsistent,")
    print("     আপনার script-এর সমস্যা না।")
elif COMBINE_MODE == "split":
    print()
    print("  💡 SPLIT MODE — {len(chunks)} টি call.")
    print(f"     প্রতিটা {SPLIT_CHARS} character। সব একই voice + seed + params।")
    print("     যদি voice বদলায়, তাহলে split কমিয়ে দিন")
    print("     বা all_in_one mode ব্যবহার করুন।")


# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(FINAL_OUTPUT_PATH):
    with open(FINAL_OUTPUT_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    dl = os.path.basename(FINAL_OUTPUT_PATH)
    mb = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)
    display(HTML(
        '<audio controls src="data:audio/wav;base64,' + b64 +
        '" style="width:100%;margin:8px 0"></audio>'
        '<a download="' + dl + '" href="data:audio/wav;base64,' + b64 + '">'
        '<button style="padding:14px 28px;background:linear-gradient(135deg,#667eea,#764ba2);'
        'color:white;border:none;border-radius:8px;cursor:pointer;font-weight:bold;font-size:16px;">'
        '⬇️ Download ' + dl + ' (' + str(round(mb, 1)) + ' MB)</button></a>'))
else:
    print("❌ ফাইল তৈরি হয়নি।")
